# Knowledge-Guided Water-Quality Classification with a Reduced Laboratory Panel
### Cross-country evaluation on 2.82 million records from five countries — v14 (corrections of v13)

**Task.** At sampling time *t*, the water-quality class is defined by eight parameters measured at *t*. The model observes
the four field-sensor parameters (DO, pH, temperature, nitrate) at *t*, *k* of the four laboratory parameters
(BOD, NH₃, PO₄, N) at *t*, and the site's history. It must infer the effect of the 4 − *k* laboratory tests that were **not**
run. The model is evaluated on countries it never saw during training.

**Label.** A single-sample (record-level) CCME-type index: the CCME WQI formula applied to one record, with the CCME dissolved-oxygen
minimum and pH range and, for the other parameters, the objectives of the published dataset label (§5, §5b). CCME recommends the
index for at least four samples per site and year; the comparison with the standard site-year index is reported (§5b).

**Approach — knowledge-guided ML.** The known CCME aggregation formula is evaluated on *only the information available at
prediction time* and supplied as features ("CCME-consistency features"). The main comparators are the **formula rule** (the same
formula with the unmeasured tests at their last genuinely measured value; no learning) and a **black box** with identical inputs, data and acquisition
pattern but without these features.

### New in v14 — corrections of v13 (pre-submission review)
| # | Problem in v13 | Correction in v14 | Where |
|---|---|---|---|
| 1 | **Adaptive selection used one Monte-Carlo draw, not `MS`.** The hypothetical values of a sample were assigned to the same cell of `LABCUR` (`LABCUR[np.repeat(rows, MS), j] = draws`); with repeated indices numpy keeps only the last value, so all `MS` draws were identical | Hypothetical values are written into a copy with one row per draw; no global array is modified | §8 `adaptive_masks` |
| 2 | **Quadratic weighted kappa re-indexed the classes** that occur in a subset (scikit-learn default, mirrored by the fast bootstrap metric), so the weight between e.g. Marginal and Good changed whenever a class was absent (Ireland has 2 Poor samples; most bootstrap draws lose a class) | QWK weights on the fixed ordinal index 0–4 (`labels=range(K)`), in both implementations; unit test with absent classes | §8 |
| 3 | **Training labels contained imputed values.** Every station row trained the models, but its label was computed from the eight values *including* country-median imputations that the model sees as missing — e.g. England's imputed PO₄ (0.144 mg/L > 0.05) made every such row fail PO₄ | Rows may train, tune or support a model only if their laboratory values are genuinely measured (`TRAIN_TIER = "B"`); tier A and the v13 behaviour are re-evaluated in §15e; the leakage audit tests it | §3, §6, §8, §9b, §15e |
| 4 | **"Last genuine value" was the value at *t−1*.** The rule, the proxy features and the cost analysis treated an unmeasured test as *passing* whenever its *t−1* value was imputed (45 % of tier-A samples, Table 33 of v13) | The most recent genuinely measured value of the same series (at most `LAST_MAX_AGE_DAYS` old) is carried forward; the leakage audit verifies its origin; Table 33 groups by its age | §6, §9b, §12c |
| 5 | **The headline was not on unseen countries.** It used 10 % of the test country's labels, and 30–84 % of the test sites also contributed training labels, while the text and Table 14 said "never seen in training" | `MAIN_BUDGET = 0.0`: the headline uses no sample of the test country; the 10–50 % budgets remain in Table 21 | §1, §9, §10 |
| 6 | **Failure rates counted imputed values** (Table 11; "PO₄ fails in 67.4 % of station site-days", Table 52) | Computed on genuinely measured values only | §6, §15 |
| 7 | **Subgroup claims rested on unadjusted p-values** (0 vs 1 vs 2 vs 3 tests; age of the last value; Tables 26–27) | Holm-adjusted p within each family; an automated claim check (Table 60) states which claims the run supports | §12, §12c, §17 |
| 8 | **Nitrate units.** Table 9 shows nitrate is reported as N in England and Ireland, but the label applies 45 mg/L, the drinking-water value *as NO₃* (= 10 mg/L as N) | The same guideline expressed as N (10 mg/L) is added to the sensitivity analysis; the inconsistency is stated as a limitation | §1, §15b, §18 |
| 9 | **Notes for the manuscript contained hard-coded v13 numbers** that would not match a re-run | Notes rewritten without numbers; every number comes from the tables of the run | §18 |
| 10 | Reproducibility table compared with v12 although v14 changes the method | Becomes "changes relative to v13" | §16b |

> **Execution status.** v14 has a new pipeline ID, so no v13 checkpoint can be reused, and all outputs are cleared: run the notebook
> end-to-end on `Combined_dataset.csv` before using any number. The code was executed end-to-end on a synthetic file with the same
> columns (smoke test only, no results).

### New in v13 (methodological audit for journal submission)
| Step | Addition | Where |
|---|---|---|
| Label audit | Published label errors, class bands, single-sample vs site-year index, every objective with its source, reporting-unit check of nitrate | §5, §5b |
| Sensitivity of the label | Alternative PO₄ objectives (Ireland good status, US EPA 1986), NH₃ objectives (Ireland good status, CCME pH- and temperature-dependent guideline) and nitrate expressed as N; classes changed and all models re-evaluated, including the formula rule | §15b, §15d |
| Leakage audit | Automated checks: test country excluded, validation-country model selection, temporal order of labels, lagged and site-history features recomputed from earlier visits only, unmeasured values never read, no shared sites, duplicate measurement vectors | §9b |
| Baselines on the same test records | Calibrated formula rule, logistic regression and random forest (with and without CCME features); paired tests against the formula rule with Holm adjustment; results with the test country completely excluded (0 % labels) per country | §12b |
| Practical value | Accuracy with 0–3 laboratory tests, by age and availability of the last laboratory value, flagging of hidden failures, selective full-panel testing, cost and turnaround from published laboratory fee schedules, comparison with published reduced-panel studies | §12c |
| Reproducibility | v13 rerun compared value-by-value with the v12 results | §16b |
| Presentation | 59 numbered tables with titles and notes, fixed method order, plain metric names, best values in bold | all sections |

### Earlier additions (v12)
ROC-AUC and PR-AUC with site-bootstrap CIs, ROC curves and calibration (§10, §10b); explainability — class-wise TreeSHAP, cross-country
stability, measurement-level exact Shapley values, domain-consistency test, LIME and SHAP–LIME agreement, SHAP-space t-SNE (§13–§13e);
deep-learning baselines and few-shot adaptation including a re-implementation of Rahman et al. (2026) (§14b); replication of the
prior-work protocol (§14c); error-rate reduction (§12).

### How each review issue is resolved (v11 → v12)
| # | Issue | Resolution | Section |
|---|---|---|---|
| 1 | Duplicates removed without `country` | Duplicates removed on (country, site, date, all 8 values); repeated same-day measurements audited and median-aggregated; single-record subset reported | §4a, §10 |
| 2 | Record-level formula vs standard CCME | Single-sample formula derived from, and unit-tested against, the standard multi-sample CCME formula; guideline values documented with sources (CCME DO 5.5 mg/L, pH 6.5–9.0); standard site-year CCME compared | §5 |
| 3 | Grouping by `site` alone | Series grouped by (country, site); identifiers audited, and aggregate identifiers (median > 2 records per day, e.g. China's single identifier) excluded from the station-level analysis | §4c, §6 |
| 4 | Imputation heuristic | Dataset documentation: country-wise **median imputation**. Primary rule = documented rule ∪ dominant-constant rule; validated by frequency-spike ratios and rule agreement; threshold sensitivity | §4b, §15c |
| 5 | Target built from model inputs | Task defined explicitly; leakage unit test; formula-rule baselines; results on the **hidden-failure** subset (an unmeasured test actually fails) | §7, §8, §10 |
| 6 | Per-country dependence | Per-country sample counts, class distributions, majority baseline and metrics; genuine-data tiers A/B | §10 |
| 7 | Adaptive vs fixed | Fixed, random, adaptive and best-possible (oracle) subsets on identical samples and budgets (asserted) | §11 |
| 8 | QRC vs XGBoost | Identical splits (asserted), identical input encoding, linear and XGBoost readouts, wider circuit tuning, 3 reservoir seeds | §16 |
| 9 | Non-independent repeats | Site-clustered bootstrap CIs within countries; random-effects meta-analysis (Hartung–Knapp) across countries | §12 |
| 10 | Stale checkpoints | Checkpoints stored under a pipeline hash and verified on load; previous checkpoints archived; everything recomputed | §1 |

> **Execution record (v13, superseded).** v13 ran as pipeline `v13-r1-e2915f9fa2` and reproduced the v12 run exactly. Its numbers
> are affected by corrections 1–7 above and must not be reused.


## List of tables

Every results table is numbered, has a title and a note, lists methods in the same order (formula rules and simple baselines → machine learning without → with CCME-consistency features) and uses the same metric names; bold marks the best value as stated in each note.

| Table | Content | Section |
|---:|---|---|
| 1 | Records per country in the dataset as distributed | §1 |
| 2 | Descriptive statistics of the eight water-quality parameters (all records, before cleaning) | §2 |
| 3 | Range checks on the raw records | §3 |
| 4 | Duplicates and repeated same-day measurements (after range checks) | §3 |
| 5 | Validation of the imputation-detection rules, by country and parameter | §4b |
| 6 | Station audit: monitoring stations vs aggregate identifiers (after cleaning) | §4c |
| 7 | Audit of the published label and of the corrected single-sample label | §5b |
| 8 | Objectives of the corrected label, their basis, and the alternatives tested | §5b |
| 9 | Reporting units of nitrate: consistency check against total nitrogen | §5b |
| 10 | Experiment panel per country | §6 |
| 11 | How often each laboratory test fails its objective (genuinely measured values) | §6 |
| 12 | Circularity check: how much of the label can be read directly from the inputs | §7 |
| 13 | Leakage audit of the evaluation protocol | §9b |
| 14 | Main results on unseen test countries (tier A, no labels from the test country) | §10 |
| 15 | All eleven metrics for the main methods (tier A) | §10 |
| 16 | Evaluation data of each test country (tier A) | §10 |
| 17 | Results by test country (tier A, 2 laboratory tests) | §10 |
| 18 | Results by evaluation tier (2 laboratory tests) | §10 |
| 19 | Results on subsets of the tier-A samples (2 laboratory tests, fixed pair) | §10 |
| 20 | Per-class results: black box vs proposed (tier A, 2 laboratory tests) | §10 |
| 21 | Effect of the number of labelled target-country samples (tier A) | §10 |
| 22 | Probability calibration (tier A, 2 laboratory tests) | §10b |
| 23 | Which laboratory tests to run: fixed vs adaptive vs random selection (proposed model, tier A) | §11 |
| 24 | Differences between selection strategies (proposed model, tier A) | §11 |
| 25 | Laboratory tests chosen by the adaptive policy | §11 |
| 26 | Statistical comparison of methods: macro-F1 | §12 |
| 27 | Statistical comparison of methods: accuracy | §12 |
| 28 | Variability between training repeats (tier A) | §12 |
| 29 | All baselines on the same test records (tier A, 2 laboratory tests, no labels from the test country) | §12b |
| 30 | Is any method better than the formula rule? Paired differences to the rule (tier A) | §12b |
| 31 | Test country completely excluded from training (0 % target labels), by country (tier A, 2 laboratory tests) | §12b |
| 32 | Accuracy when fewer laboratory tests are run (tier A, same fixed test sets for every method) | §12c |
| 33 | Accuracy by age and availability of the last laboratory value (tier A, 2 laboratory tests) | §12c |
| 34 | Flagging hidden failures: which samples have a failing unmeasured test? (tier A, 2 laboratory tests) | §12c |
| 35 | Selective full-panel testing: accuracy vs share of samples given the two remaining tests (tier A) | §12c |
| 36 | Laboratory cost and turnaround per sample under each testing policy | §12c |
| 37 | Reduced-panel water-quality classification: this study and published approaches | §12c |
| 38 | What the models rely on: attribution by feature group (England, tier A) | §13 |
| 39 | Ten most important features of each model (England, tier A) | §13 |
| 40 | Stability of explanations across test countries (proposed model) | §13b |
| 41 | Attribution by feature group in each test country (proposed model) | §13b |
| 42 | Importance of each measurement: exact measurement-level Shapley values (England, tier A) | §13c |
| 43 | Do the explanations follow the CCME objectives? Domain-consistency test (England, tier A) | §13c |
| 44 | Agreement between LIME and measurement-level SHAP explanations (England, tier A) | §13d |
| 45 | Ablation study (tier A, 2 laboratory tests) | §14 |
| 46 | Zero-shot transfer to an unseen country: deep-learning baselines vs proposed (tier A, 2 laboratory tests) | §14b |
| 47 | Few-shot adaptation: K labelled samples per class from the new country (tier A) | §14b |
| 48 | Few-shot: paired accuracy differences on identical support sets | §14b |
| 49 | The prior-work setting: Canadian subset as distributed, published label | §14c |
| 50 | Prior-work protocol with all labels: random vs station-grouped split (Canada, published label) | §14c |
| 51 | Prior-work episodic few-shot protocol (Canada, published label, all eight inputs) | §14c |
| 52 | How much the label depends on each objective | §15 |
| 53 | Robustness checks: formula rule, black box and proposed model under each variant (tier A, 2 laboratory tests) | §15 |
| 54 | Robustness checks: advantage of the proposed model in macro-F1 | §15 |
| 55 | Data as distributed, by country (all samples, imputed values kept) | §15 |
| 56 | Quantum reservoir computing (QRC) vs classical models with identical inputs (tier A) | §16 |
| 57 | QRC: paired differences in macro-F1 | §16 |
| 58 | Changes relative to v13 (effect of the v14 corrections) | §16b |
| 59 | Summary of the main results (every value is taken from the tables above) | §17 |
| 60 | Claim check: which statements the results support | §17 |

In [ ]:
# !pip install -q xgboost shap seaborn pyarrow   # uncomment on a fresh environment

In [ ]:
import numpy as np, pandas as pd, warnings, time, os, itertools, sys, json, gc, hashlib, pickle
import matplotlib.pyplot as plt, seaborn as sns, shap
from scipy import stats as sps
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score,
                             matthews_corrcoef, cohen_kappa_score, log_loss, confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score, average_precision_score, roc_curve, silhouette_score)
from sklearn.manifold import TSNE
from scipy.sparse import csr_matrix
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
%matplotlib inline

# =============================== CONFIGURATION ===============================
PIPELINE_VERSION = "v14-r1"          # bump when the code changes, so old checkpoints can never be reused
CSV             = "Combined_dataset.csv" if os.path.exists("Combined_dataset.csv") else "/mnt/user-data/uploads/Combined_dataset.csv"
SEED            = 42
DO_MIN          = 5.5                # CCME freshwater DO minimum, warm-water other life stages (mg/L)
PH_RANGE        = (6.5, 9.0)         # CCME/CCREM freshwater pH range
MAX_GAP_DAYS    = 400
MAX_REC_PER_DAY = 2                  # identifiers with a higher median number of records per day are aggregates, not stations
IMPUTE_SHARE    = 0.05               # heuristic: a constant covering > 5 % of a country's values is treated as imputed
CAP_PER_COUNTRY = 60_000             # examples per country in the experiment pool
TRAIN_CAP       = 20_000             # source training rows per repeat (identical for every model)
VAL_CAP, TEST_CAP = 4_000, 8_000
BUDGETS         = [0.0, 0.10, 0.20, 0.50]   # earliest share of the target country's history that is labelled
MAIN_BUDGET     = 0.0                # headline: no sample of the test country is used (unseen countries); budgets > 0 in §10
N_REPEATS       = 3                  # training repeats (training variability); inference uses the site bootstrap (§12)
AUG_COPIES      = 2
MS              = 12                 # Monte-Carlo draws per candidate test (adaptive selection)
NB, DENS_ITER   = 20, 100
XGB_GRID        = [dict(max_depth=6, n_estimators=300, learning_rate=.08), dict(max_depth=8, n_estimators=300, learning_rate=.08)]
W_GRID          = [1.0, 5.0, 20.0]
MIN_TIER_A_EVAL = 50                 # min genuinely measured evaluation samples for a primary test country
TRAIN_TIER      = "B"                # rows whose labels may train/tune/support a model: "A" = all 8 values genuine, "B" = the 4
                                     # laboratory values genuine (no imputed laboratory value in a label), "all" = v13 behaviour
TRAIN_TIER_VARIANTS = ["A", "all"]   # re-evaluated in §15e
LAST_MAX_AGE_DAYS = 400              # unmeasured tests enter the rule/proxy at their last GENUINE value of at most this age (§6)
VAL_OF          = {"England": "Ireland", "Ireland": "England", "Canada": "Ireland", "USA": "Ireland", "China": "Ireland"}
N_BOOT          = 1000
FRESHWATER      = ["River", "Lake", "Canal"]
XAI_COUNTRY     = "England"
RUN_ROBUSTNESS, RUN_QRC = True, True
SENS_VARIANTS   = {"DO ≥ 5.0 mg/L (v10 value)": ("DO", (5.0, None)),
                   "DO ≥ 6.5 mg/L (CCME cold-water)": ("DO", (6.5, None)),
                   "pH ≤ 8.5 (published-label upper limit)": ("pH", (6.5, 8.5)),
                   "Nitrate ≤ 13 mg/L (CCME long-term)": ("NO3", (None, 13.0)),
                   "Nitrate ≤ 10 mg/L (Health Canada drinking-water MAC, expressed as N)": ("NO3", (None, 10.0)),
                   "Nitrate ≤ 2.94 mg/L (CCME long-term, expressed as N)": ("NO3", (None, 2.94))}
IMPUTATION_VARIANTS = ["documented", "heuristic_2", "heuristic_5", "heuristic_10"]   # primary rule = "union"
# ---- v13 additions: phosphate/ammonia objectives (sensitivity, §15d) and laboratory fees (practical value, §12c) ----
THRESH_VARIANTS = {"PO₄ ≤ 0.035 mg/L (Ireland, good status, mean)": ("PO4", 0.035),               # S.I. 272/2009 as amended (S.I. 77/2019), mg P/L
                   "PO₄ ≤ 0.075 mg/L (Ireland, good status, 95th percentile)": ("PO4", 0.075),
                   "PO₄ ≤ 0.10 mg/L (US EPA 1986, flowing waters)": ("PO4", 0.10),                # Quality Criteria for Water 1986
                   "NH₃ ≤ 0.14 mg/L (Ireland, good status, 95th percentile)": ("NH3", 0.14),       # mg N/L
                   "NH₃: CCME guideline, pH- and temperature-dependent": ("NH3", "ccme")}          # CCME 2010: 0.019 mg/L un-ionized NH3
LAB_PRICES = {   # US$ per analysis; total N (Oklahoma) = TKN 42 + nitrate/nitrite 38 + calculation 10
    "Oklahoma DEQ State Environmental Laboratory, fee schedule FY2027": dict(BOD=66.0, NH3=40.0, PO4=25.0, N=90.0),
    "Montana DPHHS Environmental Laboratory, fee schedule 2024": dict(BOD=60.0, NH3=23.0, PO4=23.0, N=37.0)}
N_QUBITS, R_RES = 8, 1
V_STEPS, DT, SCALE, RESERVOIR_SEED = 3, 1.0, float(np.pi / 2), 7   # QRC defaults (tuned per country in §16)
# ---- v12 additions: deep baselines, few-shot adaptation, prior-work replication, explainability ----
RUN_DEEP        = True
TF_DIM, TF_LAYERS, TF_HEADS = 32, 2, 4       # tabular transformer (feature tokens + [CLS])
DEEP_TRAIN_CAP, DEEP_EPOCHS = 10_000, 15     # source rows and epochs for every deep model (CPU budget; disclosed)
KSHOTS          = [1, 5, 10]                 # labelled target-country samples per class (few-shot adaptation)
KSHOT_EPISODES  = 3                          # support sets per K and country (identical for every method)
GAN_EPOCHS, GAN_SYN = 500, 30                # class-specific WGAN-GP (original: 1000 epochs, 30 synthetic samples per class)
PRIOR_EPISODES, PRIOR_PRETRAIN_EPOCHS = 10, 50   # replication of the prior-work protocol (Canada, published label)
XAI_N, XAI_BG, LIME_N = 300, 25, 60          # measurement-level Shapley samples, background size, LIME explanations
QRC_SEEDS       = [7, 21, 55]
QRC_GRID        = [(dt, sc, v) for dt in (0.5, 1.0) for sc in (np.pi / 2, np.pi) for v in (2, 3)]
QRC_TRAIN_CAP, QRC_EVAL_CAP, QRC_TUNE_N = 6_000, 2_000, (2_000, 1_000)
FIG_DIR, TAB_DIR = "figures_v14", "tables_v14"

CONFIG = {k: v for k, v in dict(globals()).items() if k.isupper() and not k.startswith("_")}
PIPELINE_ID = PIPELINE_VERSION + "-" + hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:10]
CKPT_DIR = os.path.join("checkpoints", PIPELINE_ID)
for d in (FIG_DIR, TAB_DIR, CKPT_DIR): os.makedirs(d, exist_ok=True)
json.dump(CONFIG, open(f"{TAB_DIR}/config.json", "w"), indent=1, default=str)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "axes.titleweight": "bold"})
def savefig(name, fig=None):
    fig = fig or plt.gcf(); fig.savefig(f"{FIG_DIR}/{name}.png", dpi=300, bbox_inches="tight"); fig.savefig(f"{FIG_DIR}/{name}.pdf", bbox_inches="tight")
def savetab(d, name):
    d.to_csv(f"{TAB_DIR}/{name}.csv", index=False)
    try: open(f"{TAB_DIR}/{name}.tex", "w").write(d.to_latex(index=False, float_format=lambda v: "%.4f" % v))
    except Exception as e: print("  (LaTeX skipped:", e, ")")
def ck_load(name):
    # versioned checkpoints: an entry is reused only if it was written by exactly this pipeline (code version + configuration)
    p = os.path.join(CKPT_DIR, name + ".pkl")
    if not os.path.exists(p): return None
    with open(p, "rb") as fh: blob = pickle.load(fh)
    assert blob.get("pipeline") == PIPELINE_ID, f"checkpoint {p} belongs to another pipeline"
    return blob["obj"]
def ck_save(name, obj):
    p = os.path.join(CKPT_DIR, name + ".pkl")
    with open(p + ".tmp", "wb") as fh: pickle.dump({"pipeline": PIPELINE_ID, "obj": obj}, fh)
    os.replace(p + ".tmp", p)
others = [d for d in (os.listdir("checkpoints") if os.path.isdir("checkpoints") else []) if d != PIPELINE_ID]
CLASSES = ["Poor", "Marginal", "Fair", "Good", "Excellent"]; K = 5
F = ["NH3", "BOD", "DO", "PO4", "pH", "T", "N", "NO3"]
LABS, SENS = ["BOD", "NH3", "PO4", "N"], ["DO", "pH", "T", "NO3"]
LIX, SIX = [F.index(f) for f in LABS], [F.index(f) for f in SENS]
def to_class(s): return np.digitize(s, [45, 65, 80, 95])
T0 = time.time()
print(f"pipeline {PIPELINE_ID} | checkpoints in {CKPT_DIR} | other (ignored) checkpoint folders: {others or 'none'}")

In [ ]:
# ================================ Presentation of results (tables) ================================
# Every results table in this notebook is drawn by show_table() from the variables computed above it.
# Conventions used in all tables:
#   * tables are numbered (Table 1, 2, ...) and carry a title and a note;
#   * methods always appear in the same order: simple baselines -> formula rules -> black box -> proposed;
#   * metrics have plain names and a fixed order; accuracy is in %, the other scores are on a 0-1 scale;
#   * bold marks the best value as stated in each table's note.
# Full-precision machine-readable tables are written by savetab() to TAB_DIR; the formatted tables
# below are also written to TAB_DIR/paper_tables/ (one CSV per table, named by table number).
import html as _html
from IPython.display import display

COUNTRY_ORDER = ["England", "Ireland", "Canada", "USA", "China"]
PNAME = {"NH3": "NH₃", "PO4": "PO₄", "NO3": "NO₃", "BOD": "BOD", "N": "N", "DO": "DO", "pH": "pH", "T": "T"}
PLABEL = {"DO": "Dissolved oxygen (DO, mg/L)", "pH": "pH", "T": "Temperature (T, °C)", "NO3": "Nitrate (NO₃, mg/L)",
          "BOD": "Biochemical oxygen demand (BOD, mg/L)", "NH3": "Ammonia (NH₃, mg/L)",
          "PO4": "Orthophosphate (PO₄, mg/L)", "N": "Nitrogen (N, mg/L)"}
G_BASE, G_RULE = "Simple baselines", "Formula rules (no learning)"
G_ML, G_PROP = "Machine-learning baseline", "Proposed (XGBoost + CCME-consistency features)"
METHODS = {   # internal name: (row group, display name, laboratory tests measured at time t) -- this order is used everywhere
    "Majority class":                       (G_BASE, "Majority class", 0),
    "Persistence":                          (G_BASE, "Persistence (class at t−1)", 0),
    "Forecast only (history)":              (G_BASE, "Forecast from site history only", 0),
    "Formula rule (assume pass) – 2 tests": (G_RULE, "Rule: unmeasured tests pass", 2),
    "Formula rule (last value) – 2 tests":  (G_RULE, "Rule: unmeasured tests at last genuine value", 2),
    "Black box – 2 tests":                  (G_ML, "Black box (XGBoost, no CCME features)", 2),
    "Proposed – 1 tests adaptive":          (G_PROP, "Proposed, 1 test (adaptive)", 1),
    "Proposed – 2 tests fixed":             (G_PROP, "Proposed, 2 tests (fixed pair)", 2),
    "Proposed – 2 tests adaptive":          (G_PROP, "Proposed, 2 tests (adaptive)", 2)}
OTHER_NAMES = {"Proposed – 0 tests": "Proposed, 0 tests", "Black box – 0 tests": "Black box, 0 tests",
               "Proposed – 2 tests random": "Proposed, 2 tests (random pair)",
               "Ablation: without monotonicity constraint": "Proposed without the monotonicity constraint",
               "Ablation: without history": "Proposed without site history",
               "Ablation: without current sensors": "Proposed without current field sensors"}
GROUP_LABEL = {"CCME consistency": "CCME-consistency features", "history": "Site history",
               "acquired lab tests": "Measured laboratory tests", "current sensors": "Current field sensors",
               "context": "Context (water body, season)"}
DASH = "–"
def budget_txt(b=None):
    b = MAIN_BUDGET if b is None else b
    return "no labels from the test country" if b == 0 else f"{100 * b:.0f} % target labels"

# Table numbers follow the order of the tables in the notebook; notes refer to other tables through tref()
TABLE_KEYS = ['overview', 'desc', 'range', 'dups', 'impval', 'station', 'label_audit', 'objectives', 'units', 'panel', 'failshare', 'circ', 'leakage', 'headline', 'metrics11', 'evaldata', 'percountry', 'tiers', 'subsets', 'perclass', 'budgets', 'calib', 'selection', 'selection_diff', 'adaptive_use', 'stats_f1', 'stats_acc', 'repeats', 'xbase', 'xbase_tests', 'zeroshot_country', 'fewer_tests', 'gap_groups', 'hidden_detect', 'triage', 'cost', 'positioning', 'shap_groups', 'topfeat', 'xai_countries', 'xai_groups_country', 'meas_shap', 'domain_cons', 'lime', 'ablation', 'zeroshot_deep', 'fewshot', 'fewshot_diff', 'prior_facts', 'prior_full', 'prior_episodic', 'robust_labels', 'robust_models', 'robust_gains', 'asdist', 'qrc', 'qrc_diff', 'repro', 'summary', 'claims']
TN = {k: i + 1 for i, k in enumerate(TABLE_KEYS)}
def tref(*ks):
    n = sorted(TN[k] for k in ks)
    if len(n) == 1: return f"Table {n[0]}"
    if len(n) == 2 and n[1] == n[0] + 1: return f"Tables {n[0]}–{n[1]}"
    return "Tables " + ", ".join(map(str, n))


def mname(m): return METHODS[m][1] if m in METHODS else OTHER_NAMES.get(m, m)
def method_rows(names):
    """Methods present in `names`, in the fixed order, and their (group, display name) row index."""
    keys = [m for m in METHODS if m in set(names)]
    return keys, pd.MultiIndex.from_tuples([METHODS[m][:2] for m in keys])
def countries_in(values):
    s = set(values); return [c for c in COUNTRY_ORDER if c in s] + sorted(s - set(COUNTRY_ORDER))
def fixed_pair_text():
    try:
        pairs = {c: " + ".join(PNAME[LABS[j]] for j in MAINSTORE[c]["FIXED"][2]) for c in PRIMARY}
        return pairs[PRIMARY[0]] if len(set(pairs.values())) == 1 else "; ".join(f"{c}: {p}" for c, p in pairs.items())
    except Exception:
        return "see §9"


# ---------- number formatting (values are formatted, never recomputed) ----------
def _isnan(v):
    try: return v is None or bool(pd.isna(v))
    except (TypeError, ValueError): return False
def f_num(v, d=4, sign=False): return DASH if _isnan(v) else (f"{float(v):+.{d}f}" if sign else f"{float(v):.{d}f}")
def f_pct(v, d=2, sign=False): return DASH if _isnan(v) else f_num(100 * float(v), d, sign)      # fraction -> %
def f_int(v): return DASH if _isnan(v) else f"{int(round(float(v))):,}"
def f_auto(v, d=2):
    if _isnan(v): return DASH
    s = f"{float(v):.{d}f}".rstrip("0"); return s + "0" if s.endswith(".") else s
def ci_bounds(s): a, b = (float(x) for x in str(s).strip("[] ").split(",")); return a, b
def f_ci(s, d=4, scale=1.0, sign=False):
    if _isnan(s): return DASH
    a, b = ci_bounds(s); return f"[{f_num(a * scale, d, sign)}, {f_num(b * scale, d, sign)}]"
def pm_parts(s): m, sd = (float(x) for x in str(s).split("±")); return m, sd
def f_pm(s, scale=1.0, d=4):
    if _isnan(s): return DASH
    m, sd = pm_parts(s); return f"{m * scale:.{d}f} ± {sd * scale:.{d}f}"
def f_p(p, floor=None, d=4):
    if _isnan(p): return DASH
    if floor is not None and float(p) <= floor + 1e-12: return f"≤ {floor:g}"
    return f"{float(p):.{d}f}"
def fmt_list(values, fn=f_num): return [fn(v) for v in values]


# ---------- which cells are printed in bold ----------
def best_mask(num, higher=True, cols=None, by_group=False, exclude=()):
    """Best value of each column (ties included), optionally within each row group; `num` is a numeric
    DataFrame shaped like the displayed table (NaN = not ranked); `exclude` = row labels not ranked."""
    M = np.zeros(num.shape, bool)
    labs = np.asarray(num.index.get_level_values(num.index.nlevels - 1)); ok = ~np.isin(labs, list(exclude))
    if by_group and isinstance(num.index, pd.MultiIndex):
        g = np.asarray(num.index.get_level_values(0)); blocks = [g == v for v in pd.unique(g)]
    else:
        blocks = [np.ones(len(num), bool)]
    for j, c in enumerate(num.columns):
        if cols is not None and c not in cols: continue
        hb = higher if isinstance(higher, bool) else higher.get(c, True)
        v = pd.to_numeric(num[c], errors="coerce").to_numpy(float)
        for b in blocks:
            s = b & ok & ~np.isnan(v)
            if s.sum() < 2: continue
            best = v[s].max() if hb else v[s].min()
            M[s & np.isclose(v, best, rtol=0, atol=1e-12), j] = True
    return M
def better_of(num, pairs, higher=True):
    """For each column pair (a, b): bold the better of the two values in every row (ties: neither)."""
    M = np.zeros(num.shape, bool); cols = list(num.columns)
    for a, b in pairs:
        va, vb = num[a].to_numpy(float), num[b].to_numpy(float)
        M[:, cols.index(a)] |= (va > vb) if higher else (va < vb)
        M[:, cols.index(b)] |= (vb > va) if higher else (vb < va)
    return M


# ---------- rendering ----------
_TCSS = """
#{id} {{margin: .4em 0 1.8em 0; max-width: 100%; overflow-x: auto;}}
#{id} table {{border-collapse: collapse; border: none; margin: 0; font-size: 12.5px; font-variant-numeric: tabular-nums;
  table-layout: auto; color: inherit; background: transparent;}}
#{id} caption {{caption-side: top; text-align: left; font-size: 13.5px; padding: 0 0 6px 0; color: inherit; white-space: normal;}}
#{id} th, #{id} td {{padding: 3px 10px; border: none; text-align: right; vertical-align: top; white-space: nowrap;
  background: transparent; color: inherit; font-size: inherit; line-height: 1.35;}}
#{id} tr {{background: transparent !important;}}
#{id} thead th {{font-weight: 600; text-align: center; vertical-align: bottom; white-space: normal; max-width: 9.5em;}}
#{id} thead th.span, #{id} thead th.stub {{max-width: none;}}
#{id} thead th.stub {{text-align: left;}}
#{id} thead tr:first-child th {{border-top: 2px solid currentColor;}}
#{id} thead tr:last-child th, #{id} thead th.rs {{border-bottom: 1px solid currentColor;}}
#{id} thead th.span {{border-bottom: 1px solid rgba(127,127,127,.6);}}
#{id} tbody th {{text-align: left; font-weight: normal;}}
#{id} tbody tr.in th {{padding-left: 22px;}}
#{id} tbody tr.grp td {{text-align: left; font-weight: 600; font-style: italic; padding-top: 7px;
  border-top: 1px solid rgba(127,127,127,.4);}}
#{id} tbody tr.grp:first-child td {{border-top: none;}}
#{id} tbody tr:last-child th, #{id} tbody tr:last-child td {{border-bottom: 2px solid currentColor;}}
#{id} td.l {{text-align: left; white-space: normal; min-width: 10em; max-width: 36em;}}
#{id} tbody tr:not(.grp):hover th, #{id} tbody tr:not(.grp):hover td {{background: rgba(127,127,127,.10);}}
#{id} p.note {{font-size: 11.5px; opacity: .85; margin: 6px 0 0 0; max-width: 1050px; line-height: 1.45; white-space: normal;}}
"""
PAPER_TABLES = {}


class TableView:
    """Rich (HTML) and plain-text representation of one numbered table."""
    def __init__(self, html_, text_): self._html, self._text = html_, text_
    def _repr_html_(self): return self._html
    def __repr__(self): return self._text


def show_table(key, title, body, note=None, bold=None, stub="", left=(), bold_rows=()):
    """body: DataFrame of display strings. A two-level row index (group, label) is drawn as group rows;
    two-level columns are drawn as spanning headers ("" as second level = single header cell)."""
    num = TN[key]
    esc = lambda s: _html.escape(str(s))
    cols = list(body.columns); two = isinstance(body.columns, pd.MultiIndex) and any(b not in ("", a) for a, b in cols)
    grouped = isinstance(body.index, pd.MultiIndex)
    B = np.zeros(body.shape, bool) if bold is None else np.asarray(bold, bool)
    assert B.shape == body.shape, (num, B.shape, body.shape)
    tid = f"wqt-{num}"
    h = [f'<div class="wqt" id="{tid}"><style>{_TCSS.format(id=tid)}</style><table>',
         f"<caption><b>Table {num}.</b> {esc(title)}</caption><thead>"]
    if two:
        top = [c[0] for c in cols]; sub = [c[1] for c in cols]
        r1, r2, i = [f'<th class="stub rs" rowspan="2">{esc(stub)}</th>'], [], 0
        while i < len(cols):
            if sub[i] in ("", top[i]):
                r1.append(f'<th class="rs" rowspan="2">{esc(top[i])}</th>'); i += 1; continue
            j = i
            while j + 1 < len(cols) and top[j + 1] == top[i] and sub[j + 1] not in ("", top[j + 1]): j += 1
            r1.append(f'<th class="span" colspan="{j - i + 1}">{esc(top[i])}</th>')
            r2 += [f"<th>{esc(sub[k])}</th>" for k in range(i, j + 1)]; i = j + 1
        h.append("<tr>" + "".join(r1) + "</tr><tr>" + "".join(r2) + "</tr>")
    else:
        heads = [c[0] if isinstance(c, tuple) else c for c in cols]
        h.append(f'<tr><th class="stub">{esc(stub)}</th>' + "".join(f"<th>{esc(c)}</th>" for c in heads) + "</tr>")
    h.append("</thead><tbody>")
    last_g = object()
    for r, key in enumerate(body.index):
        g, lab = (key[0], key[-1]) if grouped else (None, key)
        if grouped and g != last_g:
            h.append(f'<tr class="grp"><td colspan="{len(cols) + 1}">{esc(g)}</td></tr>'); last_g = g
        rb = lab in bold_rows
        cells = []
        for c in range(len(cols)):
            v = esc(body.iat[r, c]); v = f"<b>{v}</b>" if (B[r, c] or rb) else v
            cells.append(f'<td class="l">{v}</td>' if cols[c] in left else f"<td>{v}</td>")
        h.append(f'<tr class="{"in" if grouped else "row"}"><th>{"<b>" + esc(lab) + "</b>" if rb else esc(lab)}</th>' + "".join(cells) + "</tr>")
    h.append("</tbody></table>")
    if note: h.append(f'<p class="note"><i>Note.</i> {esc(note)}</p>')
    h.append("</div>")
    # plain-text version (for viewers without HTML) and CSV export
    t = body.copy()
    t.index = pd.Index([f"{k[0]} | {k[-1]}" for k in t.index] if grouped else list(t.index), name=stub or None)
    t.columns = [(a if b in ("", a) else f"{a}: {b}") for a, b in cols] if isinstance(body.columns, pd.MultiIndex) else cols
    text = f"Table {num}. {title}\n{t.to_string()}" + (f"\nNote. {note}" if note else "")
    PAPER_TABLES[num] = title
    try:
        os.makedirs(os.path.join(TAB_DIR, "paper_tables"), exist_ok=True)
        t.to_csv(os.path.join(TAB_DIR, "paper_tables", f"table_{num:02d}.csv"), encoding="utf-8-sig")
    except Exception:
        pass
    display(TableView("".join(h), text))


# ---------- readable feature names (Table 26) ----------
_FEAT = {"ccme_proxy": "CCME index, unmeasured tests at last value", "class_proxy": "CCME class, unmeasured tests at last value",
         "nfail_proxy": "failed objectives, unmeasured tests at last value",
         "excursion_proxy": "CCME excursion term, unmeasured tests at last value",
         "ccme_ceiling": "CCME index if unmeasured tests pass (upper bound)", "class_ceiling": "CCME class if unmeasured tests pass (upper bound)",
         "nfail_known": "failed objectives among measured values", "excursion_known": "CCME excursion term of measured values",
         "ccme_t-1": "CCME index at the previous visit", "class_t-1": "class at the previous visit",
         "ccme_proxy_minus_t-1": "change of the CCME index since the previous visit",
         "expected_hidden_exceedances": "expected number of failing unmeasured tests (site history)",
         "p_any_hidden_exceedance": "probability that an unmeasured test fails (site history)",
         "hist_log_n": "number of earlier visits (log)", "y_l1": "class at t−1", "y_l2": "class at t−2",
         "gap1": "days since the previous visit", "gap2": "days between the two previous visits",
         "n_tests": "number of laboratory tests measured", "m_sin": "season (sine of month)", "m_cos": "season (cosine of month)"}
_FSUF = [("_hist_exceed_rate", "{p}: share of earlier visits failing its objective"), ("_upper_ratio", "{p} ÷ upper objective"),
         ("_lower_ratio", "lower objective ÷ {p}"), ("_t-1_ratio", "{p}: last genuine value ÷ objective"), ("_t_ratio", "{p} at t ÷ objective"),
         ("_t", "{p} measured at t"), ("_l1", "{p} at the previous visit (t−1)"), ("_l2", "{p} two visits back (t−2)"),
         ("_d", "{p} change between t−2 and t−1")]
def feat_label(code):
    code = str(code)
    if code in _FEAT: return f"{_FEAT[code]} [{code}]"
    if code.startswith("wb_"): return f"water body: {code[3:]} [{code}]"
    for suf, tpl in _FSUF:
        if code.endswith(suf) and code[: -len(suf)] in PNAME: return f"{tpl.format(p=PNAME[code[: -len(suf)]])} [{code}]"
    return code


def inf_pairs(inf, comp):
    """Attach the compared methods (pair_a, pair_b) to the rows of INF, which follow the order of COMP."""
    n_m = len(inf) // len(comp); assert n_m * len(comp) == len(inf)
    pa = [p for p in comp for _ in range(n_m)]
    for (a, b), s in zip(pa, inf.comparison):
        assert f"{a} vs {b}".startswith(str(s).rstrip(".").rstrip()), (a, b, s)
    return inf.assign(pair_a=[a for a, _ in pa], pair_b=[b for _, b in pa])


## 1. Dataset and initial inspection

In [ ]:
RENAME = {"Country": "country", "Area": "site", "Waterbody Type": "wb", "Date": "date",
          "Ammonia (mg/l)": "NH3", "Biochemical Oxygen Demand (mg/l)": "BOD", "Dissolved Oxygen (mg/l)": "DO",
          "Orthophosphate (mg/l)": "PO4", "pH (ph units)": "pH", "Temperature (cel)": "T",
          "Nitrogen (mg/l)": "N", "Nitrate (mg/l)": "NO3", "CCME_Values": "ccme", "CCME_WQI": "wqi"}
df = pd.read_csv(CSV, low_memory=False).rename(columns=RENAME)
df["dt"] = pd.to_datetime(df.date, format="%d-%m-%Y", errors="coerce"); df = df.drop(columns=["date"])
df["y_pub"] = df.wqi.map({c: i for i, c in enumerate(CLASSES)}).astype("int8")
CANADA_RAW = df[df.country == "Canada"].reset_index(drop=True).copy()   # as distributed; used for the prior-work replication (§14c)
print(f"shape {df.shape} | unparsed dates {int(df.dt.isna().sum())}"); display(df.head())
overview = df.groupby("country").agg(records=("site", "size"), identifiers=("site", "nunique"), first=("dt", "min"), last=("dt", "max"))

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    o = overview.reindex(countries_in(overview.index))
    first, last = pd.to_datetime(o["first"]), pd.to_datetime(o["last"])
    t = pd.DataFrame({"Records": fmt_list(o.records, f_int), "Site identifiers": fmt_list(o.identifiers, f_int),
                      "First sample": list(first.dt.strftime("%Y-%m-%d")), "Last sample": list(last.dt.strftime("%Y-%m-%d"))}, index=list(o.index))
    t.loc["All countries"] = [f_int(o.records.sum()), f_int(o.identifiers.sum()), first.min().strftime("%Y-%m-%d"), last.max().strftime("%Y-%m-%d")]
    show_table("overview", "Records per country in the dataset as distributed", t, stub="Country", bold_rows=["All countries"],
               note="Site identifiers as given in the file; §4c separates monitoring stations from aggregate identifiers "
                    "(e.g. China's single identifier).")
_present()

## 2. Exploratory data analysis

In [ ]:
samp = df.sample(min(400_000, len(df)), random_state=SEED)
desc = df[F].describe(percentiles=[.01, .5, .99]).T.round(3)
LOGF = ["NH3", "BOD", "PO4", "N", "NO3"]
fig, axes = plt.subplots(2, 4, figsize=(20, 7.5)); axes = axes.ravel()
for a, f in zip(axes, F):
    sns.histplot(samp[f].clip(lower=1e-3) if f in LOGF else samp[f], bins=80, log_scale=f in LOGF, ax=a, color="#4c72b0")
    a.set_title(f"{f} (skew {sps.skew(samp[f]):.1f})"); a.set_ylabel("")
plt.tight_layout(); savefig("fig01_distributions"); plt.show()
fig, ax = plt.subplots(1, 3, figsize=(24, 6.5), gridspec_kw={"width_ratios": [1.1, 1, 1]})
cm_ = samp[F].corr(method="spearman")
sns.heatmap(cm_, mask=np.triu(np.ones_like(cm_, bool), 1), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True, ax=ax[0])
ax[0].set_title("(a) Spearman correlation")
df.country.value_counts().plot(kind="bar", ax=ax[1], rot=0, color="#4c72b0"); ax[1].set_yscale("log"); ax[1].set_title("(b) Records per country")
(pd.crosstab(df.country, df.wqi, normalize="index")[CLASSES] * 100).plot(kind="barh", stacked=True, colormap="RdYlGn", ax=ax[2])
ax[2].set_title("(c) Published class mix"); ax[2].legend(fontsize=7, bbox_to_anchor=(1, 1))
plt.tight_layout(); savefig("fig02_eda"); plt.show()
del samp; _ = gc.collect()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    order = [("Field sensors (measured on site at time t)", f) for f in SENS] + [("Laboratory tests (hidden unless the test is run)", f) for f in LABS]
    d = desc.loc[[f for _, f in order]]
    t = pd.DataFrame({"Records": fmt_list(d["count"], f_int),
                      **{lab: fmt_list(d[src], lambda v: f_num(v, 3)) for lab, src in
                         [("Mean", "mean"), ("SD", "std"), ("Min", "min"), ("1st percentile", "1%"), ("Median", "50%"),
                          ("99th percentile", "99%"), ("Max", "max")]}},
                     index=pd.MultiIndex.from_tuples([(g, PLABEL[f]) for g, f in order]))
    show_table("desc", "Descriptive statistics of the eight water-quality parameters (all records, before cleaning)", t, stub="Parameter",
               note="Records outside physical ranges are removed in " + tref("range") + ". The CCME class at time t is computed from all eight "
                    "parameters at time t; the model sees the field sensors and only the laboratory tests that are run.")
_present()

## 3. Data preprocessing
### 4a. Range checks, duplicates and repeated measurements (issue 1)
Physically impossible values are removed. Exact duplicates are removed on **(country, site, date, all eight values)**.
Records that share a country, site and date but differ in value are *repeated measurements* (e.g. several samples or depths
on one day). They are audited here and later aggregated to one value per site-day (median); a single-record subset is
reported as a sensitivity analysis (§10).

In [ ]:
rules = {"pH outside 0–14": ~df.pH.between(0, 14), "negative concentration": (df[["NH3", "BOD", "PO4", "N", "NO3"]] < 0).any(axis=1),
         "temperature > 40 °C": df["T"] > 40, "temperature < -2 °C": df["T"] < -2, "DO > 20 mg/L": df.DO > 20}
VALID = ~np.logical_or.reduce(list(rules.values()))
inv = pd.DataFrame({"check": list(rules) + ["ANY (removed)"], "rows": [int(m.sum()) for m in rules.values()] + [int((~VALID).sum())]})
savetab(inv, "tab_range_checks")
KEYS = ["country", "site", "dt"]
v = df.loc[VALID]
dup = v.duplicated(subset=KEYS + F)
print(f"identifiers shared by more than one country: {int((df.groupby('site').country.nunique() > 1).sum())}")
raw = v.loc[~dup.to_numpy(), KEYS + ["wb", "ccme", "y_pub"] + F].reset_index(drop=True)
nrec = raw.groupby(KEYS).DO.transform("size")
multi = raw[nrec.to_numpy() > 1]
g_ = multi.groupby(KEYS)[["DO", "T", "pH"]]; cv = g_.std() / g_.mean().abs().clip(lower=1e-9)   # within-day coefficient of variation
rep_tab = pd.DataFrame({
    "records": raw.groupby("country").size(), "exact_duplicates_removed": dup.groupby(v.country).sum(),
    "site_days": raw.groupby(KEYS).size().groupby(level=0).size(),
    "site_days_with_repeats_%": (raw.groupby(KEYS).size() > 1).groupby(level=0).mean() * 100,
    "max_records_per_site_day": raw.groupby(KEYS).size().groupby(level=0).max(),
    "within_day_CV_DO": cv.DO.groupby(level=0).median(), "within_day_CV_T": cv["T"].groupby(level=0).median()}).round(3)
savetab(rep_tab.reset_index(), "tab_duplicates_repeats")
n_raw = len(df); del df, v, multi, cv, g_; _ = gc.collect()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    lab = {"pH outside 0–14": "pH outside 0–14", "negative concentration": "Negative concentration",
           "temperature > 40 °C": "Temperature > 40 °C", "temperature < -2 °C": "Temperature < −2 °C",
           "DO > 20 mg/L": "Dissolved oxygen > 20 mg/L", "ANY (removed)": "Any check failed (records removed)"}
    t = pd.DataFrame({"Records flagged": fmt_list(inv.rows, f_int)}, index=[lab.get(c, c) for c in inv.check])
    show_table("range", "Range checks on the raw records", t, stub="Check", bold_rows=["Any check failed (records removed)"],
               note="A record can fail several checks, so the rows do not add up to the total.")
    r = rep_tab.reindex(countries_in(rep_tab.index))
    t = pd.DataFrame({("Records", "kept"): fmt_list(r.records, f_int),
                      ("Records", "exact duplicates removed"): fmt_list(r.exact_duplicates_removed, f_int),
                      ("Site-days", "total"): fmt_list(r.site_days, f_int),
                      ("Site-days", "with repeated records (%)"): fmt_list(r["site_days_with_repeats_%"], lambda v: f_num(v, 2)),
                      ("Site-days", "max records per site-day"): fmt_list(r.max_records_per_site_day, f_int),
                      ("Within-day CV (median)", "DO"): fmt_list(r.within_day_CV_DO, lambda v: f_num(v, 3)),
                      ("Within-day CV (median)", "T"): fmt_list(r.within_day_CV_T, lambda v: f_num(v, 3))}, index=list(r.index))
    show_table("dups", "Duplicates and repeated same-day measurements (after range checks)", t, stub="Country",
               note="Exact duplicate = same country, site, date and all eight values. Repeated records of one site-day are "
                    "median-aggregated to a single value (§5). CV = coefficient of variation between the records of one site-day.")
_present()

### 4b. Imputation: documentation, detection rules and validation (issue 4)
**Dataset documentation** (Karim et al., *Scientific Data* 2025): missing parameter values were filled with the
**country-wise median** of the parameter, and rows with more than three missing parameters were dropped. Two rules are compared:
- **Documented rule (D):** value equals its country's median for that parameter.
- **Dominant-constant rule (H<sub>t</sub>):** value equals a constant that covers more than *t* of the country's values.

**Primary rule = D ∪ H<sub>5 %</sub>** — the documented imputation value plus any other heavily repeated constant (e.g. a
reporting-limit value, or DO = 11.6841 which is not a country median). Flags are computed per raw record before aggregation and
use no labels. **Validation:** a value that was imputed appears far more often than its numeric neighbours; the *spike ratio* is
the count of the flagged constant divided by the mean count of its 10 nearest distinct values, and 1 / spike ratio estimates
the share of flagged values that could be genuine coincidences.

In [ ]:
MED = raw.groupby("country")[F].median()
SHARES = {c: {f: raw.loc[raw.country == c, f].value_counts(normalize=True) for f in F} for c in MED.index}
def raw_flags(rule):
    out = np.zeros((len(raw), len(F)), bool); cc = raw.country.to_numpy()
    for j, f in enumerate(F):
        x = raw[f].to_numpy()
        for c in MED.index:
            s = cc == c
            if rule in ("documented", "union"): out[s, j] |= np.isclose(x[s], MED.loc[c, f], rtol=0, atol=1e-9)
            if rule == "union" or rule.startswith("heuristic"):
                t = IMPUTE_SHARE if rule == "union" else int(rule.split("_")[1]) / 100
                for val in SHARES[c][f][SHARES[c][f] > t].index: out[s, j] |= np.isclose(x[s], val, rtol=0, atol=1e-9)
    return out
RULES = ["union"] + IMPUTATION_VARIANTS
RAWFLAGS = {r: raw_flags(r) for r in RULES}
val_rows = []
for c in MED.index:
    for j, f in enumerate(F):
        vc = SHARES[c][f]; top = vc.index[0]; cnt = raw.loc[raw.country == c, f].value_counts()
        others_ = cnt.drop(index=top); near = others_.index[np.argsort(np.abs(others_.index.to_numpy() - top))[:10]]
        spike = cnt[top] / max(others_[near].mean(), 1)
        s = (raw.country == c).to_numpy(); D, H = RAWFLAGS["documented"][s, j], RAWFLAGS["heuristic_5"][s, j]
        val_rows.append(dict(country=c, parameter=f, country_median=MED.loc[c, f], dominant_value=top, dominant_share_pct=100 * vc.iloc[0],
                             dominant_equals_median=bool(np.isclose(top, MED.loc[c, f])), spike_ratio=spike,
                             est_genuine_share_of_flags_pct=100 / max(spike, 1), documented_pct=100 * D.mean(), heuristic5_pct=100 * H.mean(),
                             primary_union_pct=100 * RAWFLAGS["union"][s, j].mean(),
                             agreement_jaccard=(D & H).sum() / max((D | H).sum(), 1)))
imp_val = pd.DataFrame(val_rows); savetab(imp_val, "tab_imputation_validation")
print(f"dominant value equals the documented imputation value (country median) in {imp_val.dominant_equals_median.sum()}/{len(imp_val)} "
      f"country-parameter pairs; median spike ratio {imp_val.spike_ratio.median():.1f}")
fig, ax = plt.subplots(1, 2, figsize=(19, 3.8))
for a, col, t in [(ax[0], "primary_union_pct", "(a) % of values flagged as imputed (primary rule)"), (ax[1], "spike_ratio", "(b) spike ratio of the dominant constant (log scale)")]:
    pv = imp_val.pivot(index="country", columns="parameter", values=col)[F]
    sns.heatmap(np.log10(pv) if col == "spike_ratio" else pv, annot=pv.round(0 if col != "spike_ratio" else 0), fmt="g", cmap="rocket_r", ax=a); a.set_title(t)
plt.tight_layout(); savefig("fig03_imputation_validation"); plt.show()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    d = imp_val.round(3); rows, idx = [], []
    for c in countries_in(d.country):
        for f in SENS + LABS:
            r = d[(d.country == c) & (d.parameter == f)]
            if len(r): rows.append(r.iloc[0]); idx.append((c, PLABEL[f]))
    d = pd.DataFrame(rows)
    t = pd.DataFrame({("Country median", ""): fmt_list(d.country_median, lambda v: f_num(v, 3)),
                      ("Most frequent value", "value"): fmt_list(d.dominant_value, lambda v: f_num(v, 3)),
                      ("Most frequent value", "share (%)"): fmt_list(d.dominant_share_pct, lambda v: f_num(v, 1)),
                      ("Most frequent value", "= median?"): ["yes" if bool(v) else "no" for v in d.dominant_equals_median],
                      ("Spike ratio", ""): [DASH if _isnan(v) else f"{v:,.1f}" for v in d.spike_ratio],
                      ("Est. genuine share of flags (%)", ""): fmt_list(d.est_genuine_share_of_flags_pct, lambda v: f_num(v, 2)),
                      ("Flagged as imputed (%)", "documented rule"): fmt_list(d.documented_pct, lambda v: f_num(v, 1)),
                      ("Flagged as imputed (%)", "constant > 5 %"): fmt_list(d.heuristic5_pct, lambda v: f_num(v, 1)),
                      ("Flagged as imputed (%)", "primary rule"): fmt_list(d.primary_union_pct, lambda v: f_num(v, 1)),
                      ("Rule agreement (Jaccard)", ""): fmt_list(d.agreement_jaccard, lambda v: f_num(v, 3))},
                     index=pd.MultiIndex.from_tuples(idx))
    show_table("impval", "Validation of the imputation-detection rules, by country and parameter", t, stub="Parameter",
               note="Documented rule = value equals the country median (the dataset's documented imputation). Spike ratio = frequency of "
                    "the most frequent value ÷ mean frequency of its ten nearest values (large = artificial constant); estimated genuine "
                    "share of flagged values = 100 ÷ spike ratio. Primary rule = documented rule ∪ dominant-constant rule "
                    f"(> {100 * IMPUTE_SHARE:.0f} %). Jaccard = agreement between the documented and the 5 % rule.")
_present()

### 4c. Station identifiers (issue 3)
The `Area` field is not defined in the dataset documentation. An identifier is treated as a **monitoring station** only if it
records at most `MAX_REC_PER_DAY` records on a typical day (median over its sampling days). Identifiers with many records per day
mix several sampling points and are **aggregates**; they are excluded from the station-level time-series analysis and appear
only in the supplementary as-distributed analysis (§15a).

In [ ]:
sd_n = raw.groupby(KEYS).size().rename("nrec").reset_index()
idn = sd_n.groupby(["country", "site"]).nrec.agg(days="size", median_records_per_day="median", max_records_per_day="max").reset_index()
idn["station"] = idn.median_records_per_day <= MAX_REC_PER_DAY
st_tab = idn.groupby("country").agg(identifiers=("site", "size"), station_identifiers=("station", "sum"),
                                     median_records_per_day=("median_records_per_day", "median"), max_records_per_day=("max_records_per_day", "max"))
st_tab["site_days_from_stations_%"] = 100 * idn.assign(w=idn.days * idn.station).groupby("country").w.sum() / idn.groupby("country").days.sum()
savetab(st_tab.reset_index(), "tab_station_audit")
print("aggregate (non-station) identifiers:", idn.loc[~idn.station, ["country", "site", "median_records_per_day"]].head(10).to_dict("records"),
      f"... ({int((~idn.station).sum())} in total)")
STATION_ID = set(map(tuple, idn.loc[idn.station, ["country", "site"]].to_numpy()))

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    s = st_tab.round(2).reindex(countries_in(st_tab.index))
    t = pd.DataFrame({("Identifiers", "all"): fmt_list(s.identifiers, f_int),
                      ("Identifiers", "stations"): fmt_list(s.station_identifiers, f_int),
                      ("Records per identifier-day", "median"): fmt_list(s.median_records_per_day, f_auto),
                      ("Records per identifier-day", "max"): fmt_list(s.max_records_per_day, f_int),
                      ("Site-days from stations (%)", ""): fmt_list(s["site_days_from_stations_%"], lambda v: f_num(v, 2))}, index=list(s.index))
    show_table("station", "Station audit: monitoring stations vs aggregate identifiers (after cleaning)", t, stub="Country",
               note=f"Station = identifier with a median of at most {MAX_REC_PER_DAY} records per day; identifiers with more are "
                    "aggregates and are excluded from the station-level analysis.")
_present()

## 5. CCME formula: verification, guideline values and label correction (issue 2)
**Standard CCME WQI** (CCME 2001, WQI 1.0 technical report/user's manual) for one site and period, with *P* variables and all
tests of the period: F1 = 100 × (variables failing at least once)/*P*; F2 = 100 × (failed tests)/(total tests);
excursion = value/objective − 1 (objective is a maximum) or objective/value − 1 (objective is a minimum);
nse = Σ excursions/(total tests); F3 = nse/(0.01·nse + 0.01); WQI = 100 − √(F1² + F2² + F3²)/1.732.
Categories: Excellent 95–100, Good 80–94, Fair 65–79, Marginal 45–64, Poor 0–44.

**Record-level (single-sample) form.** With one sample, total tests = *P* = 8, so F1 = F2 = 100·n<sub>fail</sub>/8 and nse = Σ excursions/8.
This is exactly the formula used for the labels; a unit test below checks it against a direct implementation of the standard
multi-sample formula. **Caveat:** CCME recommends ≥ 4 variables and ≥ 4 samples per year per site. The dataset (and this study)
applies the index per daily record, as the dataset authors did; the label is therefore a *single-sample CCME-type index*.
A standard site-year CCME index is computed below for comparison.

**Guideline objectives used for the corrected label (CCME-c).**
The objectives, their sources and the alternative values tested in the sensitivity analysis are listed in §5b (objectives table); five of the eight objectives are those of the published label and are not CCME guidelines.

The published label treated DO and pH only as **maxima** and divided by 12 instead of 8 (§5 audit). Freshwater objectives do not
strictly apply to marine, estuarine or effluent samples; a freshwater-ambient subset (river, lake, canal) is reported in §10.

In [ ]:
GUIDE_PUB = {"NH3": (None, .5), "BOD": (None, 6), "DO": (None, 14), "PO4": (None, .05), "pH": (None, 8.5), "T": (None, 25), "N": (None, 20), "NO3": (None, 45)}
GUIDE = {**GUIDE_PUB, "DO": (DO_MIN, None), "pH": PH_RANGE}
def ccme_parts(V, guide=None, n_den=8):
    # record-level CCME: V maps parameter -> array of values (NaN = unknown, treated as meeting the objective)
    guide = GUIDE if guide is None else guide
    shape = np.shape(next(iter(V.values()))); nf = np.zeros(shape); exc = np.zeros(shape)
    for f, (lo, hi) in guide.items():
        x = np.asarray(V[f], float)
        with np.errstate(invalid="ignore", divide="ignore"):
            if hi is not None: m = x > hi; nf += m; exc += np.where(m, x / hi - 1, 0)
            if lo is not None: m = x < lo; nf += m; exc += np.where(m, lo / np.maximum(x, 1e-3) - 1, 0)
    nse = exc / n_den
    return 100 - np.sqrt(2 * (100 * nf / n_den) ** 2 + (nse / (.01 * nse + .01)) ** 2) / 1.732, nf, exc
def ccme_standard(X, guide=None):
    # standard CCME WQI for ONE site-period: X has shape (n_samples, 8) in the order of F
    guide = GUIDE if guide is None else guide
    fail = np.zeros(X.shape, bool); exc = np.zeros(X.shape)
    for j, f in enumerate(F):
        lo, hi = guide[f]; x = X[:, j]
        if hi is not None: m = x > hi; fail[:, j] |= m; exc[:, j] += np.where(m, x / hi - 1, 0)
        if lo is not None: m = x < lo; fail[:, j] |= m; exc[:, j] += np.where(m, lo / np.maximum(x, 1e-3) - 1, 0)
    F1 = 100 * fail.any(0).sum() / X.shape[1]; F2 = 100 * fail.sum() / X.size; nse = exc.sum() / X.size
    return 100 - np.sqrt(F1 ** 2 + F2 ** 2 + (nse / (.01 * nse + .01)) ** 2) / 1.732
# unit test: the record-level formula equals the standard formula applied to a single sample
tst = raw.sample(2000, random_state=1)[F].to_numpy(float)
rec_ = ccme_parts({f: tst[:, j] for j, f in enumerate(F)})[0]; std_ = np.array([ccme_standard(r[None, :]) for r in tst])
assert np.allclose(rec_, std_, atol=1e-9); print(f"unit test passed: record-level formula = standard CCME for single samples (max |diff| {np.abs(rec_ - std_).max():.1e})")
# audit of the published label
s_pub = ccme_parts({f: raw[f].to_numpy(float) for f in F}, GUIDE_PUB, 12)[0]
AUDIT = dict(raw=n_raw, after_cleaning=len(raw), reproduced_values=float((np.abs(s_pub - raw.ccme) < .01).mean()),
             reproduced_classes=float((to_class(s_pub) == raw.y_pub).mean()),
             hypoxic_excellent=int(((raw.DO < 2) & (raw.y_pub == 4)).sum()), acidic_excellent=int(((raw.pH < PH_RANGE[0]) & (raw.y_pub == 4)).sum()))
y_c_raw = to_class(ccme_parts({f: raw[f].to_numpy(float) for f in F})[0])
AUDIT["class_changed_by_correction_pct"] = float(100 * (y_c_raw != raw.y_pub.to_numpy()).mean())
print(f"published label reproduced with the 12-denominator, maxima-only formula: values {100*AUDIT['reproduced_values']:.3f}% | classes {100*AUDIT['reproduced_classes']:.3f}%")
print(f"published label: {AUDIT['hypoxic_excellent']:,} hypoxic (DO<2) and {AUDIT['acidic_excellent']:,} acidic records labelled Excellent | "
      f"correction (CCME DO/pH, denominator 8) changes {AUDIT['class_changed_by_correction_pct']:.1f}% of classes")
pd.DataFrame([AUDIT]).to_csv(f"{TAB_DIR}/tab_label_audit.csv", index=False)

In [ ]:
# site-day table (one value per country, site and day; repeated measurements -> median) with imputation bitmasks
grp = raw.groupby(KEYS, sort=True)
sd = grp[F].median().reset_index()
sd["wb"] = grp.wb.first().to_numpy(); sd["nrec"] = grp.size().to_numpy()
sd_key = grp.ngroup().to_numpy()
SD_BITS = {}
for r in RULES:                                         # bit j set = parameter F[j] imputed in at least one record of that site-day
    fl = pd.DataFrame(RAWFLAGS[r], columns=F).groupby(sd_key).max().to_numpy()
    SD_BITS[r] = (fl * (1 << np.arange(len(F)))).sum(1).astype(np.uint8)
sd["station"] = [(c, s) in STATION_ID for c, s in zip(sd.country, sd.site)]
del RAWFLAGS, grp, sd_key; _ = gc.collect()
# standard CCME per station site-year (>= 4 sampling days) vs. the record-level index
sd["score_rec"] = ccme_parts({f: sd[f].to_numpy(float) for f in F})[0]; sd["year"] = sd.dt.dt.year
fail = np.zeros((len(sd), len(F)), bool); exc = np.zeros((len(sd), len(F)))
for j, f in enumerate(F):
    lo, hi = GUIDE[f]; x = sd[f].to_numpy(float)
    if hi is not None: m = x > hi; fail[:, j] |= m; exc[:, j] += np.where(m, x / hi - 1, 0)
    if lo is not None: m = x < lo; fail[:, j] |= m; exc[:, j] += np.where(m, lo / np.maximum(x, 1e-3) - 1, 0)
G = pd.DataFrame(fail, columns=F).assign(n=1, exc=exc.sum(1), nfail=fail.sum(1), score_rec=sd.score_rec, key=sd.country + "|" + sd.site + "|" + sd.year.astype(str))
G = G[sd.station.to_numpy()]
agg = G.groupby("key").agg(n=("n", "sum"), exc=("exc", "sum"), nfail=("nfail", "sum"), mean_rec=("score_rec", "mean"), **{f: (f, "max") for f in F})
agg = agg[agg.n >= 4]
F1 = 100 * agg[F].sum(1) / len(F); F2 = 100 * agg.nfail / (len(F) * agg.n); nse = agg.exc / (len(F) * agg.n)
agg["standard"] = 100 - np.sqrt(F1 ** 2 + F2 ** 2 + (nse / (.01 * nse + .01)) ** 2) / 1.732
rho = sps.spearmanr(agg.standard, agg.mean_rec).statistic; agree = (to_class(agg.standard) == to_class(agg.mean_rec)).mean()
STD_CMP = dict(site_years=len(agg), spearman=rho, class_agreement=agree, mean_standard=agg.standard.mean(), mean_record_level=agg.mean_rec.mean())
print(f"standard site-year CCME (n = {len(agg):,} station site-years with >= 4 samples) vs mean record-level index: "
      f"Spearman {rho:.3f}, class agreement {100*agree:.1f}%, mean {agg.standard.mean():.1f} vs {agg.mean_rec.mean():.1f}")
print("-> the record-level index is related to, but not the same as, the standard CCME; results refer to the record-level (single-sample) index")
pd.DataFrame([STD_CMP]).to_csv(f"{TAB_DIR}/tab_standard_vs_record_ccme.csv", index=False)
del fail, exc, G, agg; _ = gc.collect()

### 5b. Label audit: class bands, objectives with sources, and reporting units
The published label is reproduced exactly by a formula that treats DO and pH only as maxima and divides by 12 (§5); the corrected
label (CCME-c) uses the CCME DO minimum and pH range and the record-level denominator 8. This section documents (i) that the
published classes follow the official CCME category limits, (ii) the source of every objective and the alternative values used in
the sensitivity analysis (§15d), and (iii) a consistency check of the nitrate and total-nitrogen reporting units, which the dataset
does not state (Karim et al., 2025).

In [ ]:
# class bands of the published label
BOUNDS = pd.DataFrame([dict(bands=nm_, published_class_reproduced_pct=100 * (np.digitize(raw.ccme.to_numpy(float), cuts) == raw.y_pub.to_numpy()).mean())
                       for nm_, cuts in [("CCME limits 45 / 65 / 80 / 95 (Excellent ≥ 95)", [45, 65, 80, 95]),
                                         ("alternative reading 45 / 65 / 80 / 90 (Excellent ≥ 90)", [45, 65, 80, 90])]])
savetab(BOUNDS, "tab_label_class_bands")
# objectives used for CCME-c, their sources, and the alternatives tested in §15b/§15d
OBJECTIVES = pd.DataFrame([
    dict(parameter="DO", objective="≥ 5.5 mg/L", kind="minimum", source="CCME (1999) freshwater aquatic life: warm water, other life stages",
         alternatives="≥ 5.0 mg/L (v10 value); ≥ 6.5 mg/L (CCME cold water, other life stages)"),
    dict(parameter="pH", objective="6.5–9.0", kind="range", source="CCME (1999) freshwater aquatic life",
         alternatives="upper limit 8.5 (published label)"),
    dict(parameter="NO3", objective="≤ 45 mg/L", kind="maximum", source="published label; equals the drinking-water maximum of 45 mg/L as NO₃ (Health Canada), not an aquatic-life value",
         alternatives="≤ 13 mg/L (CCME 2012 long-term, as NO₃); ≤ 2.94 mg/L (the same guideline expressed as N)"),
    dict(parameter="NH3", objective="≤ 0.5 mg/L", kind="maximum", source="published label; CCME (2010) sets 0.019 mg/L un-ionized NH₃, so the total-ammonia guideline depends on pH and temperature",
         alternatives="≤ 0.14 mg/L N (Ireland S.I. 272/2009 as amended, good status, 95th percentile); CCME pH- and temperature-dependent guideline"),
    dict(parameter="PO4", objective="≤ 0.05 mg/L", kind="maximum", source="published label; equals the US EPA (1986) recommendation of 0.05 mg/L P for streams entering lakes; CCME has no numeric P guideline (2004 trigger-range framework)",
         alternatives="≤ 0.035 and ≤ 0.075 mg/L P (Ireland, good status, mean and 95th percentile); ≤ 0.10 mg/L (US EPA 1986, flowing waters)"),
    dict(parameter="BOD", objective="≤ 6 mg/L", kind="maximum", source="published label; no CCME guideline", alternatives="–"),
    dict(parameter="T", objective="≤ 25 °C", kind="maximum", source="published label; CCME temperature guidance is narrative", alternatives="–"),
    dict(parameter="N", objective="≤ 20 mg/L", kind="maximum", source="published label; no CCME guideline", alternatives="–")])
savetab(OBJECTIVES, "tab_objectives_sources")
# reporting units: nitrate-N is part of total N, so nitrate reported as N cannot exceed total N (beyond analytical noise);
# nitrate reported as NO3 (x 62.0/14.0 = 4.43 times the N value) often does. Genuinely measured site-days only.
_imp = ((SD_BITS["union"][:, None] >> np.arange(len(F))) & 1).astype(bool)
_ok = ~_imp[:, F.index("N")] & ~_imp[:, F.index("NO3")] & (sd.N.to_numpy() > 0)
unit_rows = []
for c in [c for c in ["England", "Ireland", "Canada", "USA", "China"] if c in set(sd.country)]:
    s_ = (sd.country.to_numpy() == c) & _ok; no3, tn = sd.NO3.to_numpy()[s_], sd.N.to_numpy()[s_]
    if s_.sum() == 0:
        unit_rows.append(dict(country=c, site_days=0)); continue
    a_, b_ = 100 * (no3 > 1.05 * tn).mean(), 100 * (no3 * 14.0067 / 62.0049 > 1.05 * tn).mean()
    unit_rows.append(dict(country=c, site_days=int(s_.sum()), median_nitrate=np.median(no3), median_total_N=np.median(tn),
                          nitrate_above_total_N_pct=a_, nitrate_converted_to_N_above_total_N_pct=b_,
                          consistent_with=("nitrate as N" if a_ < 10 else "nitrate as NO₃" if b_ < 10 else "unclear")))
UNITS = pd.DataFrame(unit_rows); savetab(UNITS, "tab_units_check")
print("units check (genuine site-days):", dict(zip(UNITS.country, UNITS.get("consistent_with", ["–"] * len(UNITS)))))

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    A, S_ = AUDIT, STD_CMP; b = BOUNDS.set_index("bands").published_class_reproduced_pct
    rows = [("Published label (as distributed)", "Values reproduced by the formula with DO and pH as maxima and denominator 12 (%)", f_num(100 * A["reproduced_values"], 2)),
            ("Published label (as distributed)", "Classes reproduced by that formula (%)", f_num(100 * A["reproduced_classes"], 2)),
            ("Published label (as distributed)", "Classes reproduced with the CCME limits 45 / 65 / 80 / 95 (%)", f_num(b.iloc[0], 2)),
            ("Published label (as distributed)", "Classes reproduced with Excellent ≥ 90 instead (%)", f_num(b.iloc[1], 2)),
            ("Published label (as distributed)", "Hypoxic records (DO < 2 mg/L) labelled Excellent", f_int(A["hypoxic_excellent"])),
            ("Published label (as distributed)", f"Acidic records (pH < {PH_RANGE[0]}) labelled Excellent", f_int(A["acidic_excellent"])),
            ("Corrected label (CCME-c)", "Classes changed by the correction (%)", f_num(A["class_changed_by_correction_pct"], 1)),
            ("Corrected label (CCME-c)", "Record-level formula equals the standard CCME formula applied to one sample", "yes (unit test, §5)"),
            ("Single-sample vs standard site-year CCME", "Station site-years with at least 4 samples", f_int(S_["site_years"])),
            ("Single-sample vs standard site-year CCME", "Spearman correlation of the two indices", f_num(S_["spearman"], 3)),
            ("Single-sample vs standard site-year CCME", "Class agreement (%)", f_num(100 * S_["class_agreement"], 1)),
            ("Single-sample vs standard site-year CCME", "Mean index: site-year vs single-sample", f"{S_['mean_standard']:.1f} vs {S_['mean_record_level']:.1f}")]
    t = pd.DataFrame({"Value": [v for _, _, v in rows]}, index=pd.MultiIndex.from_tuples([(g, k) for g, k, _ in rows]))
    show_table("label_audit", "Audit of the published label and of the corrected single-sample label", t, stub="Item",
               note="CCME (2001; user's manual 2017): the index should not be calculated with fewer than four parameters and four sampling visits "
                    "per year; the dataset applies it to single records. The label used here is therefore a single-sample CCME-type index; "
                    "the comparison with the standard site-year index quantifies the difference.")
    o = OBJECTIVES.set_index("parameter"); order = [("Field sensors", p) for p in SENS] + [("Laboratory tests", p) for p in LABS]
    t = pd.DataFrame({"Objective": [o.loc[p, "objective"] for _, p in order], "Type": [o.loc[p, "kind"] for _, p in order],
                      "Basis": [o.loc[p, "source"] for _, p in order], "Alternatives tested (§15)": [o.loc[p, "alternatives"] for _, p in order]},
                     index=pd.MultiIndex.from_tuples([(g, PLABEL[p]) for g, p in order]))
    show_table("objectives", "Objectives of the corrected label, their basis, and the alternatives tested", t, stub="Parameter",
               left=["Basis", "Alternatives tested (§15)"],
               note="Sources: CCME Canadian Water Quality Guidelines for the Protection of Aquatic Life (dissolved oxygen 1999; ammonia 2010; nitrate 2012); "
                    "CCME (2004) phosphorus guidance framework; Health Canada drinking-water guideline for nitrate; US EPA (1986) Quality Criteria "
                    "for Water (phosphorus); Ireland S.I. No. 272/2009 as amended by S.I. No. 77/2019, Schedule 5 (river standards in mg P/L and "
                    "mg N/L). The dataset does not state whether NH₃, PO₄ and nitrate are reported as N/P or as the ion; the alternatives "
                    "assume N and P.")
    u = UNITS.set_index("country")
    t = pd.DataFrame({"Site-days": fmt_list(u.site_days, f_int), ("Median (mg/L)"): fmt_list(u.get("median_nitrate", pd.Series(np.nan, u.index)), lambda v: f_num(v, 2)),
                      "Median total N (mg/L)": fmt_list(u.get("median_total_N", pd.Series(np.nan, u.index)), lambda v: f_num(v, 2)),
                      "Nitrate > total N (%)": fmt_list(u.get("nitrate_above_total_N_pct", pd.Series(np.nan, u.index)), lambda v: f_num(v, 1)),
                      "Nitrate as N > total N (%)": fmt_list(u.get("nitrate_converted_to_N_above_total_N_pct", pd.Series(np.nan, u.index)), lambda v: f_num(v, 1)),
                      "Consistent with": [str(v) if isinstance(v, str) else DASH for v in u.get("consistent_with", pd.Series(np.nan, u.index))]},
                     index=list(u.index))
    t = t.rename(columns={"Median (mg/L)": "Median nitrate (mg/L)"})
    show_table("units", "Reporting units of nitrate: consistency check against total nitrogen", t, stub="Country", left=["Consistent with"],
               note="Genuinely measured site-days only. Nitrate-N is part of total N, so nitrate reported as N cannot exceed total N; nitrate reported "
                    "as NO₃ is 4.43 times its N value and often does (5 % tolerance for analytical noise). 'Unclear' = neither reading fits "
                    "most site-days. Indicative only: the source agencies' reporting conventions should be confirmed. The primary nitrate "
                    "objective (45 mg/L) is rarely exceeded, so the primary label is insensitive to this; the CCME aquatic-life value is "
                    "tested in both units in §15b.")
_present()

## 6. Panel construction grouped by (country, site) (issue 3)
Site-days are ordered within each **(country, site)** series. An example at time *t* needs the two previous sampling days of the
same series (gaps ≤ `MAX_GAP_DAYS`). History features: values at *t−1* and *t−2*, their CCME classes, the gaps, and the
**long-run site history** — for each parameter, the share of all earlier genuinely measured site-days of the series that exceeded
the objective, plus the number of earlier site-days (strictly before *t*; given to every learned model). An unmeasured laboratory
test enters the formula rule and the proxy features at its **last genuine value**: the most recent genuinely measured value of the
series before *t*, at most `LAST_MAX_AGE_DAYS` old (missing if there is none). Rows whose label may train a model are restricted
to `TRAIN_TIER` (§8). Station series form the main panel; aggregate identifiers (China) are kept only for the
supplementary as-distributed analysis.

In [ ]:
# sd is already ordered by (country, site, date) because it was built with groupby(..., sort=True); SD_BITS share this order
key_c, key_s = sd.country.to_numpy(), sd.site.to_numpy()
same1 = np.r_[False, (key_c[1:] == key_c[:-1]) & (key_s[1:] == key_s[:-1])]           # previous row is the same (country, site) series
same2 = np.r_[False, False, same1[2:] & same1[1:-1]]
dtv = sd.dt.to_numpy()
gap1 = np.r_[np.nan, (dtv[1:] - dtv[:-1]) / np.timedelta64(1, "D")]; gap2 = np.r_[np.nan, gap1[:-1]]
eligible = same1 & same2 & (gap1 > 0) & (gap1 <= MAX_GAP_DAYS) & (gap2 > 0) & (gap2 <= MAX_GAP_DAYS)
assert not np.any(eligible & ~same1), "an example must never mix two (country, site) series"
ex = pd.DataFrame({"i": np.where(eligible)[0]}); ex["country"] = key_c[ex.i]
print("eligible examples per country:", ex.country.value_counts().to_dict())
pool_i = np.sort(np.concatenate([d.i.sample(min(len(d), CAP_PER_COUNTRY), random_state=SEED).to_numpy() for _, d in ex.groupby("country")]))
I0, I1, I2 = pool_i, pool_i - 1, pool_i - 2
SDF = sd[F].to_numpy(float)
pool = sd.loc[I0, ["country", "site", "dt", "wb", "nrec", "station"]].reset_index(drop=True)
pool["gap1"], pool["gap2"] = gap1[I0], gap2[I0]
pool["m_sin"], pool["m_cos"] = np.sin(2 * np.pi * pool.dt.dt.month / 12), np.cos(2 * np.pi * pool.dt.dt.month / 12)
WB = sorted(pool.wb.unique())
for w in WB: pool[f"wb_{w}"] = (pool.wb == w).astype(float)
ctry, site_ids = pool.country.to_numpy(), pool.site.to_numpy()
STATION = pool.station.to_numpy(); FRESH = pool.wb.isin(FRESHWATER).to_numpy(); SINGLE = (pool.nrec == 1).to_numpy()
days = pool.dt.to_numpy().astype("datetime64[D]").astype("int64")
HIST_COLS = [f"{f}_hist_exceed_rate" for f in F] + ["hist_log_n"]
H_COLS = [f"{f}_l1" for f in F] + [f"{f}_l2" for f in F] + [f"{f}_d" for f in F] + ["y_l1", "y_l2", "gap1", "gap2"] + HIST_COLS
_first = np.where(~same1)[0]; START = _first[np.cumsum(~same1) - 1]       # first site-day of each row's (country, site) series
S_COLS, L_COLS = [f"{f}_t" for f in SENS], [f"{f}_t" for f in LABS]
CTX_COLS = [f"wb_{w}" for w in WB] + ["m_sin", "m_cos"]
BASE_COLS = H_COLS + S_COLS + CTX_COLS
CTXM = pool[CTX_COLS].to_numpy(float)
del sd_n, idn, ex; _ = gc.collect()

def bits_to_mask(b): return ((b[:, None] >> np.arange(len(F))) & 1).astype(bool)
def site_history(protocol, rule):
    # long-run site history before t: share of earlier (genuinely measured) site-days of the same (country, site) series in which
    # each parameter exceeded its objective, and the number of earlier site-days. Uses only data strictly before t.
    gen = ~bits_to_mask(SD_BITS[rule]) if protocol == "genuine" else np.ones(SDF.shape, bool)
    exc = np.zeros(SDF.shape, bool)
    for j, f in enumerate(F):
        lo, hi = GUIDE[f]
        if hi is not None: exc[:, j] |= SDF[:, j] > hi
        if lo is not None: exc[:, j] |= SDF[:, j] < lo
    exc &= gen
    Ce = np.vstack([np.zeros((1, len(F)), np.int32), np.cumsum(exc, 0, dtype=np.int32)])
    Cg = np.vstack([np.zeros((1, len(F)), np.int32), np.cumsum(gen, 0, dtype=np.int32)])
    s0 = START[I0]; n_ex = (Ce[I0] - Ce[s0]).astype(float); n_g = (Cg[I0] - Cg[s0]).astype(float)   # rows s0 .. t-1
    with np.errstate(invalid="ignore", divide="ignore"): rate = np.where(n_g > 0, n_ex / n_g, np.nan)
    return np.c_[rate, np.log1p(I0 - s0)]
SD_DAYS = dtv.astype("datetime64[D]").astype("int64")
def last_genuine(protocol, rule):
    # most recent genuinely measured value of each laboratory test strictly before t in the same (country, site) series, at most
    # LAST_MAX_AGE_DAYS old (NaN if none). Returns the values, their site-day index (-1 = none) and their age in days.
    gen = ~bits_to_mask(SD_BITS[rule]) if protocol == "genuine" else np.ones(SDF.shape, bool)
    val, src = np.full((len(I0), len(LIX)), np.nan), np.full((len(I0), len(LIX)), -1)
    for q, j in enumerate(LIX):
        last = np.maximum.accumulate(np.where(gen[:, j], np.arange(len(SDF)), -1))[I1]       # latest genuine site-day <= t-1
        ok = (last >= START[I0]) & (SD_DAYS[I0] - SD_DAYS[np.maximum(last, 0)] <= LAST_MAX_AGE_DAYS)
        val[ok, q] = SDF[last[ok], j]; src[ok, q] = last[ok]
    age = np.where(src >= 0, SD_DAYS[I0][:, None] - SD_DAYS[np.maximum(src, 0)], np.nan)
    return val, src, age
def set_state(protocol="genuine", rule="union"):
    # builds every model input for the chosen protocol: 'genuine' hides imputed values (rule), 'raw' uses the data as distributed
    global XBASE, LABCUR, LABLAG, SENCUR, LABRAW, y_all, YL1_RAW, TIER_A, TIER_B, PROTOCOL, RULE, LABEL_OK, LAST_SRC, LAST_AGE
    X0, X1, X2 = SDF[I0], SDF[I1], SDF[I2]
    y_all = to_class(ccme_parts({f: X0[:, j] for j, f in enumerate(F)})[0])
    yl1 = to_class(ccme_parts({f: X1[:, j] for j, f in enumerate(F)})[0]).astype(float)
    yl2 = to_class(ccme_parts({f: X2[:, j] for j, f in enumerate(F)})[0]).astype(float)
    YL1_RAW = yl1.astype(int)
    if protocol == "genuine":
        m0, m1, m2 = (bits_to_mask(SD_BITS[rule][I]) for I in (I0, I1, I2))
    else:
        m0 = m1 = m2 = np.zeros(X0.shape, bool)
    X0m, X1m, X2m = np.where(m0, np.nan, X0), np.where(m1, np.nan, X1), np.where(m2, np.nan, X2)
    yl1 = np.where(m1.any(1), np.nan, yl1); yl2 = np.where(m2.any(1), np.nan, yl2)
    XBASE = np.hstack([X1m, X2m, X1m - X2m, yl1[:, None], yl2[:, None], pool[["gap1", "gap2"]].to_numpy(float), site_history(protocol, rule),
                       X0m[:, SIX], CTXM])
    LABCUR, SENCUR, LABRAW = X0m[:, LIX], X0m[:, SIX], X0[:, LIX]
    LABLAG, LAST_SRC, LAST_AGE = last_genuine(protocol, rule)     # v13: X1m[:, LIX], i.e. the value at t-1 and NaN (= pass) if imputed
    TIER_A = ~m0.any(1); TIER_B = ~m0[:, LIX].any(1); PROTOCOL, RULE = protocol, rule
    # rows whose label may train, tune or support a model: no imputed value of the training tier at t (labels are computed from
    # all eight values at t, imputed ones included, so an imputed value would enter the label while the model sees it as missing)
    LABEL_OK = ({"A": TIER_A, "B": TIER_B}.get(TRAIN_TIER, np.ones(len(I0), bool)) if protocol == "genuine" else np.ones(len(I0), bool)).copy()
set_state("genuine", "union")
assert XBASE.shape[1] == len(BASE_COLS)
summary = pd.DataFrame({"examples": pd.Series(ctry).value_counts(), "station_examples_%": pd.Series(STATION).groupby(ctry).mean() * 100,
                        "tier_A_%": pd.Series(TIER_A).groupby(ctry).mean() * 100, "tier_B_%": pd.Series(TIER_B).groupby(ctry).mean() * 100,
                        "single_record_days_%": pd.Series(SINGLE).groupby(ctry).mean() * 100, "freshwater_ambient_%": pd.Series(FRESH).groupby(ctry).mean() * 100})
late = pd.Series(days).groupby(ctry).transform(lambda d: d > d.quantile(.5)).to_numpy()
summary["tier_A_station_eval_samples"] = pd.Series(late & TIER_A & STATION).groupby(ctry).sum()
summary["label_rows_%"] = pd.Series(LABEL_OK & STATION).groupby(ctry).mean() * 100
savetab(summary.reset_index().rename(columns={"index": "country"}), "tab_panel_summary")
ALL_COUNTRIES = [c for c in ["England", "Ireland", "Canada", "USA", "China"] if c in summary.index]
PRIMARY = [c for c in ALL_COUNTRIES if summary.loc[c, "tier_A_station_eval_samples"] >= MIN_TIER_A_EVAL]
print("tier A = all 8 current values genuinely measured (primary evaluation); tier B = the 4 current laboratory values genuinely measured")
print("training sources and evaluation use station series only | primary test countries:", PRIMARY,
      "| supplementary only:", [c for c in ALL_COUNTRIES if c not in PRIMARY])
print(f"rows usable as training labels (station series, TRAIN_TIER = {TRAIN_TIER}):", pd.Series(LABEL_OK & STATION).groupby(ctry).sum().to_dict())
fail_share = pd.DataFrame({f: pd.Series(np.where(np.isnan(LABCUR[:, j]), np.nan, LABCUR[:, j] > GUIDE[f][1])).groupby(ctry).mean()
                           for j, f in enumerate(LABS)}).round(3)        # genuinely measured values only (v13 included imputed values)

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    s = summary.round(1).reindex(countries_in(summary.index))
    t = pd.DataFrame({("Examples", ""): fmt_list(s.examples, f_int),
                      ("Share of examples (%)", "station series"): fmt_list(s["station_examples_%"], lambda v: f_num(v, 1)),
                      ("Share of examples (%)", "tier A"): fmt_list(s["tier_A_%"], lambda v: f_num(v, 1)),
                      ("Share of examples (%)", "tier B"): fmt_list(s["tier_B_%"], lambda v: f_num(v, 1)),
                      ("Share of examples (%)", "single-record days"): fmt_list(s["single_record_days_%"], lambda v: f_num(v, 1)),
                      ("Share of examples (%)", "freshwater ambient"): fmt_list(s["freshwater_ambient_%"], lambda v: f_num(v, 1)),
                      ("Share of examples (%)", f"training-label rows (station, tier {TRAIN_TIER})"): fmt_list(s["label_rows_%"], lambda v: f_num(v, 1)),
                      ("Tier-A station samples (later half)", ""): fmt_list(s.tier_A_station_eval_samples, f_int),
                      ("Role", ""): ["primary test country" if c in PRIMARY else "supplementary only" for c in s.index]}, index=list(s.index))
    show_table("panel", "Experiment panel per country", t, stub="Country", left=[("Role", "")],
               note="Tier A = all eight current values genuinely measured (primary evaluation); tier B = the four current laboratory "
                    f"values genuinely measured. A primary test country needs at least {MIN_TIER_A_EVAL} tier-A samples from station "
                    f"series in the later half of its history: {', '.join(PRIMARY)}. At most {CAP_PER_COUNTRY:,} examples per country. "
                    f"Training-label rows = station rows whose label contains no imputed value of tier {TRAIN_TIER} (§8).")
    fs = fail_share.reindex(countries_in(fail_share.index))
    t = pd.DataFrame({("Examples in which the test fails its objective (%)", PNAME[f]): fmt_list(fs[f], lambda v: f_pct(v, 1))
                      for f in LABS}, index=list(fs.index))
    show_table("failshare", "How often each laboratory test fails its objective (genuinely measured values)", t, stub="Country",
               note="All examples, genuinely measured laboratory values only (imputed country medians excluded). When a failing test is not measured, the failure is hidden from "
                    "the model (a hidden failure; results in " + tref("subsets") + ").")
_present()

## 7. Prediction task and circularity (issue 5)
**Target.** The CCME-c class at time *t* (record-level index, §5), computed from the eight parameters measured at *t*.

**What the model sees at time *t*.** The four field-sensor values at *t* (DO, pH, T, NO₃ — these *are* label inputs, by design,
because they are cheap to measure in situ), the *k* acquired laboratory values at *t*, and the site's history (*t−1*, *t−2*).
It never sees the 4 − *k* unacquired laboratory values at *t*.

**Consequences, quantified in this notebook.**
1. With all four laboratory values the label is a closed-form function of the inputs (circular). This configuration is only a
   ceiling reference (§7 table below).
2. Whenever none of the unmeasured tests actually fails its objective, applying the formula with the unmeasured tests assumed
   to pass gives the correct class by construction. **The genuinely difficult cases are the hidden failures:** samples in which
   an unmeasured test exceeds its objective. Two formula rules are therefore included as baselines, and every method is also
   evaluated on the hidden-failure subset (§10).
3. A unit test (§8) checks that no feature changes when the unacquired current laboratory values change.

In [ ]:
set_state("raw")
st_rows = np.where(STATION)[0]
a_, b_ = next(GroupShuffleSplit(1, test_size=.25, random_state=0).split(st_rows, groups=site_ids[st_rows])); a_, b_ = st_rows[a_], st_rows[b_]
circ = []
for tag, X_ in [("history only", XBASE[:, :len(H_COLS)]), ("history + current sensors", XBASE),
                ("history + sensors + ALL 4 current lab values (circular)", np.hstack([XBASE, LABCUR]))]:
    p_ = HistGradientBoostingClassifier(max_iter=300, random_state=0).fit(X_[a_], y_all[a_]).predict(X_[b_])
    circ.append(dict(inputs=tag, Accuracy=accuracy_score(y_all[b_], p_), MacroF1=f1_score(y_all[b_], p_, average="macro")))
circ.append(dict(inputs="exact formula on all 8 current values (no learning)", Accuracy=1.0, MacroF1=1.0))
circ = pd.DataFrame(circ); savetab(circ, "tab_circularity")
set_state("genuine", "union")
print("-> the configuration with all four current laboratory values is never used as a result")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    c = circ.round(4); assert len(c) == 4
    labels = ["Site history only", "Site history + current field sensors",
              "History + sensors + all four current laboratory values (circular)",
              "Exact CCME formula on all eight current values (no learning)"]
    t = pd.DataFrame({"Current laboratory values used": ["none", "none", "all four", "all four"],
                      "Accuracy (%)": fmt_list(c.Accuracy, f_pct), "Macro-F1": fmt_list(c.MacroF1, f_num)}, index=labels)
    show_table("circ", "Circularity check: how much of the label can be read directly from the inputs", t, stub="Model inputs at time t",
               note="Gradient boosting on station series, 25 % of sites held out, raw values. The last two rows use the values that "
                    "define the label; they only demonstrate the circularity and are never used as results.")
_present()

## 8. Methods
**Proposed (knowledge-guided).** XGBoost on history, current sensors, acquired laboratory values and CCME-consistency features:
proxy CCME score and class (each unacquired test at its **last genuine value**, §6), class ceiling (unacquired tests assumed to
pass), numbers and sizes of exceedances, guideline ratios, the CCME score at *t−1*, and the expected number of hidden exceedances
(sum of the long-run exceedance rates of the unacquired tests). Imputed values are hidden (missing).
**Training labels.** The label of a row is computed from its eight values at *t*. A row may train, tune or support a model only if
it has no imputed value of the training tier (`TRAIN_TIER`; default B = the four laboratory values genuinely measured). In v13 every
station row was used, so imputed country medians entered the labels while the model saw the same values as missing (e.g. England's
imputed PO₄ of 0.144 mg/L fails the 0.05 mg/L objective). Tier A and the v13 behaviour are re-evaluated in §15e.
**Monotonicity-constrained inference.** In the CCME formula an additional failed test can only lower the index, so the true class
can never be above the *ceiling* class (all unacquired tests passing). The proposed model's probabilities above the ceiling are set
to zero and renormalised. This never removes the true class. It is a safeguard rather than a source of accuracy: in v13 it changed no
predicted class; its effect is reported as an ablation (§14).
**Black box.** The same learner, inputs and acquisition pattern without CCME-consistency features or constraint.
**Formula rules (no learning).** CCME formula with unacquired tests (i) at their last genuine value or (ii) assumed to pass.
**Other baselines.** Persistence (class at *t−1*), majority class of the training data, forecast-only (history + context).
**Masked-acquisition training.** Each training row is duplicated with a random subset of 0–3 current laboratory values hidden, so
one model serves every acquisition pattern. **Adaptive selection.** Greedy expected-entropy minimisation with `MS` Monte-Carlo draws
per candidate test from laboratory-value density models (§11); every draw is evaluated on its own row (v13 overwrote the draws of
a sample, so only one draw was used). All tuning (learner, target weight, fixed test sets) uses a separate **validation country**.


In [ ]:
def featurize_arrays(xb, labcur, lablag, sencur, M, formula=True):
    # array form (used for measurement-level explanations); M: (n,4) bool acquisition mask over LABS.
    # Unacquired current laboratory values are never read.
    n = len(xb); cur = np.where(M, labcur, np.nan)
    blocks = [xb, cur, M.sum(1, keepdims=True).astype(float)]
    if formula:
        lag = lablag; sen = sencur
        V = {f: sen[:, SENS.index(f)] for f in SENS}
        for j, f in enumerate(LABS): V[f] = np.where(M[:, j], cur[:, j], lag[:, j])
        s, nf, ex = ccme_parts(V)
        Vb = {f: sen[:, SENS.index(f)] for f in SENS}
        for j, f in enumerate(LABS): Vb[f] = np.where(M[:, j], cur[:, j], 0.0)
        sb, nfb, exb = ccme_parts(Vb)
        Vl = {f: xb[:, BASE_COLS.index(f"{f}_l1")] for f in F}; s1 = ccme_parts(Vl)[0]
        ratios = []
        for f in SENS:
            lo, hi = GUIDE[f]; x = V[f]
            ratios.append(x / hi if hi else np.full(n, np.nan)); ratios.append(lo / np.maximum(x, 1e-3) if lo else np.full(n, np.nan))
        for j, f in enumerate(LABS):
            hi = GUIDE[f][1]; ratios.append(cur[:, j] / hi); ratios.append(lag[:, j] / hi)
        hr = np.nan_to_num(xb[:, [BASE_COLS.index(f"{f}_hist_exceed_rate") for f in LABS]], nan=0.0) * ~M   # unacquired tests only
        blocks += [np.c_[s, to_class(s), nf, ex, sb, to_class(sb), nfb, exb, s1, to_class(s1), s - s1, hr.sum(1), 1 - np.prod(1 - hr, 1)],
                   np.c_[tuple(ratios)]]
    return np.hstack(blocks)
def featurize(rows, M, formula=True):
    return featurize_arrays(XBASE[rows], LABCUR[rows], LABLAG[rows], SENCUR[rows], M, formula)
FORM_NAMES = ["ccme_proxy", "class_proxy", "nfail_proxy", "excursion_proxy", "ccme_ceiling", "class_ceiling", "nfail_known",
              "excursion_known", "ccme_t-1", "class_t-1", "ccme_proxy_minus_t-1", "expected_hidden_exceedances", "p_any_hidden_exceedance"] + \
             [f"{f}_{s}_ratio" for f in SENS for s in ("upper", "lower")] + [f"{f}_{s}_ratio" for f in LABS for s in ("t", "t-1")]
NAMES = {True: BASE_COLS + L_COLS + ["n_tests"] + FORM_NAMES, False: BASE_COLS + L_COLS + ["n_tests"]}
def random_mask(n, k, rng):
    r = rng.random((n, 4)).argsort(1).argsort(1); return r < (k if np.ndim(k) == 0 else k[:, None])
def fixed_mask(n, subset): M = np.zeros((n, 4), bool); M[:, list(subset)] = True; return M
def make_xgb(prm, seed=0):
    return XGBClassifier(**prm, subsample=.8, colsample_bytree=.8, min_child_weight=2, reg_lambda=1.0, tree_method="hist",
                         max_bin=256, n_jobs=-1, random_state=seed)
def fitx(m, X, y_, w=None):
    u, yy = np.unique(y_, return_inverse=True); m.fit(X, yy, sample_weight=w); m.cls_ = u; return m
def proba(m, X, K_=K):
    P = np.zeros((len(X), K_)); P[:, getattr(m, "cls_", m.classes_)] = m.predict_proba(X); return P
def train(src, lab, prm, w_t, seed, formula=True, y_vec=None):
    y_vec = y_all if y_vec is None else y_vec
    rng = np.random.default_rng(seed); rows = np.r_[src, lab]
    R = np.concatenate([rows] * AUG_COPIES); M = random_mask(len(R), rng.integers(0, 4, len(R)), rng)
    w = compute_sample_weight("balanced", y_vec[R]) * np.where(np.isin(R, lab), w_t, 1.0)
    return fitx(make_xgb(prm, seed), featurize(R, M, formula), y_vec[R], w)
def fit_density(rows, w):
    D = []
    for j in range(4):
        ok = ~np.isnan(LABCUR[rows, j]); r_, w_ = rows[ok], np.asarray(w)[ok]
        v = LABCUR[r_, j]; edges = np.unique(np.quantile(v, np.linspace(0, 1, NB + 1)))
        b = np.clip(np.searchsorted(edges, v, side="right") - 1, 0, len(edges) - 2)
        D.append((edges, HistGradientBoostingClassifier(max_iter=DENS_ITER, random_state=0).fit(XBASE[r_], b, sample_weight=w_)))
    return D
def adaptive_masks(model, D, rows, kmax, seed, formula=True):
    # greedy expected-entropy minimisation with MS Monte-Carlo values per candidate test, drawn from the density models D.
    # Each draw is evaluated on its own row of a copy of the inputs. (v13 wrote the MS draws of a sample into the same cell of the
    # global LABCUR -- LABCUR[np.repeat(rows, MS), j] = draws -- where numpy keeps only the last value, so one draw was used.)
    rng = np.random.default_rng(seed); n = len(rows); M = np.zeros((n, 4), bool); out = {}; smp = []
    for edges, m in D:
        P = np.zeros((n, len(edges) - 1)); P[:, m.classes_] = m.predict_proba(XBASE[rows]); cdf = P.cumsum(1)
        u = rng.random((n, MS)); bi = np.minimum((cdf[:, None, :] < u[:, :, None]).sum(2), len(edges) - 2)
        smp.append(edges[bi] + rng.random((n, MS)) * (edges[bi + 1] - edges[bi]))
    xb_, lab_, lag_, sen_ = XBASE[rows], LABCUR[rows], LABLAG[rows], SENCUR[rows]
    for k in range(1, kmax + 1):
        E = np.full((n, 4), np.inf)
        for j in range(4):
            idx = np.where(~M[:, j])[0]
            if len(idx) == 0: continue
            rep_ = np.repeat(idx, MS)                                      # row i * MS + d = draw d of sample i
            lab_h = lab_[rep_].copy(); lab_h[:, j] = smp[j][idx].ravel()   # hypothetical value of test j
            Mr = M[rep_].copy(); Mr[:, j] = True
            Pr = predict_arrays(model, xb_[rep_], lab_h, lag_[rep_], sen_[rep_], Mr, formula).reshape(len(idx), MS, -1)
            E[idx, j] = -(Pr * np.log(Pr + 1e-12)).sum(2).mean(1)
        M[np.arange(n), E.argmin(1)] = True; out[k] = M.copy()
    return out
def onehot(cls, eps=.02):
    P = np.full((len(cls), K), eps / (K - 1)); P[np.arange(len(cls)), cls] = 1 - eps; return P
def persistence_P(rows): return onehot(YL1_RAW[rows])
def rule_P(rows, M, kind):
    sen = SENCUR[rows]; V = {f: sen[:, SENS.index(f)] for f in SENS}
    for j, f in enumerate(LABS): V[f] = np.where(M[:, j], LABCUR[rows, j], LABLAG[rows, j] if kind == "last" else 0.0)
    return onehot(to_class(ccme_parts(V)[0]))
def ceiling_arrays(sen, labcur, M):
    # class with every unacquired laboratory test passing; unknown (hidden) values are treated as passing
    V = {f: sen[:, SENS.index(f)] for f in SENS}
    for j, f in enumerate(LABS): V[f] = np.where(M[:, j], labcur[:, j], 0.0)
    return to_class(ccme_parts(V)[0])
def ceiling_class(rows, M): return ceiling_arrays(SENCUR[rows], LABCUR[rows], M)
def constrain_c(P, c_):
    # CCME monotonicity: extra failures can only lower the index, so classes above the ceiling are impossible
    P = P * (np.arange(K)[None, :] <= c_[:, None]); s = P.sum(1)
    P[s <= 0] = onehot(c_[s <= 0]); return P / P.sum(1, keepdims=True)
def constrain(P, rows, M): return constrain_c(P, ceiling_class(rows, M))
def predict(model, rows, M, formula=True, constrained=True):
    P = proba(model, featurize(rows, M, formula))
    return constrain(P, rows, M) if (formula and constrained) else P
def predict_arrays(model, xb, labcur, lablag, sencur, M, formula=True, constrained=True):
    # predict() on explicit arrays (adaptive selection, measurement-level explanations); never touches the global arrays
    P = proba(model, featurize_arrays(xb, labcur, lablag, sencur, M, formula))
    return constrain_c(P, ceiling_arrays(sencur, labcur, M)) if (formula and constrained) else P
def sources(test_c):
    # training rows: other countries, station series, and (genuine protocol) labels without imputed values of the training tier
    return np.where((ctry != test_c) & STATION & LABEL_OK)[0] if PROTOCOL == "genuine" else np.where(ctry != test_c)[0]
def target_split(c, b):
    # target labels = earliest b of the country's history; evaluation = its latest 50 % (genuine protocol: station series, tier B)
    idx = np.where(ctry == c)[0]
    if PROTOCOL == "genuine": idx = idx[STATION[idx]]
    d = days[idx]; ev = idx[d > np.quantile(d, .5)]
    if PROTOCOL == "genuine": ev = ev[TIER_B[ev]]
    lab = idx[d <= np.quantile(d, b)] if b > 0 else np.array([], int)
    lab = lab[LABEL_OK[lab]]                                   # target labels without imputed values of the training tier
    ev = np.sort(np.random.default_rng(7).choice(ev, min(TEST_CAP, len(ev)), replace=False))
    if len(lab): assert days[lab].max() < days[ev].min(), "leakage: target labels must precede evaluation data"
    return lab, ev
def macro_auc(yt, P):
    # macro one-vs-rest ROC-AUC and PR-AUC (average precision) over the classes present in yt (with >= 1 positive and negative)
    ks = [k for k in range(K) if 0 < (yt == k).sum() < len(yt)]
    if not ks: return np.nan, np.nan
    return (np.mean([roc_auc_score(yt == k, P[:, k]) for k in ks]), np.mean([average_precision_score(yt == k, P[:, k]) for k in ks]))
def metrics9(yt, P):
    # 11 metrics (name kept for continuity): 9 label metrics + macro ROC-AUC + macro PR-AUC
    P = np.clip(P, 1e-7, 1); P = P / P.sum(1, keepdims=True); yp = P.argmax(1); auc, ap = macro_auc(yt, P)
    return dict(Accuracy=accuracy_score(yt, yp), MacroPrecision=precision_score(yt, yp, average="macro", zero_division=0),
                MacroRecall=recall_score(yt, yp, average="macro", zero_division=0), MacroF1=f1_score(yt, yp, average="macro"),
                BalancedAcc=balanced_accuracy_score(yt, yp), MCC=matthews_corrcoef(yt, yp),
                QWK=cohen_kappa_score(yt, yp, weights="quadratic", labels=list(range(K))), MeanClassDistance=np.abs(yt - yp).mean(),
                LogLoss=log_loss(yt, P, labels=list(range(K))), ROC_AUC=auc, PR_AUC=ap)
METRICS = ["Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "BalancedAcc", "MCC", "QWK", "MeanClassDistance", "LogLoss", "ROC_AUC", "PR_AUC"]
FM = ["Accuracy", "MacroF1", "MCC", "QWK"]
def fm_batch(cm):
    # fast Accuracy, macro-F1, MCC, QWK for a batch of confusion matrices (B, K, K); identical to scikit-learn
    cm = cm.astype(float); n = cm.sum((1, 2)); tp = np.einsum("bkk->bk", cm); tr = cm.sum(2); pr = cm.sum(1)
    acc = tp.sum(1) / n
    prec = tp / np.where(pr > 0, pr, 1); rec = tp / np.where(tr > 0, tr, 1)
    f1 = np.where(prec + rec > 0, 2 * prec * rec / np.where(prec + rec > 0, prec + rec, 1), 0); present = (tr + pr) > 0
    mf1 = (f1 * present).sum(1) / present.sum(1)
    cov_tp = tp.sum(1) * n - (tr * pr).sum(1); cov_t = n * n - (tr ** 2).sum(1); cov_p = n * n - (pr ** 2).sum(1)
    den = np.sqrt(cov_t * cov_p); mcc = np.where(den > 0, cov_tp / np.where(den > 0, den, 1), 0)
    Wq = ((np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2.)[None]   # weights on the fixed ordinal index (v13: re-indexed present classes)
    E = tr[:, :, None] * pr[:, None, :] / n[:, None, None]; dq = (Wq * E).sum((1, 2))
    qwk = np.where(dq > 0, 1 - (Wq * cm).sum((1, 2)) / np.where(dq > 0, dq, 1), 0)
    return np.c_[acc, mf1, mcc, qwk]
_r = np.random.default_rng(0); _y = _r.integers(0, K, 3000); _p = np.where(_r.random(3000) < .7, _y, _r.integers(0, K, 3000))
for _yy, _pp in [(_y, _p), (np.where(_y % 2 == 0, _y, 0), np.where(_p % 2 == 0, _p, 0))]:      # second case: classes 1 and 3 absent
    _cm = np.bincount(_yy * K + _pp, minlength=K * K).reshape(1, K, K)
    assert np.allclose(fm_batch(_cm)[0], [accuracy_score(_yy, _pp), f1_score(_yy, _pp, average="macro"), matthews_corrcoef(_yy, _pp),
                                          cohen_kappa_score(_yy, _pp, weights="quadratic", labels=list(range(K)))]), "fast metrics must equal scikit-learn"
SUBSETS = {k: list(itertools.combinations(range(4), k)) for k in (1, 2, 3)}
HIDX = [BASE_COLS.index(c) for c in H_COLS + CTX_COLS]
# leakage unit test: features must not change when the UNACQUIRED current laboratory values change
rows_t = np.where(TIER_A & STATION)[0][:1000]; M_t = fixed_mask(len(rows_t), (1, 2)); saved = LABCUR.copy()
Xa = {fm: featurize(rows_t, M_t, fm) for fm in (True, False)}
LABCUR[np.ix_(rows_t, [0, 3])] = np.random.default_rng(1).uniform(0, 100, (len(rows_t), 2))
Xb = {fm: featurize(rows_t, M_t, fm) for fm in (True, False)}; LABCUR[:] = saved
assert all(np.array_equal(np.nan_to_num(Xa[fm], nan=-9e9), np.nan_to_num(Xb[fm], nan=-9e9)) for fm in (True, False))
assert featurize(np.arange(5), fixed_mask(5, (0, 1))).shape[1] == len(NAMES[True])
def wauc(yb, sc, W):
    # weighted Mann-Whitney ROC-AUC of scores sc for binary labels yb under B weight vectors W (B, n); ties count 1/2
    u, inv = np.unique(sc, return_inverse=True); n = len(sc); U = len(u)
    Ip = csr_matrix((np.ones(yb.sum()), (np.where(yb)[0], inv[yb])), shape=(n, U))
    In = csr_matrix((np.ones((~yb).sum()), (np.where(~yb)[0], inv[~yb])), shape=(n, U))
    Wp = np.asarray((Ip.T @ W.T).T); Wn = np.asarray((In.T @ W.T).T); below = np.cumsum(Wn, 1) - Wn
    with np.errstate(invalid="ignore", divide="ignore"): return (Wp * (below + 0.5 * Wn)).sum(1) / (Wp.sum(1) * Wn.sum(1))
def macro_wauc(yt, P, W):
    ks = [k for k in range(K) if 0 < (yt == k).sum() < len(yt)]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); return np.nanmean(np.stack([wauc(yt == k, P[:, k], W) for k in ks], 1), 1)
_Pr = _r.dirichlet(np.ones(K), 3000); _Pr[np.arange(3000), _y] += 0.5; _Pr /= _Pr.sum(1, keepdims=True)
assert np.isclose(macro_wauc(_y, _Pr, np.ones((1, 3000)))[0], macro_auc(_y, _Pr)[0]), "weighted AUC must equal scikit-learn"
print("unit tests passed: fast metrics = scikit-learn; weighted ROC-AUC = scikit-learn; features are invariant to unacquired current laboratory values")
print("features: proposed", len(NAMES[True]), "| black box", len(NAMES[False]))

## 9. Main experiment — leave-one-country-out (checkpointed)
For each primary test country: tuning on *training → validation country*, then `N_REPEATS` training repeats × label budgets.
Every method in a repeat uses the same source sample, target labels and evaluation samples; acquisition budgets are asserted.
Evaluation: tier B samples of the target country's later history (station series); **tier A** (all eight values genuinely measured)
is the primary subset.

In [ ]:
results, acq_log, tune_log, sel_log = [], [], [], []
MAINSTORE, TUNED = {}, {}
def tune_country(test_c):
    val_c = VAL_OF[test_c]; r0 = np.random.default_rng(0)
    pv = np.where((ctry == val_c) & (STATION if PROTOCOL == "genuine" else True))[0]
    if PROTOCOL == "genuine":
        pv = pv[TIER_A[pv]] if TIER_A[pv].sum() >= 300 else pv[TIER_B[pv]]
    idx_val = np.sort(r0.choice(pv, min(VAL_CAP, len(pv)), replace=False))
    idx_tr = sources(test_c); idx_tr = idx_tr[ctry[idx_tr] != val_c]; idx_tr = r0.choice(idx_tr, min(TRAIN_CAP, len(idx_tr)), replace=False)
    yv = y_all[idx_val]; dv = days[idx_val]; lab_v = idx_val[dv <= np.quantile(dv, .10)]; ev_v = idx_val[dv > np.quantile(dv, .5)]
    T = {}
    for fm in (True, False):
        best = (None, -1, None)
        for prm in XGB_GRID:
            m = train(idx_tr, np.array([], int), prm, 1.0, 0, fm)
            s = np.mean([f1_score(yv, predict(m, idx_val, random_mask(len(idx_val), k, np.random.default_rng(k)), fm).argmax(1), average="macro") for k in range(4)])
            tune_log.append(dict(test=test_c, protocol=PROTOCOL, model="proposed" if fm else "black box", what="xgb", setting=str(prm), val_MacroF1=s))
            if s > best[1]: best = (prm, s, m)
        T[fm] = dict(prm=best[0], m_val=best[2])
    FIXED = {}
    for k, subs in SUBSETS.items():
        sc = {s_: f1_score(yv, predict(T[True]["m_val"], idx_val, fixed_mask(len(idx_val), s_)).argmax(1), average="macro") for s_ in subs}
        FIXED[k] = max(sc, key=sc.get)
        sel_log.append(dict(test=test_c, protocol=PROTOCOL, k=k, chosen="+".join(LABS[j] for j in FIXED[k]), val_MacroF1=sc[FIXED[k]]))
    for fm in (True, False):
        bw = (1.0, -1)
        for w in W_GRID:
            m = train(idx_tr, lab_v, T[fm]["prm"], w, 0, fm)
            s = f1_score(y_all[ev_v], predict(m, ev_v, fixed_mask(len(ev_v), FIXED[2]), fm).argmax(1), average="macro")
            tune_log.append(dict(test=test_c, protocol=PROTOCOL, model="proposed" if fm else "black box", what="target_weight", setting=w, val_MacroF1=s))
            if s > bw[1]: bw = (w, s)
        T[fm]["w"] = bw[0]; T[fm].pop("m_val", None)
    return T, FIXED, val_c
def tname(k): return "Proposed – 0 tests" if k == 0 else f"Proposed – {k} tests fixed"

for test_c in PRIMARY:
    t_c = time.time(); ck = ck_load(f"tune_{test_c}")
    if ck is None:
        n_t, n_s = len(tune_log), len(sel_log); T, FIXED, val_c = tune_country(test_c)
        ck_save(f"tune_{test_c}", dict(T=T, FIXED=FIXED, val_c=val_c, tune=tune_log[n_t:], sel=sel_log[n_s:]))
    else:
        T, FIXED, val_c = ck["T"], ck["FIXED"], ck["val_c"]; tune_log += ck["tune"]; sel_log += ck["sel"]
    TUNED[test_c] = (T, FIXED); MAINSTORE[test_c] = dict(T=T, FIXED=FIXED, reps=[])
    src_all = sources(test_c)
    print(f"[{test_c}] val={val_c} | proposed {T[True]['prm']} w={T[True]['w']} | black box {T[False]['prm']} w={T[False]['w']} | fixed "
          + ", ".join(f"k{k}={'+'.join(LABS[j] for j in FIXED[k])}" for k in FIXED), flush=True)
    for rep in range(N_REPEATS):
        ck = ck_load(f"main_{test_c}_rep{rep}")
        if ck is None:
            n_r, n_a = len(results), len(acq_log); store_rep = None
            src = np.random.default_rng(100 + rep).choice(src_all, min(TRAIN_CAP, len(src_all)), replace=False)
            for b in BUDGETS:
                lab, ev = target_split(test_c, b); n = len(ev); seed = 1000 * rep + int(100 * b)
                rng = np.random.default_rng(seed); trw = np.r_[src, lab]; wtr = np.where(np.isin(trw, lab), T[True]["w"], 1.0)
                prop = train(src, lab, T[True]["prm"], T[True]["w"], seed, True)
                bb = train(src, lab, T[False]["prm"], T[False]["w"], seed, False)
                maj = np.bincount(y_all[trw], minlength=K).argmax()
                P = {"Persistence": persistence_P(ev), "Majority class": onehot(np.full(n, maj))}
                wfc = compute_sample_weight("balanced", y_all[trw]) * np.where(np.isin(trw, lab), T[False]["w"], 1.0)
                fc = fitx(make_xgb(T[False]["prm"], seed), XBASE[trw][:, HIDX], y_all[trw], wfc)
                P["Forecast only (history)"] = proba(fc, XBASE[ev][:, HIDX])
                masks = {0: np.zeros((n, 4), bool)}
                for k in (1, 2, 3): masks[k] = fixed_mask(n, FIXED[k])
                for k in (0, 1, 2, 3):
                    P[f"Formula rule (last value) – {k} tests"] = rule_P(ev, masks[k], "last")
                    P[f"Formula rule (assume pass) – {k} tests"] = rule_P(ev, masks[k], "pass")
                    P[f"Black box – {k} tests"] = predict(bb, ev, masks[k], False)
                    P[tname(k)] = predict(prop, ev, masks[k])
                AM = adaptive_masks(prop, fit_density(trw, wtr), ev, 3, seed)
                for k in (1, 2, 3):
                    RM = random_mask(n, k, rng)
                    assert (masks[k].sum(1) == k).all() and (AM[k].sum(1) == k).all() and (RM.sum(1) == k).all(), "unequal laboratory budgets"
                    P[f"Proposed – {k} tests adaptive"] = predict(prop, ev, AM[k])
                    P[f"Proposed – {k} tests random"] = predict(prop, ev, RM)
                    for j in range(4): acq_log.append(dict(test_country=test_c, repeat=rep, budget=b, k=k, test=LABS[j], share=AM[k][:, j].mean()))
                P["Proposed – 4 tests (circular reference)"] = predict(prop, ev, np.ones((n, 4), bool))
                if b == MAIN_BUDGET:
                    for k, subs in SUBSETS.items():
                        for s_ in subs: P["Proposed – subset " + "+".join(LABS[j] for j in s_)] = predict(prop, ev, fixed_mask(n, s_))
                    P["Ablation: without monotonicity constraint"] = predict(prop, ev, masks[2], constrained=False)
                    for nm_, cols in [("Ablation: without history", H_COLS), ("Ablation: without current sensors", S_COLS)]:
                        bk = (XBASE.copy(), SENCUR.copy(), LABLAG.copy())
                        XBASE[:, [BASE_COLS.index(c) for c in cols]] = 0.0
                        if "sensors" in nm_: SENCUR[:] = np.array([[8.0, 7.5, 12.0, 1.0]])     # neutral constants meeting every objective
                        else: LABLAG[:] = 0.0
                        m_ = train(src, lab, T[True]["prm"], T[True]["w"], seed, True)
                        P[nm_] = predict(m_, ev, masks[2]); XBASE[:], SENCUR[:], LABLAG[:] = bk
                tA = TIER_A[ev]
                for nm, Pm in P.items():
                    for tier, sel in (("A", tA), ("B", np.ones(n, bool))):
                        if sel.sum() >= 5:
                            results.append(dict(test_country=test_c, repeat=rep, budget=b, method=nm, tier=tier, n_eval=int(sel.sum()),
                                                **metrics9(y_all[ev][sel], Pm[sel])))
                if b == MAIN_BUDGET:
                    store_rep = dict(ev=ev, P={nm: Pm.astype(np.float32) for nm, Pm in P.items()}, AM=AM,
                                     prop=prop if rep == 0 else None, bb=bb if rep == 0 else None)
            ck_save(f"main_{test_c}_rep{rep}", dict(results=results[n_r:], acq=acq_log[n_a:], store=store_rep))
        else:
            results += ck["results"]; acq_log += ck["acq"]; store_rep = ck["store"]
        MAINSTORE[test_c]["reps"].append(store_rep)
        r_ = pd.DataFrame(results).query("test_country == @test_c and repeat == @rep and budget == @MAIN_BUDGET and tier == 'A'").set_index("method")
        print(f"   rep {rep+1}/{N_REPEATS} tier A (n={int(r_.n_eval.iloc[0])}) accuracy: rule(last) {r_.Accuracy['Formula rule (last value) – 2 tests']:.4f} | "
              f"black box-2 {r_.Accuracy['Black box – 2 tests']:.4f} | proposed-2 fixed {r_.Accuracy['Proposed – 2 tests fixed']:.4f} | "
              f"proposed-2 adaptive {r_.Accuracy['Proposed – 2 tests adaptive']:.4f} | {time.time()-t_c:.0f}s", flush=True)
    evs = [e["ev"] for e in MAINSTORE[test_c]["reps"]]
    assert all(np.array_equal(evs[0], e) for e in evs), "every repeat must use identical evaluation samples"
res = pd.DataFrame(results); acq = pd.DataFrame(acq_log)
savetab(res, "tab_all_results"); savetab(pd.DataFrame(sel_log), "tab_fixed_set_selection"); savetab(pd.DataFrame(tune_log), "tab_tuning_log")
print(f"main experiment done | elapsed {(time.time()-T0)/60:.1f} min")

### 9b. Leakage audit
Automated checks that no information from the test country's evaluation samples, or from the future, reaches training or model
selection. Checks marked "by design" are properties of the code that are stated, with the section where they are implemented.

In [ ]:
LEAK = []
def _chk(check, ok, detail, how="tested"):
    LEAK.append(dict(check=check, result=detail, status=("pass" if ok else "FAIL") if how == "tested" else how))
d_ = {c: sorted(set(ctry[sources(c)])) for c in PRIMARY}
_chk("Test country excluded from every training set", all(c not in d_[c] for c in PRIMARY),
     "; ".join(f"{c}: trained on {', '.join(v)}" for c, v in d_.items()))
_chk("Tuning, target weight and fixed test set chosen on a validation country, never the test country", all(VAL_OF[c] != c for c in PRIMARY),
     "; ".join(f"{c} → {VAL_OF[c]}" for c in PRIMARY))
gaps, same_ev = [], True
for c in PRIMARY:
    ev0 = target_split(c, 0.0)[1]
    for b in BUDGETS:
        lab, ev = target_split(c, b); same_ev &= np.array_equal(ev0, ev)
        if b > 0: gaps.append(int(days[ev].min() - days[lab].max()))
_chk("Labelled target-country samples precede every evaluation sample", min(gaps) > 0,
     f"smallest gap {min(gaps)} days over {len(gaps)} country × label-budget splits")
_chk("Identical evaluation samples for every label budget, method and repeat", same_ev, "asserted for all budgets; repeats asserted in §9")
same_series = (key_c[I1] == key_c[I0]) & (key_s[I1] == key_s[I0]) & (key_c[I2] == key_c[I0]) & (key_s[I2] == key_s[I0])
earlier = (dtv[I1] < dtv[I0]) & (dtv[I2] < dtv[I1])
_chk("Lagged values (t−1, t−2) come from earlier visits of the same site", bool(same_series.all() and earlier.all()), f"{len(I0):,} examples")
hc = [BASE_COLS.index(f"{f}_hist_exceed_rate") for f in F]; genm = ~bits_to_mask(SD_BITS[RULE]); okh = True
for p in np.random.default_rng(11).choice(len(I0), 300, replace=False):
    t = I0[p]; rr = np.arange(START[t], t)
    okh &= bool((dtv[rr] < dtv[t]).all() and (key_s[rr] == key_s[t]).all() and (key_c[rr] == key_c[t]).all())
    ex_ = np.zeros((len(rr), len(F)), bool)
    for j, f in enumerate(F):
        lo, hi = GUIDE[f]
        if hi is not None: ex_[:, j] |= SDF[rr, j] > hi
        if lo is not None: ex_[:, j] |= SDF[rr, j] < lo
    g_ = genm[rr]; ng = g_.sum(0); rate = np.where(ng > 0, (ex_ & g_).sum(0) / np.maximum(ng, 1), np.nan)
    okh &= np.allclose(np.nan_to_num(rate, nan=-1.0), np.nan_to_num(XBASE[p, hc], nan=-1.0))
_chk("Site-history features use only visits before t", okh, "300 random examples recomputed independently from earlier visits")
okf = True
for c in PRIMARY:
    ev = MAINSTORE[c]["reps"][0]["ev"]; F2 = list(MAINSTORE[c]["FIXED"][2]); hid = [j for j in range(4) if j not in F2]
    M_ = fixed_mask(len(ev), F2); saved = LABCUR.copy(); Xa = featurize(ev, M_, True)
    LABCUR[np.ix_(ev, hid)] = np.random.default_rng(2).uniform(0, 100, (len(ev), len(hid))); Xb = featurize(ev, M_, True); LABCUR[:] = saved
    okf &= np.array_equal(np.nan_to_num(Xa, nan=-9e9), np.nan_to_num(Xb, nan=-9e9))
_chk("Unmeasured current laboratory values are never read", okf, "features unchanged when the unmeasured values of every evaluation sample are randomised")
ovl = {c: len(set(site_ids[MAINSTORE[c]["reps"][0]["ev"]]) & set(site_ids[sources(c)])) for c in PRIMARY}
_chk("No monitoring site shared between training and test data", all(v == 0 for v in ovl.values()), "; ".join(f"{c}: {v} shared sites" for c, v in ovl.items()))
def _vec_hash(rows): return pd.util.hash_pandas_object(pd.DataFrame(np.round(SDF[I0[rows]], 6)), index=False).to_numpy()
dup = {}
for c in PRIMARY:
    ev = MAINSTORE[c]["reps"][0]["ev"]; evA = ev[TIER_A[ev]]
    dup[c] = 100 * np.isin(_vec_hash(evA), _vec_hash(sources(c))).mean()
_chk("Test measurement vectors that also occur in the training data", all(v < 1 for v in dup.values()),
     "; ".join(f"{c}: {v:.2f} %" for c, v in dup.items()) + " of tier-A test samples (identical eight values)")
b_info = min([b for b in BUDGETS if b > 0], default=MAIN_BUDGET); ss = {}
for c in PRIMARY:
    lab, ev = target_split(c, b_info); evA = ev[TIER_A[ev]]; ss[c] = 100 * np.isin(site_ids[evA], site_ids[lab]).mean()
_chk(f"Test sites that also contribute earlier labelled samples ({100 * b_info:.0f} % budget, label-budget analysis only)", True,
     "; ".join(f"{c}: {v:.1f} %" for c, v in ss.items()) + f" — permitted when target labels are used; the headline uses {budget_txt()}", how="info")
m0_ = bits_to_mask(SD_BITS[RULE][I0]); tier_ok = {"A": ~m0_.any(1), "B": ~m0_[:, LIX].any(1)}.get(TRAIN_TIER, np.ones(len(I0), bool))
tr_rows = np.unique(np.concatenate([sources(c) for c in PRIMARY] + [target_split(c, b)[0] for c in PRIMARY for b in BUDGETS]))
_chk(f"Training and target labels contain no imputed values (training tier {TRAIN_TIER})", bool(tier_ok[tr_rows].all()),
     f"{len(tr_rows):,} training/target-label rows checked against the imputation flags"
     + ("" if TRAIN_TIER in ("A", "B") else " (tier 'all' = v13 behaviour: imputed values allowed)"), how="tested" if TRAIN_TIER in ("A", "B") else "info")
rr_, cc_ = np.where(LAST_SRC >= 0); s_ = LAST_SRC[rr_, cc_]; t_ = I0[rr_]; jj_ = np.array(LIX)[cc_]; gm_ = ~bits_to_mask(SD_BITS[RULE])
okl = bool(((key_c[s_] == key_c[t_]) & (key_s[s_] == key_s[t_]) & (dtv[s_] < dtv[t_]) & gm_[s_, jj_]
            & np.isclose(SDF[s_, jj_], LABLAG[rr_, cc_])).all())
_chk("Last genuine laboratory values come from earlier, genuinely measured visits of the same site", okl,
     f"{len(s_):,} carried-forward values recomputed (at most {LAST_MAX_AGE_DAYS} days old)")
_chk("Preprocessing fitted on training rows only", True,
     "XGBoost uses no scaling or imputation (missing = NaN); logistic-regression imputer and scaler are fitted inside a pipeline on training rows (§12b); "
     "deep-model scalers on source rows (§14b); QRC encoder on source rows (§16); density models for adaptive selection on training rows (§9)", how="by design")
_chk("Imputation detection uses no labels", True,
     "the detection rules use only each country's value distribution (country median, frequent constants), never the class (§4b)", how="by design")
LEAKAGE = pd.DataFrame(LEAK); savetab(LEAKAGE, "tab_leakage_audit")
assert not (LEAKAGE.status == "FAIL").any(), LEAKAGE[LEAKAGE.status == "FAIL"]
print(f"leakage audit: {(LEAKAGE.status == 'pass').sum()} tested checks passed, {(LEAKAGE.status == 'FAIL').sum()} failed")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    t = pd.DataFrame({"Result": list(LEAKAGE.result), "Status": list(LEAKAGE.status)}, index=list(LEAKAGE.check))
    show_table("leakage", "Leakage audit of the evaluation protocol", t, stub="Check", left=["Result", "Status"],
               note="'pass' = tested in the cell above (the notebook stops if a tested check fails); 'by design' = property of the code, "
                    "with the section where it is implemented; 'info' = reported for transparency.")
_present()

## 10. Results (issues 5 and 6)
All intervals are **95 % site-clustered bootstrap intervals** (sites resampled within each country; §12). Headline values are the
equal-weight mean over primary countries of each country's mean over training repeats, on **tier A** samples at
`MAIN_BUDGET` target labels (0 = no sample of the test country is used for training or model selection; the 10–50 % budgets are in
Table 21). Canada contributes few tier-A samples but one third of the weight; per-country results are in Table 17.


In [ ]:
def country_cms(c, methods, extra_sel=None):
    reps = MAINSTORE[c]["reps"]; ev = reps[0]["ev"]; sel = TIER_A[ev].copy()
    if extra_sel is not None: sel &= extra_sel(c, ev)
    rows = ev[sel]; yt = y_all[rows]; su, sidx = np.unique(site_ids[rows], return_inverse=True); out = {}
    for m in methods:
        cms = np.zeros((len(reps), len(su), K, K))
        for r, e in enumerate(reps): np.add.at(cms[r], (sidx, yt, e["P"][m][sel].argmax(1)), 1)
        out[m] = cms
    return out, len(su), len(rows)
def boot(methods, extra_sel=None, countries=None, B=N_BOOT):
    # returns {country: {"n", "sites", "point": {m: (4,)}, "draws": {m: (B, 4)}}}; repeats are averaged inside every draw
    out = {}
    for ci, c in enumerate(countries or PRIMARY):
        cms, S, n = country_cms(c, methods, extra_sel)
        if n < 10: continue
        W = np.random.default_rng(1000 + ci).multinomial(S, np.full(S, 1 / S), size=B).astype(float)
        out[c] = dict(n=n, sites=S, point={m: np.mean([fm_batch(cms[m][r].sum(0)[None])[0] for r in range(len(cms[m]))], 0) for m in methods},
                      draws={m: np.mean([fm_batch(np.einsum("bs,skl->bkl", W, cms[m][r])) for r in range(len(cms[m]))], 0) for m in methods})
    return out
def summarise(bt, methods):
    rows = []
    for m in methods:
        pts = np.array([bt[c]["point"][m] for c in bt]); drw = np.mean([bt[c]["draws"][m] for c in bt], 0)
        lo, hi = np.percentile(drw, [2.5, 97.5], axis=0)
        rows.append(dict(method=m, **{f"{f}": pts.mean(0)[i] for i, f in enumerate(FM)}, **{f"{f}_CI": f"[{lo[i]:.4f}, {hi[i]:.4f}]" for i, f in enumerate(FM)}))
    return pd.DataFrame(rows).set_index("method")
def boot_auc(methods, extra_sel=None, countries=None, B=N_BOOT):
    # macro ROC-AUC with the SAME site draws as boot(); repeats averaged inside every draw
    out = {}
    for ci, c in enumerate(countries or PRIMARY):
        reps = MAINSTORE[c]["reps"]; ev = reps[0]["ev"]; sel = TIER_A[ev].copy()
        if extra_sel is not None: sel &= extra_sel(c, ev)
        rows = ev[sel]
        if len(rows) < 10: continue
        yt = y_all[rows]; su, sidx = np.unique(site_ids[rows], return_inverse=True); S = len(su)
        W = np.random.default_rng(1000 + ci).multinomial(S, np.full(S, 1 / S), size=B).astype(float)[:, sidx]
        out[c] = dict(point={m: np.mean([macro_auc(yt, e["P"][m][sel].astype(float))[0] for e in reps]) for m in methods},
                      draws={m: np.mean([macro_wauc(yt, e["P"][m][sel].astype(float), W) for e in reps], 0) for m in methods})
    return out
def summarise_auc(bta, methods):
    rows = []
    for m in methods:
        drw = np.nanmean([bta[c]["draws"][m] for c in bta], 0); lo, hi = np.nanpercentile(drw, [2.5, 97.5])
        rows.append(dict(method=m, ROC_AUC=np.mean([bta[c]["point"][m] for c in bta]), ROC_AUC_CI=f"[{lo:.4f}, {hi:.4f}]"))
    return pd.DataFrame(rows).set_index("method")
KEY = ["Persistence", "Majority class", "Forecast only (history)", "Formula rule (assume pass) – 2 tests", "Formula rule (last value) – 2 tests",
       "Black box – 2 tests", "Proposed – 1 tests adaptive", "Proposed – 2 tests fixed", "Proposed – 2 tests adaptive"]
BT = boot(KEY); BTA = boot_auc(KEY)
HEAD = summarise(BT, KEY).join(summarise_auc(BTA, KEY)).round(4)
savetab(HEAD.reset_index(), "tab_headline")
mA = res[(res.budget == MAIN_BUDGET) & (res.tier == "A")]
full9 = mA.groupby(["test_country", "method"])[METRICS].mean().groupby("method").mean().round(4)
savetab(full9.reindex(KEY).reset_index(), "tab_main_11metrics")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    seen_txt = " and never seen in training or model selection" if MAIN_BUDGET == 0 else f"; {100 * MAIN_BUDGET:.0f} % of each test country's earliest history is labelled"
    keys, idx = method_rows(HEAD.index); H = HEAD.loc[keys]
    t = pd.DataFrame({("Lab tests", ""): [str(METHODS[m][2]) for m in keys],
                      ("Accuracy (%)", "mean"): fmt_list(H.Accuracy, f_pct),
                      ("Accuracy (%)", "95% CI"): fmt_list(H.Accuracy_CI, lambda s: f_ci(s, 2, 100)),
                      ("Macro-F1", "mean"): fmt_list(H.MacroF1), ("Macro-F1", "95% CI"): fmt_list(H.MacroF1_CI, f_ci),
                      ("MCC", ""): fmt_list(H.MCC), ("QWK", ""): fmt_list(H.QWK),
                      ("ROC-AUC", "mean"): fmt_list(H.ROC_AUC), ("ROC-AUC", "95% CI"): fmt_list(H.ROC_AUC_CI, f_ci)}, index=idx)
    num = pd.DataFrame(np.nan, index=idx, columns=t.columns)
    for c, src in [(("Accuracy (%)", "mean"), "Accuracy"), (("Macro-F1", "mean"), "MacroF1"), (("MCC", ""), "MCC"),
                   (("QWK", ""), "QWK"), (("ROC-AUC", "mean"), "ROC_AUC")]:
        num[c] = H[src].to_numpy(float)
    show_table("headline", f"Main results on unseen test countries (tier A, {budget_txt()})", t, stub="Method", bold=best_mask(num),
               note=f"Mean over the test countries ({', '.join(PRIMARY)}), each weighted equally{seen_txt}. 95% CI = "
                    f"site-clustered bootstrap ({N_BOOT:,} draws). Lab tests = number of the four laboratory tests (BOD, NH₃, PO₄, N) "
                    f"measured at time t; fixed pair = {fixed_pair_text()}, chosen on the validation country. Rules, persistence and the "
                    "majority class output a single class, so their ROC-AUC reflects one operating point. MCC = Matthews correlation; "
                    "QWK = quadratic weighted kappa (CIs in tab_headline.csv). Bold = best value in each column.")
    keys, idx = method_rows(full9.index); d = full9.loc[keys]
    spec = [("Accuracy (%) ↑", "Accuracy", f_pct, True), ("Balanced accuracy ↑", "BalancedAcc", f_num, True),
            ("Macro precision ↑", "MacroPrecision", f_num, True), ("Macro recall ↑", "MacroRecall", f_num, True),
            ("Macro-F1 ↑", "MacroF1", f_num, True), ("MCC ↑", "MCC", f_num, True), ("QWK ↑", "QWK", f_num, True),
            ("ROC-AUC ↑", "ROC_AUC", f_num, True), ("PR-AUC ↑", "PR_AUC", f_num, True),
            ("Mean class distance ↓", "MeanClassDistance", f_num, False), ("Log-loss ↓", "LogLoss", f_num, False)]
    t = pd.DataFrame({lab: fmt_list(d[src], fn) for lab, src, fn, _ in spec}, index=idx)
    num = pd.DataFrame({lab: d[src].to_numpy(float) for lab, src, _, _ in spec}, index=idx)
    show_table("metrics11", "All eleven metrics for the main methods (tier A)", t, stub="Method",
               bold=best_mask(num, higher={lab: hb for lab, _, _, hb in spec}),
               note="Equal-weight mean over the test countries and the training repeats. ↑ higher is better, ↓ lower is better. Balanced "
                    "accuracy equals macro recall when every predicted class occurs in the test data. Mean class distance = mean |predicted − true class| in classes. "
                    "Bold = best value in each column.")
_present()

In [ ]:
# per-country results with sample counts, class distributions and the majority-class baseline (issue 6)
pcr = []
for c in BT:
    ev = MAINSTORE[c]["reps"][0]["ev"]; rows = ev[TIER_A[ev]]; dist = np.bincount(y_all[rows], minlength=K) / len(rows) * 100
    for m in ["Majority class", "Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 2 tests fixed", "Proposed – 2 tests adaptive"]:
        lo, hi = np.percentile(BT[c]["draws"][m], [2.5, 97.5], axis=0); p = BT[c]["point"][m]
        err = np.mean([(e["P"][m][TIER_A[ev]].argmax(1) != y_all[rows]).sum() for e in MAINSTORE[c]["reps"]])
        pcr.append(dict(country=c, n_tierA=len(rows), sites=BT[c]["sites"], **{f"%{CLASSES[k]}": dist[k] for k in range(K)},
                        method=m, Accuracy=p[0], Acc_CI=f"[{lo[0]:.4f}, {hi[0]:.4f}]", MacroF1=p[1], F1_CI=f"[{lo[1]:.4f}, {hi[1]:.4f}]",
                        MCC=p[2], QWK=p[3], ROC_AUC=BTA[c]["point"][m],
                        AUC_CI="[{:.4f}, {:.4f}]".format(*np.nanpercentile(BTA[c]["draws"][m], [2.5, 97.5])), errors=err))
pcr = pd.DataFrame(pcr); savetab(pcr, "tab_per_country")
tb = res[(res.budget == MAIN_BUDGET) & res.method.isin(KEY)].groupby(["tier", "test_country", "method"])[["n_eval", "Accuracy", "MacroF1", "MCC", "QWK", "ROC_AUC"]].mean().round(4)
savetab(tb.reset_index(), "tab_tiers")
# subsets: hidden failures, single-record days, freshwater ambient waters
def hidden_fail(c, ev):
    M = fixed_mask(len(ev), MAINSTORE[c]["FIXED"][2]); hi = np.array([GUIDE[f][1] for f in LABS])
    return ((LABRAW[ev] > hi) & ~M).any(1)
SUB = {"hidden failure (an unmeasured test fails)": hidden_fail, "no hidden failure": lambda c, ev: ~hidden_fail(c, ev),
       "single-record site-days": lambda c, ev: SINGLE[ev], "freshwater ambient (river/lake/canal)": lambda c, ev: FRESH[ev]}
SM = ["Formula rule (assume pass) – 2 tests", "Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 2 tests fixed"]
sub_rows = []
for sn, fn in SUB.items():
    bt_ = boot(SM, fn)
    if not bt_: continue
    sm = summarise(bt_, SM)
    for m in SM: sub_rows.append(dict(subset=sn, n=sum(bt_[c]["n"] for c in bt_), countries=len(bt_), method=m,
                                      Accuracy=sm.loc[m, "Accuracy"], Acc_CI=sm.loc[m, "Accuracy_CI"], MacroF1=sm.loc[m, "MacroF1"], F1_CI=sm.loc[m, "MacroF1_CI"]))
sub_tab = pd.DataFrame(sub_rows); savetab(sub_tab, "tab_subsets")
hf_share = {c: hidden_fail(c, MAINSTORE[c]["reps"][0]["ev"])[TIER_A[MAINSTORE[c]["reps"][0]["ev"]]].mean() for c in BT}
# share of tier-A samples with a hidden failure: shown in the evaluation-data table below

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    pc = pcr.round(4); cs = countries_in(pc.country)
    first = pc.drop_duplicates("country").set_index("country").loc[cs]
    t = pd.DataFrame({("Samples (tier A)", ""): fmt_list(first.n_tierA, f_int), ("Sites", ""): fmt_list(first.sites, f_int),
                      **{("Class distribution (%)", k): fmt_list(first[f"%{k}"], lambda v: f_num(v, 1)) for k in CLASSES},
                      ("Hidden failure (%)", ""): [f"{round(100 * float(hf_share[c]), 1):.1f}" for c in cs]}, index=cs)
    show_table("evaldata", "Evaluation data of each test country (tier A)", t, stub="Test country",
               note="Samples = tier-A evaluation samples of the country; Sites = monitoring sites that contribute them. Hidden failure = "
                    "share of samples in which a laboratory test that is not measured (fixed 2-test set) fails its objective.")
    acc = lambda s: f_ci(s, 2, 100)
    rows, idx = [], []
    for c in cs:
        d = pc[pc.country == c].set_index("method")
        lab = f"{c}: {f_int(d.n_tierA.iloc[0])} samples from {f_int(d.sites.iloc[0])} sites"
        for m in [m for m in METHODS if m in d.index]: rows.append(d.loc[m]); idx.append((lab, METHODS[m][1]))
    d = pd.DataFrame(rows); idx = pd.MultiIndex.from_tuples(idx)
    t = pd.DataFrame({("Accuracy (%)", "mean"): fmt_list(d.Accuracy, f_pct), ("Accuracy (%)", "95% CI"): fmt_list(d.Acc_CI, acc),
                      ("Macro-F1", "mean"): fmt_list(d.MacroF1), ("Macro-F1", "95% CI"): fmt_list(d.F1_CI, f_ci),
                      ("MCC", ""): fmt_list(d.MCC), ("QWK", ""): fmt_list(d.QWK),
                      ("ROC-AUC", "mean"): fmt_list(d.ROC_AUC), ("ROC-AUC", "95% CI"): fmt_list(d.AUC_CI, f_ci),
                      ("Errors", ""): fmt_list(d.errors, lambda v: DASH if _isnan(v) else f"{float(v):,.1f}")}, index=idx)
    num = pd.DataFrame(np.nan, index=idx, columns=t.columns)
    for c, src in [(("Accuracy (%)", "mean"), "Accuracy"), (("Macro-F1", "mean"), "MacroF1"), (("MCC", ""), "MCC"), (("QWK", ""), "QWK"),
                   (("ROC-AUC", "mean"), "ROC_AUC"), (("Errors", ""), "errors")]:
        num[c] = d[src].to_numpy(float)
    show_table("percountry", "Results by test country (tier A, 2 laboratory tests)", t, stub="Method",
               bold=best_mask(num, higher={("Errors", ""): False}, by_group=True),
               note="Errors = misclassified samples (mean of the training repeats). 95% CI = site-clustered bootstrap within the country. "
                    "Bold = best within each country.")
    T = {"A": "Tier A: all 8 values measured", "B": "Tier B: 4 laboratory values measured"}
    MET = [("Accuracy (%)", "Accuracy", f_pct), ("Macro-F1", "MacroF1", f_num), ("MCC", "MCC", f_num), ("QWK", "QWK", f_num), ("ROC-AUC", "ROC_AUC", f_num)]
    rows, nums, idx = [], [], []
    for c in countries_in(tb.index.get_level_values("test_country")):
        nA, nB = tb.loc[("A", c)].n_eval.iloc[0], tb.loc[("B", c)].n_eval.iloc[0]
        lab = f"{c}: tier A {f_int(nA)} samples, tier B {f_int(nB)} samples"
        for m in [m for m in METHODS if ("A", c, m) in tb.index]:
            rows.append([fn(tb.loc[(ti, c, m), src]) for ti in T for _, src, fn in MET])
            nums.append([float(tb.loc[(ti, c, m), src]) for ti in T for _, src, _ in MET]); idx.append((lab, METHODS[m][1]))
    cols = pd.MultiIndex.from_tuples([(T[ti], lab) for ti in T for lab, _, _ in MET]); idx = pd.MultiIndex.from_tuples(idx)
    show_table("tiers", "Results by evaluation tier (2 laboratory tests)", pd.DataFrame(rows, index=idx, columns=cols), stub="Method",
               bold=best_mask(pd.DataFrame(nums, index=idx, columns=cols), by_group=True),
               note="Tier A (primary) = all eight current values genuinely measured; tier B adds samples in which only the four laboratory "
                    "values are known to be genuine. Bold = best within each country and tier.")
    SUBL = {"hidden failure (an unmeasured test fails)": "Hidden failure (an unmeasured test fails its objective)",
            "no hidden failure": "No hidden failure", "single-record site-days": "Single-record site-days",
            "freshwater ambient (river/lake/canal)": "Freshwater ambient waters (river, lake, canal)"}
    st = sub_tab.round(4); rows, idx = [], []
    for sname in pd.unique(st.subset):
        d = st[st.subset == sname].set_index("method")
        lab = f"{SUBL.get(sname, sname[:1].upper() + sname[1:])}: {f_int(d.n.iloc[0])} samples, {int(d.countries.iloc[0])} countries"
        for m in [m for m in METHODS if m in d.index]: rows.append(d.loc[m]); idx.append((lab, METHODS[m][1]))
    d = pd.DataFrame(rows); idx = pd.MultiIndex.from_tuples(idx)
    t = pd.DataFrame({("Accuracy (%)", "mean"): fmt_list(d.Accuracy, f_pct), ("Accuracy (%)", "95% CI"): fmt_list(d.Acc_CI, acc),
                      ("Macro-F1", "mean"): fmt_list(d.MacroF1), ("Macro-F1", "95% CI"): fmt_list(d.F1_CI, f_ci)}, index=idx)
    num = pd.DataFrame(np.nan, index=idx, columns=t.columns)
    num[("Accuracy (%)", "mean")] = d.Accuracy.to_numpy(float); num[("Macro-F1", "mean")] = d.MacroF1.to_numpy(float)
    show_table("subsets", "Results on subsets of the tier-A samples (2 laboratory tests, fixed pair)", t, stub="Method",
               bold=best_mask(num, by_group=True),
               note="Hidden failure = at least one of the two unmeasured laboratory tests fails its objective, so the class cannot be "
                    "read from the measured values alone. Mean over the countries that have at least 10 samples in the subset. "
                    "Bold = best within each subset.")
_present()

In [ ]:
yt_ = np.concatenate([y_all[MAINSTORE[c]["reps"][0]["ev"][TIER_A[MAINSTORE[c]["reps"][0]["ev"]]]] for c in BT]); pcl = []
for m in ["Black box – 2 tests", "Proposed – 2 tests fixed"]:
    yp_ = np.concatenate([MAINSTORE[c]["reps"][0]["P"][m][TIER_A[MAINSTORE[c]["reps"][0]["ev"]]].argmax(1) for c in BT])
    pr, rc, f1_, sup = precision_recall_fscore_support(yt_, yp_, labels=range(K), zero_division=0)
    for k in range(K): pcl.append(dict(method=m, cls=CLASSES[k], Precision=pr[k], Recall=rc[k], F1=f1_[k], Support=int(sup[k])))
pcl = pd.DataFrame(pcl)
savetab(pcl, "tab_per_class")
mb = res[(res.tier == "A") & res.method.isin(["Persistence", "Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 2 tests fixed", "Proposed – 2 tests adaptive"])]
bud = mb.groupby(["test_country", "budget", "method"])[["Accuracy", "MacroF1"]].mean().groupby(["budget", "method"]).mean().round(4)
savetab(bud.reset_index(), "tab_budgets")
fig, ax = plt.subplots(1, 2, figsize=(17, 4.6))
for nm, sty in [("Persistence", "k:"), ("Formula rule (last value) – 2 tests", "x--"), ("Black box – 2 tests", "s--"), ("Proposed – 2 tests fixed", "o-"), ("Proposed – 2 tests adaptive", "^-")]:
    d_ = bud.xs(nm, level="method"); ax[0].plot(d_.index * 100, d_.MacroF1, sty, label=nm)
ax[0].set_xlabel("target-country labels (%)"); ax[0].set_ylabel("macro-F1 (tier A)"); ax[0].set_title("(a) Effect of target labels"); ax[0].legend(fontsize=7)
pv = pcr.pivot_table(index="country", columns="method", values="Accuracy")
pv.plot(kind="bar", ax=ax[1], rot=0, colormap="viridis"); ax[1].set_ylim(0, 1.02); ax[1].set_title("(b) Accuracy by country (tier A, 2 tests)"); ax[1].legend(fontsize=6, loc="lower right")
plt.tight_layout(); savefig("fig04_results"); plt.show()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    p = pcl.round(4); bbm, prm = "Black box – 2 tests", "Proposed – 2 tests fixed"
    a = p[p.method == bbm].set_index("cls").reindex(CLASSES); b = p[p.method == prm].set_index("cls").reindex(CLASSES)
    MET = ["Precision", "Recall", "F1"]
    t = pd.DataFrame({("Samples", ""): fmt_list(a.Support, f_int), **{(mname(bbm), k): fmt_list(a[k]) for k in MET},
                      **{(mname(prm), k): fmt_list(b[k]) for k in MET}}, index=CLASSES)
    num = pd.DataFrame({("Samples", ""): a.Support.to_numpy(float), **{(mname(bbm), k): a[k].to_numpy(float) for k in MET},
                        **{(mname(prm), k): b[k].to_numpy(float) for k in MET}}, index=CLASSES)
    show_table("perclass", "Per-class results: black box vs proposed (tier A, 2 laboratory tests)", t, stub="Class",
               bold=better_of(num, [((mname(bbm), k), (mname(prm), k)) for k in MET]),
               note="Pooled over the test countries, training repeat 0. Bold = better of the two models.")
    keys, idx = method_rows(bud.index.get_level_values("method")); budgets = sorted(set(bud.index.get_level_values("budget")))
    cols, nums = {}, {}
    for met, lab, fn in [("Accuracy", "Accuracy (%)", f_pct), ("MacroF1", "Macro-F1", f_num)]:
        for b_ in budgets:
            v = [float(bud.loc[(b_, m), met]) for m in keys]; cols[(lab, f"{100 * b_:.0f} %")] = [fn(x) for x in v]; nums[(lab, f"{100 * b_:.0f} %")] = v
    t = pd.DataFrame(cols, index=idx)
    show_table("budgets", "Effect of the number of labelled target-country samples (tier A)", t, stub="Method",
               bold=best_mask(pd.DataFrame(nums, index=idx)),
               note="Column = share of the test country's earliest history whose labels are used for training (0 % = no target labels). "
                    "Persistence and the formula rule use no labels. Mean over test countries and training repeats. Bold = best in each column.")
_present()

## 10b. ROC curves and probability calibration
One-vs-rest ROC curves per class and the macro-average curve (tier A, all primary countries pooled, repeat 0). Calibration: reliability
diagram of the top-class probability, expected calibration error (ECE, 10 equal-width bins) and the multi-class Brier score.
Formula rules give a single class (probability 0.98), i.e. one operating point; they are shown for reference.

In [ ]:
RM_ = ["Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 2 tests fixed", "Proposed – 2 tests adaptive"]
ev_all = {c: MAINSTORE[c]["reps"][0]["ev"] for c in BT}; selA = {c: TIER_A[ev_all[c]] for c in BT}
yT = np.concatenate([y_all[ev_all[c][selA[c]]] for c in BT])
PP = {m: np.concatenate([MAINSTORE[c]["reps"][0]["P"][m][selA[c]].astype(float) for c in BT]) for m in RM_}
fig, axes = plt.subplots(2, 3, figsize=(17, 10)); axes = axes.ravel(); grid = np.linspace(0, 1, 201); sty_ = dict(zip(RM_, ["x--", "s--", "o-", "^-"]))
for m in RM_:
    tprs = []
    for k in range(K):
        if not 0 < (yT == k).sum() < len(yT): continue
        fpr, tpr, _ = roc_curve(yT == k, PP[m][:, k]); tprs.append(np.interp(grid, fpr, tpr))
        axes[k].plot(fpr, tpr, "-", lw=1.4, label=f"{m} (AUC {roc_auc_score(yT == k, PP[m][:, k]):.3f})")
    axes[5].plot(grid, np.mean(tprs, 0), lw=1.6, label=f"{m} (macro AUC {macro_auc(yT, PP[m])[0]:.3f})")
for k, a in enumerate(axes):
    a.plot([0, 1], [0, 1], "k:", lw=.7); a.set_xlabel("false positive rate"); a.set_ylabel("true positive rate"); a.legend(fontsize=6.5, loc="lower right")
    a.set_title(f"({'abcdef'[k]}) {CLASSES[k] if k < K else 'Macro-average'} vs rest (n+ = {int((yT == k).sum()) if k < K else len(yT)})")
plt.tight_layout(); savefig("fig04b_roc_curves"); plt.show()
def calib(yt, P, nb=10):
    conf = P.max(1); hit = (P.argmax(1) == yt).astype(float); b = np.minimum((conf * nb).astype(int), nb - 1)
    tab = pd.DataFrame({"b": b, "conf": conf, "hit": hit}).groupby("b").agg(conf=("conf", "mean"), acc=("hit", "mean"), n=("hit", "size"))
    ece = (tab.n * (tab.conf - tab.acc).abs()).sum() / len(yt); brier = ((P - np.eye(K)[yt]) ** 2).sum(1).mean()
    return tab, ece, brier
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2)); cal_rows = []
for m in RM_:
    tab, ece, br = calib(yT, PP[m]); cal_rows.append(dict(method=m, ECE=ece, Brier=br, LogLoss=log_loss(yT, np.clip(PP[m], 1e-7, 1), labels=list(range(K)))))
    tv = tab[tab.n >= 10]; ax[0].plot(tv.conf, tv.acc, sty_[m], label=f"{m} (ECE {ece:.3f})")
    ax[1].bar(tab.index.to_numpy() / 10 + 0.05 + 0.02 * (RM_.index(m) - 1.5), tab.n, width=.02, label=m)
ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_xlabel("mean predicted probability (top class)"); ax[0].set_ylabel("observed accuracy"); ax[0].legend(fontsize=7)
ax[0].set_title("(a) Reliability diagram (tier A, pooled; bins with >= 10 samples)"); ax[1].set_yscale("log"); ax[1].set_xlim(0, 1)
ax[1].set_xlabel("top-class probability (bin)"); ax[1].set_ylabel("samples"); ax[1].set_title("(b) Samples per confidence bin"); ax[1].legend(fontsize=7)
plt.tight_layout(); savefig("fig04c_calibration"); plt.show()
CAL = pd.DataFrame(cal_rows).set_index("method").round(4); savetab(CAL.reset_index(), "tab_calibration")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    keys, idx = method_rows(CAL.index); d = CAL.loc[keys]
    spec = [("ECE ↓", "ECE"), ("Brier score ↓", "Brier"), ("Log-loss ↓", "LogLoss")]
    t = pd.DataFrame({lab: fmt_list(d[src]) for lab, src in spec}, index=idx)
    show_table("calib", "Probability calibration (tier A, 2 laboratory tests)", t, stub="Method",
               bold=best_mask(pd.DataFrame({lab: d[src].to_numpy(float) for lab, src in spec}, index=idx), higher=False),
               note="Pooled tier-A samples of the test countries, training repeat 0. ECE = expected calibration error (10 bins). "
                    "Lower is better for all three. Bold = best in each column.")
_present()

## 11. Fixed vs random vs adaptive laboratory-test selection (issue 7)
Identical evaluation samples and identical budgets (exactly *k* tests per sample, asserted in §9). **Fixed** = subset chosen on the
validation country; **random** = a random subset per sample; **adaptive** = per-sample expected-entropy minimisation;
**oracle** = the best single subset chosen *on the test samples* (an optimistic reference, not a valid method);
**subset mean** = average over all subsets of size *k*.

In [ ]:
sel_rows = []
for k in (1, 2, 3):
    subs = ["Proposed – subset " + "+".join(LABS[j] for j in s_) for s_ in SUBSETS[k]]
    mlist = [f"Proposed – {k} tests fixed", f"Proposed – {k} tests random", f"Proposed – {k} tests adaptive"] + subs
    bt_ = boot(mlist)
    orc = {c: max(subs, key=lambda m: bt_[c]["point"][m][1]) for c in bt_}
    for c in bt_: bt_[c]["point"]["oracle"], bt_[c]["draws"]["oracle"] = bt_[c]["point"][orc[c]], bt_[c]["draws"][orc[c]]
    for c in bt_:
        bt_[c]["point"]["subset mean"] = np.mean([bt_[c]["point"][m] for m in subs], 0); bt_[c]["draws"]["subset mean"] = np.mean([bt_[c]["draws"][m] for m in subs], 0)
    sm = summarise(bt_, mlist[:3] + ["oracle", "subset mean"])
    for m in sm.index: sel_rows.append(dict(k=k, strategy=m.replace(f"Proposed – {k} tests ", ""), **sm.loc[m].to_dict()))
    for a, b in [("adaptive", "fixed"), ("adaptive", "random"), ("fixed", "random")]:
        ma, mb_ = f"Proposed – {k} tests {a}", f"Proposed – {k} tests {b}"
        d = np.mean([bt_[c]["draws"][ma] - bt_[c]["draws"][mb_] for c in bt_], 0); p = np.mean([bt_[c]["point"][ma] - bt_[c]["point"][mb_] for c in bt_], 0)
        lo, hi = np.percentile(d, [2.5, 97.5], axis=0)
        sel_rows.append(dict(k=k, strategy=f"Δ {a} − {b}", Accuracy=p[0], Accuracy_CI=f"[{lo[0]:.4f}, {hi[0]:.4f}]", MacroF1=p[1], MacroF1_CI=f"[{lo[1]:.4f}, {hi[1]:.4f}]"))
    print(f"k={k}: oracle subset per country:", {c: orc[c].replace("Proposed – subset ", "") for c in orc}, "| validation-chosen:",
          {c: "+".join(LABS[j] for j in MAINSTORE[c]["FIXED"][k]) for c in bt_})
sel_tab = pd.DataFrame(sel_rows); savetab(sel_tab, "tab_selection_strategies")
aq = acq[acq.budget == MAIN_BUDGET].groupby(["k", "test_country", "test"]).share.mean().unstack("test")[LABS] * 100
savetab(aq.reset_index(), "tab_adaptive_test_usage")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    s = sel_tab.round(4)
    SL = {"fixed": "Fixed set (chosen on the validation country)", "adaptive": "Adaptive (chosen for each sample)",
          "random": "Random set", "subset mean": "Average over all possible sets", "oracle": "Best set in hindsight (oracle)"}
    rows, idx = [], []
    for k in sorted(s.k.unique()):
        d = s[s.k == k].set_index("strategy"); g = f"{k} laboratory test{'s' if k > 1 else ''} per sample"
        for st_ in [x for x in SL if x in d.index]: rows.append(d.loc[st_]); idx.append((g, SL[st_]))
    d = pd.DataFrame(rows); idx = pd.MultiIndex.from_tuples(idx)
    t = pd.DataFrame({("Accuracy (%)", "mean"): fmt_list(d.Accuracy, f_pct), ("Accuracy (%)", "95% CI"): fmt_list(d.Accuracy_CI, lambda x: f_ci(x, 2, 100)),
                      ("Macro-F1", "mean"): fmt_list(d.MacroF1), ("Macro-F1", "95% CI"): fmt_list(d.MacroF1_CI, f_ci),
                      ("MCC", ""): fmt_list(d.MCC), ("QWK", ""): fmt_list(d.QWK)}, index=idx)
    num = pd.DataFrame(np.nan, index=idx, columns=t.columns)
    for c, src in [(("Accuracy (%)", "mean"), "Accuracy"), (("Macro-F1", "mean"), "MacroF1"), (("MCC", ""), "MCC"), (("QWK", ""), "QWK")]:
        num[c] = d[src].to_numpy(float)
    show_table("selection", "Which laboratory tests to run: fixed vs adaptive vs random selection (proposed model, tier A)", t, stub="Selection strategy",
               bold=best_mask(num, by_group=True, exclude=[SL["oracle"]]),
               note="Identical samples and label budget for every strategy. The oracle picks, per country, the best set after seeing the "
                    "test results; it is an upper reference, not a usable strategy, and is not ranked. Bold = best usable strategy "
                    "for each number of tests.")
    DL = {"Δ adaptive − fixed": "Adaptive − fixed", "Δ adaptive − random": "Adaptive − random", "Δ fixed − random": "Fixed − random"}
    rows, idx, B = [], [], []
    for k in sorted(s.k.unique()):
        d = s[s.k == k].set_index("strategy")
        for key, lab in DL.items():
            if key not in d.index: continue
            r = d.loc[key]; ea = (lambda lo_hi: lo_hi[0] > 0 or lo_hi[1] < 0)
            xa, xf = ea(ci_bounds(r.Accuracy_CI)), ea(ci_bounds(r.MacroF1_CI))
            rows.append([f_pct(r.Accuracy, 2, True), f_ci(r.Accuracy_CI, 2, 100, True), f_num(r.MacroF1, 4, True), f_ci(r.MacroF1_CI, 4, 1, True)])
            B.append([xa, xa, xf, xf]); idx.append((f"{k} laboratory test{'s' if k > 1 else ''} per sample", lab))
    cols = pd.MultiIndex.from_tuples([("Δ accuracy (percentage points)", "mean"), ("Δ accuracy (percentage points)", "95% CI"),
                                      ("Δ macro-F1", "mean"), ("Δ macro-F1", "95% CI")])
    show_table("selection_diff", "Differences between selection strategies (proposed model, tier A)", pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(idx), columns=cols),
               stub="Comparison", bold=np.array(B, bool),
               note="Positive = first strategy better. 95% CI from the site-clustered bootstrap. Bold = CI excludes zero.")
    a = aq.round(1); rows, idx = [], []
    for k in sorted(set(a.index.get_level_values(0))):
        for c in countries_in(a.loc[k].index):
            rows.append(fmt_list(a.loc[(k, c), LABS], lambda v: f_num(v, 1))); idx.append((f"{k} test{'s' if k > 1 else ''} per sample", c))
    t = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(idx),
                     columns=pd.MultiIndex.from_tuples([("Samples in which the test was chosen (%)", PNAME[f]) for f in LABS]))
    show_table("adaptive_use", "Laboratory tests chosen by the adaptive policy", t, stub="Test country",
               note=f"Proposed model, {100 * MAIN_BUDGET:.0f} % target labels. k tests are chosen for every sample, so each row sums to k × 100 %.")
_present()

## 12. Statistical inference (issue 9)
Training repeats on the same country are **not** independent samples, and samples from one site are correlated. Therefore:
- **Within a country:** sites are resampled with replacement (cluster bootstrap, `N_BOOT` draws); each draw averages the repeats.
- **For the countries studied** (conditional inference): the equal-weight mean difference over countries with a stratified
  site-bootstrap interval (draws averaged over countries) and a bootstrap p-value.
- **Generalisation to new countries:** per-country differences are combined by a random-effects meta-analysis (DerSimonian–Laird τ²)
  with the modified Hartung–Knapp interval (variance factor max(1, q), t-distribution with *k − 1* df). A country in which both
  methods agree on every sample has a bootstrap SE of 0; its SE is floored at 1/n so it cannot dominate the weights.
  With 3 countries the random-effects interval is wide by construction; per-country intervals are reported as well.

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty(len(p)); run = 0.0
    for r, i in enumerate(o): run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj
COMP = [("Proposed – 2 tests fixed", "Black box – 2 tests"), ("Proposed – 2 tests fixed", "Formula rule (last value) – 2 tests"),
        ("Proposed – 2 tests fixed", "Formula rule (assume pass) – 2 tests"), ("Proposed – 2 tests adaptive", "Proposed – 2 tests fixed"),
        ("Proposed – 2 tests fixed", "Persistence"), ("Proposed – 2 tests fixed", "Ablation: without current sensors"),
        ("Proposed – 2 tests fixed", "Ablation: without history"), ("Proposed – 2 tests fixed", "Ablation: without monotonicity constraint"),
        ("Proposed – 0 tests", "Black box – 0 tests")]
BTC = boot(sorted({m for p in COMP for m in p}))
def meta(d, se, n):
    # DerSimonian-Laird random effects + modified Hartung-Knapp (variance factor max(1, q); Roever et al. 2015).
    # A country whose two methods agree on every sample has bootstrap SE 0; its SE is floored at one sample's worth (1/n)
    # so that it cannot receive unlimited weight.
    d, se = np.asarray(d, float), np.maximum(np.asarray(se, float), 1 / np.asarray(n, float)); w = 1 / se ** 2; k = len(d)
    mu_f = (w * d).sum() / w.sum(); Q = (w * (d - mu_f) ** 2).sum(); c = w.sum() - (w ** 2).sum() / w.sum()
    tau2 = max(0.0, (Q - (k - 1)) / c) if k > 1 else 0.0; ws = 1 / (se ** 2 + tau2); mu = (ws * d).sum() / ws.sum()
    if k > 1:
        q = (ws * (d - mu) ** 2).sum() / (k - 1); se_hk = np.sqrt(max(1.0, q) / ws.sum()); tq = sps.t.ppf(.975, k - 1)
        lo, hi = mu - tq * se_hk, mu + tq * se_hk; p = 2 * sps.t.sf(abs(mu / max(se_hk, 1e-12)), k - 1)
    else: lo, hi, p = mu - 1.96 * se[0], mu + 1.96 * se[0], 2 * sps.norm.sf(abs(mu / se[0]))
    i2 = max(0.0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0.0
    return mu, lo, hi, p, tau2, i2
inf_rows, forest = [], []
for a, b in COMP:
    for i, met in enumerate(FM[:2]):
        ds, ses, ns, dall = [], [], [], []
        for c in BTC:
            dd = BTC[c]["draws"][a][:, i] - BTC[c]["draws"][b][:, i]; pt = BTC[c]["point"][a][i] - BTC[c]["point"][b][i]
            lo, hi = np.percentile(dd, [2.5, 97.5]); ds.append(pt); ses.append(dd.std()); ns.append(BTC[c]["n"]); dall.append(dd)
            forest.append(dict(comparison=f"{a} vs {b}", metric=met, country=c, diff=pt, lo=lo, hi=hi))
        mu, lo, hi, p, tau2, i2 = meta(ds, ses, ns)
        de = np.mean(dall, 0); blo, bhi = np.percentile(de, [2.5, 97.5]); pb = min(1.0, 2 * min((de <= 0).mean(), (de >= 0).mean()) + 1 / len(de))
        inf_rows.append(dict(comparison=f"{a} vs {b}", metric=met, mean_diff=np.mean(ds), boot_CI_low=blo, boot_CI_high=bhi, p_boot=pb,
                             RE_diff=mu, RE_CI_low=lo, RE_CI_high=hi, p_RE=p, tau2=tau2, I2_pct=i2,
                             countries_positive=f"{sum(np.array(ds) > 0)}/{len(ds)}",
                             countries_CI_excludes_0=f"{sum(1 for f in forest[-len(ds):] if f['lo'] > 0 or f['hi'] < 0)}/{len(ds)}"))
        if met == "Accuracy":      # effect size: share of the comparator's errors that the first method avoids
            acc_a = np.mean([BTC[c]["point"][a][0] for c in BTC]); acc_b = np.mean([BTC[c]["point"][b][0] for c in BTC])
            inf_rows[-1]["error_rate_reduction_pct"] = 100 * (acc_a - acc_b) / max(1 - acc_b, 1e-9)
INF = pd.DataFrame(inf_rows)
INF["p_boot_holm"] = INF.groupby("metric").p_boot.transform(lambda s: holm(s.to_numpy()))   # Holm over the comparisons, per metric
INF["p_RE_holm"] = INF.groupby("metric").p_RE.transform(lambda s: holm(s.to_numpy()))
savetab(INF, "tab_inference_meta"); FOREST = pd.DataFrame(forest); savetab(FOREST, "tab_forest")
fd = FOREST[FOREST.metric == "MacroF1"]; comps = fd.comparison.unique()
fig, ax = plt.subplots(figsize=(10, 0.42 * (len(fd) + len(comps)) + 1.5)); yk, ypos, labels_ = 0.0, [], []
for cp in comps:
    for _, r in fd[fd.comparison == cp].iterrows():
        ax.errorbar(r["diff"], yk, xerr=[[r["diff"] - r.lo], [r.hi - r["diff"]]], fmt="o", color="#4c72b0", capsize=3)
        ypos.append(yk); labels_.append(f"{cp}  [{r.country}]"); yk += 1
    pr_ = INF[(INF.comparison == cp) & (INF.metric == "MacroF1")].iloc[0]
    ax.errorbar(pr_.RE_diff, yk, xerr=[[pr_.RE_diff - pr_.RE_CI_low], [pr_.RE_CI_high - pr_.RE_diff]], fmt="D", color="#c44e52", capsize=3)
    ypos.append(yk); labels_.append(f"{cp}  [random effects, mHK]"); yk += 1.5
ax.axvline(0, c="k", lw=.8); ax.set_yticks(ypos); ax.set_yticklabels(labels_, fontsize=7); ax.invert_yaxis(); ax.set_xlabel("difference in macro-F1 (tier A)")
ax.set_title("Per-country (site-bootstrap 95% CI) and pooled random-effects differences"); plt.tight_layout(); savefig("fig05_forest"); plt.show()
rv = res[(res.budget == MAIN_BUDGET) & (res.tier == "A") & res.method.isin(KEY)].groupby(["test_country", "method"]).MacroF1.agg(["mean", "std"]).round(4)
savetab(rv.reset_index(), "tab_repeat_variability")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    I = inf_pairs(INF.round(4), COMP); pf = "Proposed – 2 tests fixed"
    G1, G2 = f"{mname(pf)} compared with", "Other comparisons"
    def table(num_, metric, title, pp):
        d = I[I.metric == metric].copy()
        d["_g"] = np.where(d.pair_a == pf, 0, 1); d = d.sort_values("_g", kind="stable")
        sc, dd, sg = (100, 2, "pp") if pp else (1, 4, "")
        idx = [(G1, mname(b)) if a == pf else (G2, f"{mname(a)} vs {mname(b)}") for a, b in zip(d.pair_a, d.pair_b)]
        rows, B = [], []
        for _, r in d.iterrows():
            row = [f_num(r.mean_diff * sc, dd, True), f"[{f_num(r.boot_CI_low * sc, dd, True)}, {f_num(r.boot_CI_high * sc, dd, True)}]",
                   f_p(r.p_boot_holm, 1 / N_BOOT, 3), f_num(r.RE_diff * sc, dd, True),
                   f"[{f_num(r.RE_CI_low * sc, dd, True)}, {f_num(r.RE_CI_high * sc, dd, True)}]", f_p(r.p_RE_holm),
                   f_num(r.tau2, 4), f_num(r.I2_pct, 1), str(r.countries_positive).replace("/", " of "),
                   str(r.countries_CI_excludes_0).replace("/", " of ")]
            b = [False] * len(row); b[2] = float(r.p_boot_holm) < 0.05; b[5] = float(r.p_RE_holm) < 0.05
            if pp: row.append(f_num(r.error_rate_reduction_pct, 1)); b.append(False)
            rows.append(row); B.append(b)
        lab = "Δ accuracy (pp)" if pp else "Δ macro-F1"
        cols = [(lab, ""), ("Site bootstrap", "95% CI"), ("Site bootstrap", "p (Holm)"), ("Random effects", "Δ"), ("Random effects", "95% CI"),
                ("Random effects", "p (Holm)"), ("Random effects", "τ²"), ("Random effects", "I² (%)"), ("Countries", "Δ > 0"), ("Countries", "CI excludes 0")]
        if pp: cols.append(("Error-rate reduction (%)", ""))
        show_table(num_, title, pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(idx), columns=pd.MultiIndex.from_tuples(cols)),
                   stub="Comparison", bold=np.array(B, bool),
                   note=f"Δ = first method − comparator, mean over the {len(PRIMARY)} test countries (tier A). Site bootstrap: {N_BOOT:,} draws "
                        "of sites within each country (p ≤ 0.001 is the resolution limit). Random effects: DerSimonian–Laird with the modified "
                        "Hartung–Knapp interval across countries (wide with three countries); τ² on the 0–1 scale."
                        + (" Error-rate reduction = share of the comparator's errors avoided by the first method." if pp else "")
                        + " p-values are Holm-adjusted over the comparisons of this table. Bold = adjusted p < 0.05.")
    table("stats_f1", "MacroF1", "Statistical comparison of methods: macro-F1", False)
    table("stats_acc", "Accuracy", "Statistical comparison of methods: accuracy", True)
    keys, idx = method_rows(rv.index.get_level_values("method")); cs = countries_in(rv.index.get_level_values("test_country"))
    t = pd.DataFrame({(f"Macro-F1, mean ± SD over {N_REPEATS} training repeats", c):
                      [f"{rv.loc[(c, m), 'mean']:.4f} ± {rv.loc[(c, m), 'std']:.4f}" for m in keys] for c in cs}, index=idx)
    show_table("repeats", "Variability between training repeats (tier A)", t, stub="Method",
               note="Descriptive only; statistical inference uses the site bootstrap (" + tref("stats_f1", "stats_acc") + "). SD = 0 for methods without training randomness.")
_present()

## 12b. Extended baselines on the same test records, and tests against the formula rule
Every method below is evaluated on the identical tier-A samples of each test country (same repeats, same 2 laboratory tests).
Added: a **calibrated formula rule** (probability of each true class given the rule's class, estimated on the training rows only,
so that its Brier score and log-loss are a fair comparison), **logistic regression** and **random forest** with the black-box
inputs, and both learners with the CCME-consistency features. All learners use the same training rows, acquisition masks and
sample weights as the XGBoost models (verified). Differences to the formula rule are tested with the site bootstrap and the
random-effects model of §12; p-values are Holm-adjusted within each metric.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
RULE_M, PF, BBX = "Formula rule (last value) – 2 tests", "Proposed – 2 tests fixed", "Black box – 2 tests"
XN = {"rcal": "Formula rule (last value), calibrated – 2 tests", "lr": "Logistic regression – 2 tests", "rf": "Random forest – 2 tests",
      "lr+": "Logistic regression + CCME features – 2 tests", "rf+": "Random forest + CCME features – 2 tests"}
def make_lr(seed): return Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                                    ("sc", StandardScaler()), ("m", LogisticRegression(max_iter=3000, C=1.0))])
def make_rf(seed): return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, max_features="sqrt", n_jobs=-1, random_state=seed)
def train_sk(make, src, lab, w_t, seed, formula):
    # identical rows, acquisition masks and sample weights as train() (§8)
    rng = np.random.default_rng(seed); rows = np.r_[src, lab]
    R = np.concatenate([rows] * AUG_COPIES); M = random_mask(len(R), rng.integers(0, 4, len(R)), rng)
    w = compute_sample_weight("balanced", y_all[R]) * np.where(np.isin(R, lab), w_t, 1.0); m = make(seed); X = featurize(R, M, formula)
    return m.fit(X, y_all[R], m__sample_weight=w) if isinstance(m, Pipeline) else m.fit(X, y_all[R], sample_weight=w)
def predict_sk(m, rows, M, formula):
    P = np.zeros((len(rows), K)); P[:, m.classes_] = m.predict_proba(featurize(rows, M, formula))
    return constrain(P, rows, M) if formula else P
def rule_calibrated(trw, rows, M_tr, M):
    C_ = np.ones((K, K)); np.add.at(C_, (rule_P(trw, M_tr, "last").argmax(1), y_all[trw]), 1)    # Laplace-smoothed, training rows only
    return (C_ / C_.sum(1, keepdims=True))[rule_P(rows, M, "last").argmax(1)]
XB_SPLIT_CHECK = {}
for c in PRIMARY:
    T, FIXED = TUNED[c]; src_all = sources(c); lab, _ = target_split(c, MAIN_BUDGET)
    for rep, e in enumerate(MAINSTORE[c]["reps"]):
        ev = e["ev"]; M2 = fixed_mask(len(ev), FIXED[2]); seed = 1000 * rep + int(100 * MAIN_BUDGET)
        src = np.random.default_rng(100 + rep).choice(src_all, min(TRAIN_CAP, len(src_all)), replace=False); trw = np.r_[src, lab]
        if rep == 0:   # the reconstructed training split must reproduce the stored black-box predictions of §9
            XB_SPLIT_CHECK[c] = float(np.abs(predict(train(src, lab, T[False]["prm"], T[False]["w"], seed, False), ev, M2, False) - e["P"][BBX]).max())
        ck = ck_load(f"xbase_{c}_rep{rep}")
        if ck is None:
            t_x = time.time(); ck = {XN["rcal"]: rule_calibrated(trw, ev, fixed_mask(len(trw), FIXED[2]), M2)}
            for key, make, fm in [("lr", make_lr, False), ("rf", make_rf, False), ("lr+", make_lr, True), ("rf+", make_rf, True)]:
                ck[XN[key]] = predict_sk(train_sk(make, src, lab, T[fm]["w"], seed, fm), ev, M2, fm)
            ck = {k: v.astype(np.float32) for k, v in ck.items()}; ck_save(f"xbase_{c}_rep{rep}", ck)
            print(f"[{c}] rep {rep + 1}/{len(MAINSTORE[c]['reps'])}: extended baselines trained ({time.time() - t_x:.0f}s)", flush=True)
        e["P"].update(ck)
print("max |difference| between re-trained and stored black-box probabilities:", {c: f"{v:.1e}" for c, v in XB_SPLIT_CHECK.items()})
XB = ["Persistence", "Forecast only (history)", RULE_M, XN["rcal"], XN["lr"], XN["rf"], BBX, XN["lr+"], XN["rf+"], PF]
BTX = boot(XB); BTXA = boot_auc(XB)
XBT = summarise(BTX, XB).join(summarise_auc(BTXA, XB))
yT0 = np.concatenate([y_all[MAINSTORE[c]["reps"][0]["ev"][TIER_A[MAINSTORE[c]["reps"][0]["ev"]]]] for c in BT])
for m in XB:
    P_ = np.concatenate([MAINSTORE[c]["reps"][0]["P"][m][TIER_A[MAINSTORE[c]["reps"][0]["ev"]]].astype(float) for c in BT])
    _, ece_, br_ = calib(yT0, P_); XBT.loc[m, "ECE"] = ece_; XBT.loc[m, "Brier"] = br_
    XBT.loc[m, "LogLoss"] = log_loss(yT0, np.clip(P_, 1e-7, 1), labels=list(range(K)))
XBT = XBT.round(4); savetab(XBT.reset_index(), "tab_extended_baselines")
def paired(bt, a, b, i):
    # Δ = a − b on metric i: equal-weight mean over countries, site-bootstrap CI and p, random effects (meta(), §12)
    ds, ses, ns, dall = [], [], [], []
    for c in bt:
        dd = bt[c]["draws"][a][:, i] - bt[c]["draws"][b][:, i]
        ds.append(bt[c]["point"][a][i] - bt[c]["point"][b][i]); ses.append(dd.std()); ns.append(bt[c]["n"]); dall.append(dd)
    mu, lo, hi, p, tau2, i2 = meta(ds, ses, ns); de = np.mean(dall, 0); blo, bhi = np.percentile(de, [2.5, 97.5])
    pb = min(1.0, 2 * min((de <= 0).mean(), (de >= 0).mean()) + 1 / len(de))
    return dict(mean_diff=float(np.mean(ds)), boot_CI_low=blo, boot_CI_high=bhi, p_boot=pb, RE_diff=mu, RE_CI_low=lo, RE_CI_high=hi, p_RE=p,
                I2_pct=i2, countries_positive=f"{int(sum(np.array(ds) > 0))}/{len(ds)}")
VSRULE = pd.DataFrame([dict(method=m, metric=met, **paired(BTX, m, RULE_M, i)) for m in XB if m != RULE_M
                       for i, met in enumerate(["Accuracy", "MacroF1", "MCC"])])
VSRULE["p_boot_holm"] = VSRULE.groupby("metric").p_boot.transform(lambda s: holm(s.to_numpy()))
VSRULE["p_RE_holm"] = VSRULE.groupby("metric").p_RE.transform(lambda s: holm(s.to_numpy()))
VSRULE = VSRULE.round(4); savetab(VSRULE, "tab_tests_vs_formula_rule")
# test country completely excluded from training (0 % target labels), by country
ZM_ = ["Persistence", RULE_M, BBX, PF]
ZC = (res[(res.budget == 0.0) & (res.tier == "A") & res.method.isin(ZM_)]
      .groupby(["test_country", "method"])[["Accuracy", "MacroF1", "MCC"]].agg(["mean", "std"]).round(4))
savetab(ZC.reset_index(), "tab_zero_label_by_country")
print("extended baselines done")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    G = [("No learning", [RULE_M, XN["rcal"]]), ("Simple baselines", ["Persistence", "Forecast only (history)"]),
         ("Machine learning without CCME features", [XN["lr"], XN["rf"], BBX]),
         ("Machine learning with CCME-consistency features", [XN["lr+"], XN["rf+"], PF])]
    NM = {RULE_M: "Formula rule (last value)", XN["rcal"]: "Formula rule (last value), calibrated", "Persistence": "Persistence (class at t−1)",
          "Forecast only (history)": "Forecast from site history only", XN["lr"]: "Logistic regression", XN["rf"]: "Random forest",
          BBX: "XGBoost (black box)", XN["lr+"]: "Logistic regression + CCME features", XN["rf+"]: "Random forest + CCME features",
          PF: "Proposed: XGBoost + CCME features + constraint"}
    keys = [(g, m) for g, ms in G for m in ms if m in XBT.index]; d = XBT.loc[[m for _, m in keys]]
    idx = pd.MultiIndex.from_tuples([(g, NM[m]) for g, m in keys])
    t = pd.DataFrame({("Accuracy (%)", "mean"): fmt_list(d.Accuracy, f_pct), ("Accuracy (%)", "95% CI"): fmt_list(d.Accuracy_CI, lambda s: f_ci(s, 2, 100)),
                      ("Macro-F1", "mean"): fmt_list(d.MacroF1), ("Macro-F1", "95% CI"): fmt_list(d.MacroF1_CI, f_ci),
                      ("MCC", "mean"): fmt_list(d.MCC), ("MCC", "95% CI"): fmt_list(d.MCC_CI, f_ci),
                      ("ROC-AUC", ""): fmt_list(d.ROC_AUC), ("Brier ↓", ""): fmt_list(d.Brier), ("Log-loss ↓", ""): fmt_list(d.LogLoss),
                      ("ECE ↓", ""): fmt_list(d.ECE)}, index=idx)
    num = pd.DataFrame(np.nan, index=idx, columns=t.columns)
    for c, src in [(("Accuracy (%)", "mean"), "Accuracy"), (("Macro-F1", "mean"), "MacroF1"), (("MCC", "mean"), "MCC"), (("ROC-AUC", ""), "ROC_AUC"),
                   (("Brier ↓", ""), "Brier"), (("Log-loss ↓", ""), "LogLoss"), (("ECE ↓", ""), "ECE")]:
        num[c] = d[src].to_numpy(float)
    show_table("xbase", f"All baselines on the same test records (tier A, 2 laboratory tests, {budget_txt()})", t, stub="Method",
               bold=best_mask(num, higher={("Brier ↓", ""): False, ("Log-loss ↓", ""): False, ("ECE ↓", ""): False}),
               note="Identical samples, repeats and laboratory tests for every method; all learners use the same training rows, acquisition masks "
                    "and sample weights. Calibrated rule = probability of each true class given the rule's class, estimated on training rows "
                    "only. 95% CI = site-clustered bootstrap; Brier, log-loss and ECE on pooled samples of repeat 0. Bold = best in each column.")
    v = VSRULE.set_index(["method", "metric"]); ms = [m for _, mm in G for m in mm if m != RULE_M and (m, "Accuracy") in v.index]
    idx = pd.MultiIndex.from_tuples([(g, NM[m]) for g, mm in G for m in mm if m in ms]); rows, B = [], []
    for m in ms:
        row, b = [], []
        for met, sc, dd in [("Accuracy", 100, 2), ("MacroF1", 1, 4), ("MCC", 1, 4)]:
            r = v.loc[(m, met)]
            row += [f_num(r.mean_diff * sc, dd, True), f"[{f_num(r.boot_CI_low * sc, dd, True)}, {f_num(r.boot_CI_high * sc, dd, True)}]",
                    f_p(r.p_boot_holm, 1 / N_BOOT, 3)]
            b += [False, False, float(r.p_boot_holm) < 0.05]
        rows.append(row); B.append(b)
    cols = pd.MultiIndex.from_tuples([(lab, w) for lab in ("Δ accuracy (pp)", "Δ macro-F1", "Δ MCC") for w in ("mean", "95% CI", "p (Holm)")])
    show_table("xbase_tests", "Is any method better than the formula rule? Paired differences to the rule (tier A)", pd.DataFrame(rows, index=idx, columns=cols),
               stub="Method − formula rule (last value)", bold=np.array(B, bool),
               note="Δ = method − formula rule, equal-weight mean over the test countries; site-bootstrap 95% CI and p, Holm-adjusted over the "
                    "methods within each metric (with 1,000 bootstrap draws the smallest attainable adjusted p is 0.009); random-effects estimates in "
                    "tab_tests_vs_formula_rule.csv. Positive = better than the rule. "
                    "Bold = adjusted p < 0.05.")
    ZN = {"Persistence": "Persistence (class at t−1)", RULE_M: "Formula rule (last value)", BBX: "XGBoost (black box)", PF: "Proposed"}
    rows, nums, idx = [], [], []
    for c in countries_in(ZC.index.get_level_values("test_country")):
        for m in [m for m in ZN if (c, m) in ZC.index]:
            r = ZC.loc[(c, m)]
            rows.append([f"{100 * r[('Accuracy', 'mean')]:.2f} ± {100 * r[('Accuracy', 'std')]:.2f}", f"{r[('MacroF1', 'mean')]:.4f} ± {r[('MacroF1', 'std')]:.4f}",
                         f"{r[('MCC', 'mean')]:.4f} ± {r[('MCC', 'std')]:.4f}"])
            nums.append([r[("Accuracy", "mean")], r[("MacroF1", "mean")], r[("MCC", "mean")]]); idx.append((f"Test country: {c}", ZN[m]))
    cols = pd.MultiIndex.from_tuples([("Accuracy (%)", "mean ± SD"), ("Macro-F1", "mean ± SD"), ("MCC", "mean ± SD")]); idx = pd.MultiIndex.from_tuples(idx)
    show_table("zeroshot_country", "Test country completely excluded from training (0 % target labels), by country (tier A, 2 laboratory tests)",
               pd.DataFrame(rows, index=idx, columns=cols), stub="Method", bold=best_mask(pd.DataFrame(nums, index=idx, columns=cols), by_group=True),
               note=f"No sample of the test country is used for training or model selection. Mean ± SD over {N_REPEATS} training repeats. "
                    "Bold = best mean within each country.")
_present()

## 12c. Practical value of the reduced laboratory panel
The class in this study is a deterministic function of eight values, so running all four laboratory tests always gives the correct
class. The practical questions are therefore (i) how much accuracy is kept when fewer tests are run, (ii) how well each method
works when the last laboratory value is old or missing, (iii) whether a model can flag the samples in which an unmeasured test
fails, and (iv) how many samples must still receive the full panel to reach a target accuracy (**selective full-panel testing**),
and at what cost and turnaround. Costs use published fee schedules of two US public environmental laboratories (configuration, §1);
BOD needs a 5-day incubation (Standard Methods 5210 B).

In [ ]:
# (i) number of laboratory tests: formula rule, black box and proposed with the same fixed test sets
KT_ = {k: [f"Formula rule (last value) – {k} tests", f"Black box – {k} tests", tname(k)] for k in (0, 1, 2, 3)}
BTK = boot(sorted({m for v in KT_.values() for m in v}))
few_rows = []
for k, (ru, bb_, pr_) in KT_.items():
    pts = {m: np.mean([BTK[c]["point"][m] for c in BTK], 0) for m in (ru, bb_, pr_)}
    da, df1, dm = paired(BTK, pr_, ru, 0), paired(BTK, pr_, ru, 1), paired(BTK, pr_, ru, 2)
    few_rows.append(dict(k=k, tests=("none" if k == 0 else "+".join(LABS[j] for j in MAINSTORE[PRIMARY[0]]["FIXED"][k])),
                         rule_Accuracy=pts[ru][0], rule_MacroF1=pts[ru][1], rule_MCC=pts[ru][2],
                         bb_Accuracy=pts[bb_][0], bb_MacroF1=pts[bb_][1], bb_MCC=pts[bb_][2],
                         prop_Accuracy=pts[pr_][0], prop_MacroF1=pts[pr_][1], prop_MCC=pts[pr_][2],
                         dAcc=da["mean_diff"], dAcc_lo=da["boot_CI_low"], dAcc_hi=da["boot_CI_high"], dAcc_p=da["p_boot"],
                         dF1=df1["mean_diff"], dF1_lo=df1["boot_CI_low"], dF1_hi=df1["boot_CI_high"], dF1_p=df1["p_boot"],
                         dMCC=dm["mean_diff"], dMCC_lo=dm["boot_CI_low"], dMCC_hi=dm["boot_CI_high"], dMCC_p=dm["p_boot"]))
FEWER = pd.DataFrame(few_rows); FEWER["dAcc_p_holm"] = holm(FEWER.dAcc_p.to_numpy())        # Holm over the numbers of tests
FEWER = FEWER.round(4); savetab(FEWER, "tab_accuracy_by_number_of_tests")
# (ii) age and availability of the last laboratory value
def _age_last(c, ev):
    # age (days) of the older last genuine value of the unmeasured tests; NaN if one of them has none within LAST_MAX_AGE_DAYS
    hid = ~fixed_mask(len(ev), MAINSTORE[c]["FIXED"][2]); return np.where(hid, LAST_AGE[ev], -1.0).max(1)
GG = {"last genuine values ≤ 31 days old": lambda c, ev: _age_last(c, ev) <= 31,
      "last genuine values 32–92 days old": lambda c, ev: (_age_last(c, ev) > 31) & (_age_last(c, ev) <= 92),
      f"last genuine values 93–{LAST_MAX_AGE_DAYS} days old": lambda c, ev: _age_last(c, ev) > 92,
      "last value of an unmeasured test not available": lambda c, ev: np.isnan(_age_last(c, ev))}
GM = [RULE_M, BBX, PF]; gg_rows = []
for gname, fn in GG.items():
    bt_ = boot(GM, fn)
    if not bt_: continue
    sm = summarise(bt_, GM); da = paired(bt_, PF, RULE_M, 0); df1 = paired(bt_, PF, RULE_M, 1)
    gg_rows.append(dict(group=gname, n=sum(bt_[c]["n"] for c in bt_), countries=len(bt_),
                        **{f"{tag}_{met}": sm.loc[m, met] for tag, m in [("rule", RULE_M), ("bb", BBX), ("prop", PF)] for met in ("Accuracy", "MacroF1")},
                        dAcc=da["mean_diff"], dAcc_lo=da["boot_CI_low"], dAcc_hi=da["boot_CI_high"], dAcc_p=da["p_boot"],
                        dF1=df1["mean_diff"], dF1_lo=df1["boot_CI_low"], dF1_hi=df1["boot_CI_high"], dF1_p=df1["p_boot"]))
GAPS = pd.DataFrame(gg_rows); GAPS["dAcc_p_holm"] = holm(GAPS.dAcc_p.to_numpy())             # Holm over the groups
GAPS = GAPS.round(4); savetab(GAPS, "tab_accuracy_by_last_lab_value")
# (iii) flagging hidden failures (an unmeasured test fails its objective)
HIDX_L = [BASE_COLS.index(f"{f}_hist_exceed_rate") for f in LABS]
def _below_ceiling(P, ceil_): return (P * (np.arange(K)[None, :] < ceil_[:, None])).sum(1)
def _top(score, frac, rng):
    perm = rng.permutation(len(score)); o = perm[np.argsort(-score[perm], kind="stable")]; return o[: int(np.ceil(frac * len(score)))]
HDN = {"Proposed: probability of a class below the ceiling": "prop", "Black box: probability of a class below the ceiling": "bb",
       "Site history: probability from past exceedance rates": "hist", "Formula rule: last value of an unmeasured test fails": "last"}
hd_rows = []
for c in PRIMARY:
    F2 = MAINSTORE[c]["FIXED"][2]
    for rep, e in enumerate(MAINSTORE[c]["reps"]):
        ev = e["ev"]; s = TIER_A[ev]; rows = ev[s]; M2 = fixed_mask(len(rows), F2); yh = hidden_fail(c, ev)[s]
        if yh.sum() < 10 or (~yh).sum() < 10: continue
        ceil_ = ceiling_class(rows, M2); hr = np.nan_to_num(XBASE[rows][:, HIDX_L], nan=0.0) * ~M2
        sc_ = {"prop": _below_ceiling(e["P"][PF][s].astype(float), ceil_), "bb": _below_ceiling(e["P"][BBX][s].astype(float), ceil_),
               "hist": 1 - np.prod(1 - hr, 1), "last": (rule_P(rows, M2, "last").argmax(1) < ceil_).astype(float)}
        for nm, key in HDN.items():
            r_ = dict(test_country=c, repeat=rep, method=nm, n=len(rows), positives=int(yh.sum()), ROC_AUC=roc_auc_score(yh, sc_[key]),
                      PR_AUC=average_precision_score(yh, sc_[key]))
            for a in (0.1, 0.2):
                top = _top(sc_[key], a, np.random.default_rng(rep)); r_[f"recall_at_{int(100 * a)}pct_flagged"] = yh[top].sum() / yh.sum()
            hd_rows.append(r_)
HDET = pd.DataFrame(hd_rows); savetab(HDET, "tab_hidden_failure_detection_all")
HDET_S = HDET.groupby(["test_country", "method"]).mean(numeric_only=True).groupby("method").mean().drop(columns=["repeat"]).round(4)
HDET_COUNTRIES = sorted(set(HDET.test_country), key=list(BT).index); savetab(HDET_S.reset_index(), "tab_hidden_failure_detection")
# (iv) selective full-panel testing: the least-certain samples receive the two remaining tests (and are then classified exactly)
QG = np.round(np.arange(0, 1.0001, 0.01), 2); QB = (0.1, 0.2, 0.3)
TRN = {"Proposed: least confident first": "prop", "Black box: least confident first": "bb", "Calibrated rule: least confident first": "rcal",
       "Formula rule: risk from last values and history first": "rulehist", "Formula rule: random samples": "rand"}
TRI, CORR = {nm: {} for nm in TRN}, {nm: {} for nm in TRN}
for c in PRIMARY:
    F2 = MAINSTORE[c]["FIXED"][2]
    for rep, e in enumerate(MAINSTORE[c]["reps"]):
        ev = e["ev"]; s = TIER_A[ev]; rows = ev[s]; y_ = y_all[rows]; n = len(rows); M2 = fixed_mask(n, F2)
        hr = np.nan_to_num(XBASE[rows][:, HIDX_L], nan=0.0) * ~M2; ceil_ = ceiling_class(rows, M2); rule_c = rule_P(rows, M2, "last").argmax(1)
        Pp, Pb, Pc = (e["P"][m][s].astype(float) for m in (PF, BBX, XN["rcal"]))
        plan = {"prop": (Pp.argmax(1), 1 - Pp.max(1)), "bb": (Pb.argmax(1), 1 - Pb.max(1)), "rcal": (Pc.argmax(1), 1 - Pc.max(1)),
                "rulehist": (rule_c, (rule_c < ceil_).astype(float) + (1 - np.prod(1 - hr, 1))),
                "rand": (rule_c, np.random.default_rng(500 + rep).random(n))}
        for nm, key in TRN.items():
            yp, sc_ = plan[key]; wrong = yp != y_; rng_ = np.random.default_rng(rep); perm = rng_.permutation(n)
            o = perm[np.argsort(-sc_[perm], kind="stable")]; cw = np.r_[0, np.cumsum(wrong[o])]; ns_ = np.ceil(QG * n).astype(int)
            if key == "rand": acc_ = 1 - wrong.sum() * (n - ns_) / n / n          # exact expectation of random selection
            else: acc_ = 1 - (wrong.sum() - cw[ns_]) / n
            TRI[nm].setdefault(c, []).append(acc_)
            corr = {}
            for q in QB:
                sent = np.zeros(n, bool); sent[o[: int(np.ceil(q * n))]] = True
                corr[q] = np.where(sent, 1.0, (~wrong).astype(float)) if key != "rand" else 1 - wrong * (1 - q)
            CORR[nm].setdefault(c, []).append(corr)
TRIC = pd.DataFrame({nm: np.mean([np.mean(v, 0) for v in d.values()], 0) for nm, d in TRI.items()}, index=QG)   # equal-weight over countries
savetab(TRIC.reset_index().rename(columns={"index": "share_sent_to_full_panel"}), "tab_selective_testing_curves")
def _tri_draws(nm, q):
    out = []
    for ci, c in enumerate(PRIMARY):
        ev = MAINSTORE[c]["reps"][0]["ev"]; rows = ev[TIER_A[ev]]; su, sidx = np.unique(site_ids[rows], return_inverse=True); S = len(su)
        W = np.random.default_rng(3000 + ci).multinomial(S, np.full(S, 1 / S), size=N_BOOT).astype(float); den = W @ np.bincount(sidx, minlength=S)
        out.append(np.mean([(W @ np.bincount(sidx, weights=cr[q], minlength=S)) / den for cr in CORR[nm][c]], 0))
    return np.mean(out, 0)
REF_T = "Formula rule: risk from last values and history first"; tri_rows = []
for nm in TRN:
    r_ = dict(method=nm)
    for q in (0.0, 0.1, 0.2, 0.3, 0.5): r_[f"acc_at_{int(100 * q)}pct"] = float(TRIC.loc[q, nm])
    for tgt in (0.97, 0.98, 0.99):
        hit = TRIC.index[TRIC[nm].to_numpy() >= tgt]; r_[f"share_for_{int(100 * tgt)}pct"] = float(hit.min()) if len(hit) else np.nan
    dr = _tri_draws(nm, 0.2); r_["acc20_CI_low"], r_["acc20_CI_high"] = np.percentile(dr, [2.5, 97.5])
    if nm != REF_T:
        dd = dr - _tri_draws(REF_T, 0.2); r_["d20_vs_rule_triage"] = float(TRIC.loc[0.2, nm] - TRIC.loc[0.2, REF_T])
        r_["d20_CI_low"], r_["d20_CI_high"] = np.percentile(dd, [2.5, 97.5]); r_["d20_p"] = min(1.0, 2 * min((dd <= 0).mean(), (dd >= 0).mean()) + 1 / len(dd))
    tri_rows.append(r_)
TRIAGE = pd.DataFrame(tri_rows).round(4); savetab(TRIAGE, "tab_selective_testing")
# (v) cost and turnaround (fee schedules in LAB_PRICES, §1); field-sensor readings are taken on site and not priced
cost_rows = []
pairs_ = {c: [LABS[j] for j in MAINSTORE[c]["FIXED"][2]] for c in PRIMARY}
for lab_name, pr in LAB_PRICES.items():
    full = sum(pr[t] for t in LABS); meas = np.mean([sum(pr[t] for t in pairs_[c]) for c in PRIMARY]); skip = full - meas
    for policy, q, acc_ in ([("all four laboratory tests", 1.0, 1.0), ("two tests, proposed model, no follow-up", 0.0, float(TRIC.loc[0.0, "Proposed: least confident first"])),
                             ("two tests, formula rule, no follow-up", 0.0, float(TRIC.loc[0.0, REF_T]))]
                            + [(f"{nm} — {int(100 * tgt)} % accuracy", float(TRIAGE.set_index("method").loc[nm, f"share_for_{int(100 * tgt)}pct"]), tgt)
                               for nm in ("Proposed: least confident first", REF_T) for tgt in (0.98, 0.99)]):
        cost = meas + q * skip if policy != "all four laboratory tests" else full
        cost_rows.append(dict(price_list=lab_name, policy=policy, share_full_panel=q, accuracy=acc_, cost_per_sample=cost,
                              saving_vs_full_pct=100 * (1 - cost / full), same_day_result_pct=100 * (1 - q)))
COST = pd.DataFrame(cost_rows).round(4); savetab(COST, "tab_cost_turnaround")
# figure: accuracy vs share of samples sent to the full panel; accuracy vs number of laboratory tests
fig, ax = plt.subplots(1, 2, figsize=(16, 5.2)); pl0 = list(LAB_PRICES.values())[0]
meas0 = np.mean([sum(pl0[t] for t in pairs_[c]) for c in PRIMARY]); skip0 = sum(pl0[t] for t in LABS) - meas0
for nm, sty in zip(TRN, ["o-", "s--", "d-.", "x--", "k:"]):
    ax[0].plot(100 * TRIC.index, 100 * TRIC[nm], sty, markevery=10, lw=1.4, label=nm)
ax[0].set_xlabel("samples receiving the two remaining laboratory tests (%)"); ax[0].set_ylabel("accuracy (%)"); ax[0].set_ylim(85, 100.5)
ax[0].set_title("(a) Selective full-panel testing (tier A, mean over test countries)"); ax[0].legend(fontsize=7, loc="lower right")
sec = ax[0].secondary_xaxis("top", functions=(lambda x: meas0 + x / 100 * skip0, lambda y: 100 * (y - meas0) / skip0))
sec.set_xlabel(f"laboratory cost per sample (US$; {list(LAB_PRICES)[0].split(',')[0]})")
for col, lab_, sty in [("rule_Accuracy", "Formula rule (last value)", "x--"), ("bb_Accuracy", "Black box", "s--"), ("prop_Accuracy", "Proposed", "o-")]:
    ax[1].plot(FEWER.k, 100 * FEWER[col], sty, label=lab_)
ax[1].set_xticks(FEWER.k); ax[1].set_xticklabels([f"{k}\n({t})" for k, t in zip(FEWER.k, FEWER.tests)])
ax[1].set_xlabel("laboratory tests measured at t"); ax[1].set_ylabel("accuracy (%)"); ax[1].set_title("(b) Accuracy by number of laboratory tests (tier A)"); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig("fig13_practical_value"); plt.show()
print("practical-value analysis done")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    f = FEWER; tl = lambda k, s: "no laboratory test" if k == 0 else f"{k} test{'s' if k > 1 else ''}: " + " + ".join(PNAME[x] for x in s.split("+"))
    idx = [tl(k, s) for k, s in zip(f.k, f.tests)]; W = [("rule", "formula rule"), ("bb", "black box"), ("prop", "proposed")]
    t = pd.DataFrame({**{(lab, w): fmt_list(f[f"{tag}_{met}"], fn) for met, lab, fn in [("Accuracy", "Accuracy (%)", f_pct), ("MacroF1", "Macro-F1", f_num), ("MCC", "MCC", f_num)]
                         for tag, w in W},
                      ("Δ accuracy, proposed − rule (pp)", "mean"): [f_pct(v, 2, True) for v in f.dAcc],
                      ("Δ accuracy, proposed − rule (pp)", "95% CI"): [f"[{f_pct(a, 2, True)}, {f_pct(b, 2, True)}]" for a, b in zip(f.dAcc_lo, f.dAcc_hi)],
                      ("Δ accuracy, proposed − rule (pp)", "p (Holm)"): [f_p(p, 1 / N_BOOT, 3) for p in f.dAcc_p_holm]}, index=idx)
    B = np.zeros(t.shape, bool); cols = list(t.columns)
    for met, lab in [("Accuracy", "Accuracy (%)"), ("MacroF1", "Macro-F1"), ("MCC", "MCC")]:
        vals = np.c_[[f[f"{tag}_{met}"].to_numpy(float) for tag, _ in W]].T; best = vals.max(1, keepdims=True)
        for j, (_, w) in enumerate(W): B[:, cols.index((lab, w))] = np.isclose(vals[:, j], best[:, 0], atol=1e-12)
    B[:, cols.index(("Δ accuracy, proposed − rule (pp)", "p (Holm)"))] = f.dAcc_p_holm.to_numpy() < 0.05
    show_table("fewer_tests", "Accuracy when fewer laboratory tests are run (tier A, same fixed test sets for every method)", t, stub="Laboratory tests at t",
               bold=B, note="Unmeasured tests enter the formula rule at their last genuine value. Test sets chosen on the validation country. "
                            "Δ: site-bootstrap 95% CI; p Holm-adjusted over the rows. Bold = best of the three methods for each metric; adjusted p < 0.05.")
    g = GAPS; idx = [f"{a} ({f_int(n)} samples, {k} countries)" for a, n, k in zip(g.group, g.n, g.countries)]
    t = pd.DataFrame({**{(lab, w): fmt_list(g[f"{tag}_{met}"], fn) for met, lab, fn in [("Accuracy", "Accuracy (%)", f_pct), ("MacroF1", "Macro-F1", f_num)]
                         for tag, w in W},
                      ("Δ accuracy, proposed − rule (pp)", "mean"): [f_pct(v, 2, True) for v in g.dAcc],
                      ("Δ accuracy, proposed − rule (pp)", "95% CI"): [f"[{f_pct(a, 2, True)}, {f_pct(b, 2, True)}]" for a, b in zip(g.dAcc_lo, g.dAcc_hi)],
                      ("Δ accuracy, proposed − rule (pp)", "p (Holm)"): [f_p(p, 1 / N_BOOT, 3) for p in g.dAcc_p_holm]}, index=idx)
    B = np.zeros(t.shape, bool); cols = list(t.columns)
    for met, lab in [("Accuracy", "Accuracy (%)"), ("MacroF1", "Macro-F1")]:
        vals = np.c_[[g[f"{tag}_{met}"].to_numpy(float) for tag, _ in W]].T; best = vals.max(1, keepdims=True)
        for j, (_, w) in enumerate(W): B[:, cols.index((lab, w))] = np.isclose(vals[:, j], best[:, 0], atol=1e-12)
    B[:, cols.index(("Δ accuracy, proposed − rule (pp)", "p (Holm)"))] = g.dAcc_p_holm.to_numpy() < 0.05
    show_table("gap_groups", "Accuracy by age and availability of the last laboratory value (tier A, 2 laboratory tests)", t, stub="Samples", bold=B,
               note=f"Age = days since the last genuinely measured value of the unmeasured tests (the older of the two). 'Not available' = no "
                    f"genuine value of an unmeasured test within {LAST_MAX_AGE_DAYS} days, so the rule must assume it passes. Countries with fewer "
                    "than 10 samples in a group are omitted; p Holm-adjusted over the groups. Bold = best of the three methods; adjusted p < 0.05.")
    h = HDET_S; base = HDET.groupby("test_country")[["positives", "n"]].mean()
    t = pd.DataFrame({"ROC-AUC": fmt_list(h.ROC_AUC), "PR-AUC": fmt_list(h.PR_AUC),
                      "Failures found when flagging 10 % of samples (%)": fmt_list(h.recall_at_10pct_flagged, lambda v: f_pct(v, 1)),
                      "Failures found when flagging 20 % of samples (%)": fmt_list(h.recall_at_20pct_flagged, lambda v: f_pct(v, 1))}, index=list(h.index))
    order = [m for m in HDN if m in t.index]; t = t.loc[order]
    show_table("hidden_detect", "Flagging hidden failures: which samples have a failing unmeasured test? (tier A, 2 laboratory tests)", t, stub="Score",
               bold=best_mask(pd.DataFrame({c: h.loc[order, s].to_numpy(float) for c, s in zip(t.columns, ["ROC_AUC", "PR_AUC", "recall_at_10pct_flagged", "recall_at_20pct_flagged"])}, index=t.index)),
               note=f"Countries with at least 10 hidden failures: {', '.join(HDET_COUNTRIES)} (share of samples with a hidden failure: "
                    + ", ".join(f"{c} {100 * base.loc[c, 'positives'] / base.loc[c, 'n']:.1f} %" for c in HDET_COUNTRIES)
                    + "). Mean over countries and repeats. Ceiling = class if every unmeasured test passed. Bold = best in each column.")
    tr = TRIAGE.set_index("method"); order = list(TRN)
    t = pd.DataFrame({**{("Accuracy (%) with this share sent to the full panel", f"{q} %"): fmt_list(tr.loc[order, f"acc_at_{q}pct"], f_pct) for q in (0, 10, 20, 30, 50)},
                      **{("Share sent to the full panel to reach", f"{p} %"): [DASH if _isnan(v) else f"{100 * v:.0f} %" for v in tr.loc[order, f"share_for_{p}pct"]] for p in (97, 98, 99)},
                      ("Δ accuracy at 20 % vs rule-based selection (pp)", "mean"): [DASH if _isnan(v) else f_pct(v, 2, True) for v in tr.loc[order].get("d20_vs_rule_triage", pd.Series(np.nan, order))],
                      ("Δ accuracy at 20 % vs rule-based selection (pp)", "95% CI"): [DASH if _isnan(a) else f"[{f_pct(a, 2, True)}, {f_pct(b, 2, True)}]"
                                                                                     for a, b in zip(tr.loc[order].get("d20_CI_low", pd.Series(np.nan, order)), tr.loc[order].get("d20_CI_high", pd.Series(np.nan, order)))],
                      ("Δ accuracy at 20 % vs rule-based selection (pp)", "p"): [f_p(p, 1 / N_BOOT, 3) for p in tr.loc[order].get("d20_p", pd.Series(np.nan, order))]},
                     index=order)
    num = pd.DataFrame(np.nan, index=order, columns=t.columns)
    for q in (0, 10, 20, 30, 50): num[("Accuracy (%) with this share sent to the full panel", f"{q} %")] = tr.loc[order, f"acc_at_{q}pct"].to_numpy(float)
    for p in (97, 98, 99): num[("Share sent to the full panel to reach", f"{p} %")] = tr.loc[order, f"share_for_{p}pct"].to_numpy(float)
    show_table("triage", "Selective full-panel testing: accuracy vs share of samples given the two remaining tests (tier A)", t, stub="Selection of samples",
               bold=best_mask(num, higher={("Share sent to the full panel to reach", f"{p} %"): False for p in (97, 98, 99)}),
               note="All samples get the two fixed tests; the selected share also gets the other two and is then classified exactly. Rule-based "
                    "selection ranks samples by whether the last values of the unmeasured tests fail, then by their historical failure rate. "
                    "Mean over test countries and repeats; Δ: site-bootstrap 95% CI and p. Bold = best in each column.")
    cs = COST.copy(); idx = pd.MultiIndex.from_tuples([(f"Prices: {pl}", po) for pl, po in zip(cs.price_list, cs.policy)])
    t = pd.DataFrame({"Samples given all four tests (%)": fmt_list(cs.share_full_panel, lambda v: f_pct(v, 0)), "Accuracy (%)": fmt_list(cs.accuracy, f_pct),
                      "Laboratory cost per sample (US$)": fmt_list(cs.cost_per_sample, lambda v: f_num(v, 2)),
                      "Saving vs all four tests (%)": fmt_list(cs.saving_vs_full_pct, lambda v: f_num(v, 1)),
                      "Results without the 5-day BOD wait (%)": fmt_list(cs.same_day_result_pct, lambda v: f_num(v, 0))}, index=idx)
    show_table("cost", "Laboratory cost and turnaround per sample under each testing policy", t, stub="Policy",
               note="Fees per analysis from published schedules: Oklahoma DEQ State Environmental Laboratory (FY2027: BOD 66, ammonia-N 40, "
                    "orthophosphate 25, total N = TKN 42 + nitrate/nitrite 38 + calculation 10 US$) and Montana DPHHS Environmental Laboratory "
                    "(2024: BOD 60, ammonia 23, ortho-P 23, total persulfate N 37 US$). Field-sensor readings are not priced. BOD requires a 5-day "
                    "incubation (Standard Methods 5210 B). Accuracy = agreement with the class from all eight values. If 'N' is total oxidised "
                    "nitrogen rather than total nitrogen (its median is close to that of nitrate in England and Ireland, " + tref("units")
                    + "), its fee is that of the nitrate/nitrite analysis, not TKN + nitrate/nitrite.")
    h0 = HEAD; tr0 = TRIAGE.set_index("method"); pf_ = "Proposed – 2 tests fixed"
    q98 = tr0.loc["Proposed: least confident first", "share_for_98pct"]
    rows = [("Xu et al. (2024), Heliyon", "coastal cities, China; weighted WQI grade", "total P + ammonia-N (2 of 9)", "accuracy > 85 % (random forest)", "random split, one region"),
            ("García-Ceja & González-Mendoza (2026), LNCS 16221", "Nuevo León, Mexico (1,302 samples); multiplicative WQI", "5-test panel (hardness, PO₄, pH, NH₃, SST)",
             "accuracy 85.7 % (92.1 % with 11 indicators)", "cross-validation, one region; cheap panel then full confirmation"),
            ("Bordjihene et al. (2026), Front. Earth Sci.", "one dam, Algeria; weighted WQI value", "NO₂⁻ + PO₄³⁻ + NH₄⁺ (3 of 10)", "R² = 0.987 (XGBoost)", "random 70/30 split, one site"),
            ("Rahman et al. (2026), Front. Water", "Canadian subset of this dataset; published label", "all 8 parameters", "accuracy 97.33 % (10-shot)",
             "random split; label reproduced exactly by the published formula (" + tref("prior_facts") + ")"),
            ("This study", "5 countries; corrected single-sample CCME-type label", "4 field sensors + 2 laboratory tests",
             f"accuracy {f_pct(h0.loc[pf_, 'Accuracy'])} % on unseen countries (formula rule {f_pct(h0.loc['Formula rule (last value) – 2 tests', 'Accuracy'])} %); "
             + (f"98 % accuracy when {100 * q98:.0f} % of samples get the full panel" if not _isnan(q98) else ""),
             "leave-one-country-out, genuinely measured samples, site bootstrap")]
    t = pd.DataFrame({"Setting": [r[1] for r in rows], "Laboratory parameters used": [r[2] for r in rows], "Reported result": [r[3] for r in rows],
                      "Evaluation": [r[4] for r in rows]}, index=[r[0] for r in rows])
    show_table("positioning", "Reduced-panel water-quality classification: this study and published approaches", t, stub="Study", left=list(t.columns),
               note="Values as reported by each study; not directly comparable (different indices, data and evaluation protocols).")
_present()

## 13. Confusion matrices and explainability (XAI)
The XAI analysis has five parts: **(13a)** class-wise TreeSHAP of the proposed model and the black box; **(13b)** stability of the
explanations across test countries; **(13c)** exact Shapley values at the level of the *measurements* (through the knowledge-guided
feature map) and a **domain-consistency test** against the CCME objectives; **(13d)** LIME and SHAP–LIME agreement; **(13e)** SHAP-space
embedding. All on tier A samples, repeat-0 models.

In [ ]:
cs_ = list(BT)
fig, axes = plt.subplots(2, len(cs_), figsize=(4.4 * len(cs_), 8)); axes = np.array(axes).reshape(2, -1)
for j, c in enumerate(cs_):
    e = MAINSTORE[c]["reps"][0]; s_ = TIER_A[e["ev"]]
    for i, nm in enumerate(["Black box – 2 tests", "Proposed – 2 tests fixed"]):
        cmx = confusion_matrix(y_all[e["ev"]][s_], e["P"][nm][s_].argmax(1), labels=range(K), normalize="true")
        sns.heatmap(cmx, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=axes[i, j], xticklabels=[x[:4] for x in CLASSES], yticklabels=[x[:4] for x in CLASSES])
        axes[i, j].set_title(f"{nm.split(' –')[0]} — {c}", fontsize=10); axes[i, j].set_xlabel("predicted"); axes[i, j].set_ylabel("actual")
plt.tight_layout(); savefig("fig06_confusion"); plt.show()
xc = XAI_COUNTRY if XAI_COUNTRY in BT else cs_[0]; e = MAINSTORE[xc]["reps"][0]; rows = e["ev"][TIER_A[e["ev"]]]
sub = np.random.default_rng(0).choice(len(rows), min(800, len(rows)), replace=False)
Xs = pd.DataFrame(featurize(rows[sub], fixed_mask(len(sub), MAINSTORE[xc]["FIXED"][2]), True), columns=NAMES[True])
sv = shap.TreeExplainer(e["prop"]).shap_values(Xs); sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)
imp = pd.DataFrame(np.abs(sv).mean(0), index=NAMES[True], columns=[CLASSES[c] for c in e["prop"].cls_])
grp_ = {c: ("CCME consistency" if c in FORM_NAMES else "history" if c in H_COLS else "current sensors" if c in S_COLS
            else "acquired lab tests" if c in L_COLS + ["n_tests"] else "context") for c in NAMES[True]}
share = imp.sum(1).groupby(pd.Series(grp_)).sum(); share = 100 * share / share.sum()
P0 = proba(e["prop"], Xs.to_numpy()); i0 = int(np.argmax(P0.max(1))); pc = P0[i0].argmax(); ci = list(e["prop"].cls_).index(pc)
loc = pd.Series(sv[i0, :, ci], index=NAMES[True]); loc = loc.reindex(loc.abs().sort_values().index).tail(12)
fig, ax = plt.subplots(1, 3, figsize=(25, 6), gridspec_kw={"width_ratios": [1.2, .6, 1]})
imp.loc[imp.sum(1).sort_values().index].tail(18).plot(kind="barh", stacked=True, colormap="RdYlGn", ax=ax[0]); ax[0].set_title(f"(a) Global SHAP by class ({xc}, tier A)")
share.sort_values().plot(kind="barh", ax=ax[1], color="#8172b2"); ax[1].set_xlabel("% of total |SHAP|"); ax[1].set_title("(b) Attribution by feature group")
ax[2].barh(loc.index, loc.values, color=["#c44e52" if v > 0 else "#4c72b0" for v in loc.values]); ax[2].set_title(f"(c) Local SHAP toward '{CLASSES[pc]}'")
plt.tight_layout(); savefig("fig07_shap"); plt.show()
# 13a (cont.): the same analysis for the black box (identical samples and acquisition pattern)
Xb_ = pd.DataFrame(featurize(rows[sub], fixed_mask(len(sub), MAINSTORE[xc]["FIXED"][2]), False), columns=NAMES[False])
svb = shap.TreeExplainer(e["bb"]).shap_values(Xb_); svb = np.stack(svb, -1) if isinstance(svb, list) else np.asarray(svb)
impb = pd.Series(np.abs(svb).mean(0).sum(1), index=NAMES[False])
shareb = impb.groupby(pd.Series({c: grp_[c] for c in NAMES[False]})).sum(); shareb = 100 * shareb / shareb.sum()
GS = pd.DataFrame({"proposed_%": share, "black_box_%": shareb}).fillna(0).round(1).sort_values("proposed_%", ascending=False)
TOPF = pd.DataFrame({"proposed": imp.sum(1).sort_values(ascending=False).index[:10], "black box": impb.sort_values(ascending=False).index[:10]})
savetab(GS.reset_index().rename(columns={"index": "group"}), "tab_xai_group_share")
savetab(TOPF, "tab_xai_top_features")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    g = GS.round(1)
    t = pd.DataFrame({("Share of total |SHAP| (%)", "proposed model"): fmt_list(g["proposed_%"], lambda v: f_num(v, 1)),
                      ("Share of total |SHAP| (%)", "black box"): [DASH if (k == "CCME consistency" and float(v) == 0) else f_num(v, 1)
                                                                   for k, v in g["black_box_%"].items()]},
                     index=[GROUP_LABEL.get(k, k) for k in g.index])
    show_table("shap_groups", f"What the models rely on: attribution by feature group ({xc}, tier A)", t, stub="Feature group",
               note="TreeSHAP, mean |SHAP| summed over the five classes, up to 800 tier-A samples with the fixed 2-test set. "
                    "– = the black box has no CCME-consistency features.")
    grp = lambda f: GROUP_LABEL.get(grp_.get(f, ""), "")
    t = pd.DataFrame({("Proposed model", "feature"): [feat_label(f) for f in TOPF["proposed"]], ("Proposed model", "group"): [grp(f) for f in TOPF["proposed"]],
                      ("Black box", "feature"): [feat_label(f) for f in TOPF["black box"]], ("Black box", "group"): [grp(f) for f in TOPF["black box"]]},
                     index=list(range(1, len(TOPF) + 1)))
    show_table("topfeat", f"Ten most important features of each model ({xc}, tier A)", t, stub="Rank", left=list(t.columns),
               note="Ranked by mean |SHAP| summed over classes; feature code in brackets. t = current sample, t−1 / t−2 = previous visits.")
_present()

### 13b. Stability of explanations across test countries
Each country's own repeat-0 model is explained on its own tier A evaluation samples. Agreement of the feature-importance rankings
(Spearman ρ of mean |SHAP|) and overlap of the top-10 features show whether the model relies on the same evidence everywhere.

In [ ]:
IMPC, GSC = {}, {}
for c in BT:
    e_ = MAINSTORE[c]["reps"][0]; r_ = e_["ev"][TIER_A[e_["ev"]]]; r_ = r_[np.random.default_rng(1).choice(len(r_), min(600, len(r_)), replace=False)]
    X_ = pd.DataFrame(featurize(r_, fixed_mask(len(r_), MAINSTORE[c]["FIXED"][2]), True), columns=NAMES[True])
    v_ = shap.TreeExplainer(e_["prop"]).shap_values(X_); v_ = np.stack(v_, -1) if isinstance(v_, list) else np.asarray(v_)
    IMPC[c] = pd.Series(np.abs(v_).mean(0).sum(1), index=NAMES[True]); g_ = IMPC[c].groupby(pd.Series(grp_)).sum(); GSC[c] = 100 * g_ / g_.sum()
IMPC = pd.DataFrame(IMPC); RHO = IMPC.corr(method="spearman").round(3)
top = {c: set(IMPC[c].sort_values(ascending=False).index[:10]) for c in IMPC}
OVL = pd.DataFrame([[len(top[a] & top[b]) for b in IMPC] for a in IMPC], index=IMPC.columns, columns=IMPC.columns)
savetab(RHO.reset_index(), "tab_xai_country_rank_corr")
savetab(OVL.reset_index(), "tab_xai_top10_overlap")
GSC = pd.DataFrame(GSC).round(1); savetab(GSC.reset_index().rename(columns={"index": "group"}), "tab_xai_group_share_by_country")
print("features in the top 10 of every country:", sorted(set.intersection(*top.values())))

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    cs = list(RHO.columns); pairs = [(a, b) for i, a in enumerate(cs) for b in cs[i + 1:]]
    t = pd.DataFrame({"Spearman ρ of importance rankings": [f"{float(RHO.loc[a, b]):.3f}" for a, b in pairs],
                      "Shared top-10 features": [f"{int(OVL.loc[a, b])} of 10" for a, b in pairs]}, index=[f"{a} – {b}" for a, b in pairs])
    show_table("xai_countries", "Stability of explanations across test countries (proposed model)", t, stub="Country pair",
               note="Feature importance (mean |SHAP|) computed separately in each test country on up to 600 tier-A samples.")
    order = [k for k in GS.index if k in GSC.index] + [k for k in GSC.index if k not in GS.index]
    g = GSC.loc[order]; cs = countries_in(g.columns)
    t = pd.DataFrame({("Share of total |SHAP| (%)", c): fmt_list(g[c], lambda v: f_num(v, 1)) for c in cs}, index=[GROUP_LABEL.get(k, k) for k in order])
    show_table("xai_groups_country", "Attribution by feature group in each test country (proposed model)", t, stub="Feature group",
               note="Same computation as " + tref("shap_groups") + ", repeated for every test country.")
_present()

### 13c. Measurement-level explanations and a domain-consistency test
Tree-SHAP explains engineered features, many of which are functions of the same measurement. Here the explained quantity is the
model's **expected class index** E[y] = Σ k·P(k) (0 = Poor … 4 = Excellent), and the *players* are what a monitoring officer actually
supplies at time *t*: the four sensor values, the two acquired laboratory values and the site history (all history features
jointly). Exact interventional Shapley values are computed over all 2⁷ coalitions; players outside a coalition take the values of
background samples (other tier A samples of the same country), and every engineered feature — including the CCME-consistency
features — is recomputed from the resulting measurements. Efficiency (Σφ = f(x) − E f) is verified.
**Domain-consistency test.** A measurement that violates its CCME objective should push the prediction towards a worse class
more than one that meets it. Consistency = P(φ of a violating sample < φ of a meeting sample) (0.5 = no relation, 1 = fully
consistent), reported for the proposed model and the black box. Samples violating each objective are over-sampled (up to 25 each);
importance (mean |φ|) uses the random sample only.

In [ ]:
# predict_arrays() is defined in §8 (it is also used by the adaptive selection)
SIDX = [BASE_COLS.index(f"{f}_t") for f in SENS]; HIDX_ALL = [BASE_COLS.index(c) for c in H_COLS]
def build_arrays(R, Bg, take, F2):
    # R: explained rows, Bg: background rows, take: (N, 7) True = player value from R, False = from Bg
    xb, sen, lab, lag = XBASE[R].copy(), SENCUR[R].copy(), LABCUR[R].copy(), LABLAG[R].copy()
    for p_, (sc, fj) in enumerate(zip(SIDX, range(4))):
        o = ~take[:, p_]; xb[o, sc] = XBASE[Bg[o], sc]; sen[o, fj] = SENCUR[Bg[o], fj]
    for q, j in enumerate(F2):
        o = ~take[:, 4 + q]; lab[o, j] = LABCUR[Bg[o], j]
    o = ~take[:, 6]; xb[np.ix_(o, HIDX_ALL)] = XBASE[np.ix_(Bg[o], HIDX_ALL)]; lag[o] = LABLAG[Bg[o]]
    return xb, lab, lag, sen
def measurement_shapley(model, rows, bg, F2, formula):
    n_p = 7; coal = ((np.arange(2 ** n_p)[:, None] >> np.arange(n_p)) & 1).astype(bool); nb = len(bg); Mf = fixed_mask(1, F2)[0]
    V = np.zeros((len(rows), 2 ** n_p))
    for ch in np.array_split(np.arange(len(rows)), max(1, len(rows) // 8)):
        r = rows[ch]; R = np.repeat(r, 2 ** n_p * nb); Bg = np.tile(bg, len(r) * 2 ** n_p); C = np.tile(np.repeat(np.arange(2 ** n_p), nb), len(r))
        xb, lab, lag, sen = build_arrays(R, Bg, coal[C], F2)
        f = predict_arrays(model, xb, lab, lag, sen, np.repeat(Mf[None], len(R), 0), formula) @ np.arange(K)
        V[ch] = f.reshape(len(r), 2 ** n_p, nb).mean(2)
    from math import factorial
    phi = np.zeros((len(rows), n_p)); size = coal.sum(1)
    for j in range(n_p):
        S = np.where(~coal[:, j])[0]; w = np.array([factorial(size[t]) * factorial(n_p - size[t] - 1) / factorial(n_p) for t in S])
        phi[:, j] = ((V[:, S | (1 << j)] - V[:, S]) * w).sum(1)
    assert np.allclose(phi.sum(1), V[:, -1] - V[:, 0], atol=1e-8), "efficiency violated"
    return phi, V[:, -1], V[:, 0]
c_x = xc; e_x = MAINSTORE[c_x]["reps"][0]; F2x = list(MAINSTORE[c_x]["FIXED"][2]); rA = e_x["ev"][TIER_A[e_x["ev"]]]
PLAYERS = SENS + [LABS[j] for j in F2x] + ["site history"]
def violates(p_, x_):
    lo, hi = GUIDE[p_]; bad = np.zeros(len(x_), bool)
    if lo is not None: bad |= x_ < lo
    if hi is not None: bad |= x_ > hi
    return bad
# explained samples: a random sample (for importance) + up to 25 samples violating each objective (for the consistency test)
rng_x = np.random.default_rng(3); xr_rand = rng_x.choice(rA, min(max(XAI_N - 100, XAI_N // 2), len(rA)), replace=False)
VALA = np.c_[SENCUR[rA], LABCUR[rA][:, F2x]]
xr_viol = [rng_x.choice(rA[violates(p_, VALA[:, j])], min(25, int(violates(p_, VALA[:, j]).sum())), replace=False) for j, p_ in enumerate(PLAYERS[:6])]
xr = np.unique(np.concatenate([xr_rand] + xr_viol)); IS_RAND = np.isin(xr, xr_rand)
bgr = rng_x.choice(np.setdiff1d(rA, xr), min(XAI_BG, len(np.setdiff1d(rA, xr))), replace=False)
t_x = time.time(); MSH = {}
for nm_, mdl, fm in [("Proposed", e_x["prop"], True), ("Black box", e_x["bb"], False)]:
    phi, fx, f0 = measurement_shapley(mdl, xr, bgr, F2x, fm); MSH[nm_] = pd.DataFrame(phi, columns=PLAYERS)
print(f"exact measurement-level Shapley values: {len(xr)} samples x 2^7 coalitions x {len(bgr)} background samples, both models, "
      f"{time.time()-t_x:.0f}s; efficiency verified")
VAL = pd.DataFrame(np.c_[SENCUR[xr], LABCUR[xr][:, F2x]], columns=PLAYERS[:6])
imp_m = pd.DataFrame({k: v[IS_RAND].abs().mean() for k, v in MSH.items()}).round(4)   # random samples only
savetab(imp_m.reset_index().rename(columns={"index": "player"}), "tab_xai_measurement_importance")
cons = []
for p_ in PLAYERS[:6]:
    lo, hi = GUIDE[p_]; bad = violates(p_, VAL[p_].to_numpy())
    row = dict(measurement=p_, objective=(f">= {lo}" if hi is None else f"<= {hi}" if lo is None else f"{lo}-{hi}"), n_violating=int(bad.sum()), n_meeting=int((~bad).sum()))
    for nm_, ph in MSH.items():
        v_ = ph[p_].to_numpy(); ok_ = bad.any() and (~bad).any()
        row[f"{nm_}: mean φ violating"] = v_[bad].mean() if bad.any() else np.nan
        row[f"{nm_}: mean φ meeting"] = v_[~bad].mean() if (~bad).any() else np.nan
        # directional consistency: probability that a violating sample receives a LOWER attribution than a meeting sample (1 = fully consistent)
        row[f"{nm_}: consistency"] = roc_auc_score(bad, -v_) if ok_ else np.nan
    cons.append(row)
CONS = pd.DataFrame(cons).round(3)
savetab(CONS, "tab_xai_domain_consistency")
fig, ax = plt.subplots(1, 2, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1, 1]})
for a, (nm_, ph) in zip(ax, MSH.items()):
    for i_, p_ in enumerate(PLAYERS):
        v_ = ph[p_].to_numpy(); jit = np.random.default_rng(i_).normal(0, .08, len(v_))
        col = VAL[p_].rank(pct=True).to_numpy() if p_ in VAL else np.full(len(v_), .5)
        sc_ = a.scatter(v_, i_ + jit, c=col, cmap="coolwarm", s=7, vmin=0, vmax=1)
    a.axvline(0, c="k", lw=.7); a.set_yticks(range(len(PLAYERS))); a.set_yticklabels(PLAYERS); a.set_xlabel("φ (effect on expected class index)")
    a.set_title(f"({'ab'[list(MSH).index(nm_)]}) {nm_}: measurement-level Shapley values ({c_x})")
plt.colorbar(sc_, ax=ax, label="measurement value (percentile)"); savefig("fig08_measurement_shap"); plt.show()
dep = [p_ for p_ in ["DO", "pH"] + [LABS[j] for j in F2x]]
fig, ax = plt.subplots(1, len(dep), figsize=(5.2 * len(dep), 4.2))
for a, p_ in zip(ax, dep):
    for nm_, col in [("Black box", "#999999"), ("Proposed", "#c44e52")]:
        a.scatter(VAL[p_], MSH[nm_][p_], s=8, alpha=.6, c=col, label=nm_)
    lo, hi = GUIDE[p_]
    for t_ in [v for v in (lo, hi) if v is not None]: a.axvline(t_, c="k", ls="--", lw=1)
    if p_ in ("PO4", "NH3", "BOD", "N"): a.set_xscale("log")
    a.axhline(0, c="k", lw=.5); a.set_xlabel(f"{p_} (dashed = CCME objective)"); a.set_ylabel("φ"); a.legend(fontsize=7)
plt.suptitle("Dependence of the measurement-level Shapley value on the measured value"); plt.tight_layout(); savefig("fig09_dependence"); plt.show()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    im = imp_m.round(4)
    rows = ([("Field sensors (at t)", p) for p in SENS if p in im.index] + [("Laboratory tests (measured at t)", p) for p in LABS if p in im.index]
            + [("Site history", p) for p in im.index if p not in SENS + LABS])
    lab = lambda p: PNAME.get(p, "all history features (one player)" if p == "site history" else p)
    t = pd.DataFrame({("Mean |φ| (class-index units)", "proposed model"): [f_num(im.loc[p, "Proposed"]) for _, p in rows],
                      ("Mean |φ| (class-index units)", "black box"): [f_num(im.loc[p, "Black box"]) for _, p in rows]},
                     index=pd.MultiIndex.from_tuples([(g, lab(p)) for g, p in rows]))
    show_table("meas_shap", f"Importance of each measurement: exact measurement-level Shapley values ({xc}, tier A)", t, stub="Measurement",
               note="The seven measurements are the players (2⁷ coalitions, background = tier-A samples of the same country); mean |φ| "
                    "over randomly drawn samples. φ is the effect on the expected class index.")
    d = CONS.round(3)
    unit = {"DO": " mg/L", "T": " °C", "NO3": " mg/L", "NH3": " mg/L", "PO4": " mg/L", "BOD": " mg/L", "N": " mg/L"}
    def obj(m, s):
        s = str(s).replace(">=", "≥").replace("<=", "≤")
        if s[:1].isdigit(): s = s.replace("-", "–")
        return s + unit.get(m, "")
    MODELS = [("Proposed", "Proposed model"), ("Black box", "Black box")]
    t = pd.DataFrame({("CCME objective", ""): [obj(m, s) for m, s in zip(d.measurement, d.objective)],
                      ("Samples", "violating"): fmt_list(d.n_violating, f_int), ("Samples", "meeting"): fmt_list(d.n_meeting, f_int),
                      **{(lab, k): fmt_list(d[f"{src}: {col}"], lambda v: f_num(v, 3)) for src, lab in MODELS
                         for k, col in [("mean φ, violating", "mean φ violating"), ("mean φ, meeting", "mean φ meeting"), ("consistency", "consistency")]}},
                     index=[PNAME.get(m, m) for m in d.measurement])
    num = pd.DataFrame({(lab, "consistency"): d[f"{src}: consistency"].to_numpy(float) for src, lab in MODELS}, index=t.index)
    B = np.zeros(t.shape, bool); bm = better_of(num, [tuple(num.columns)])
    for j, c in enumerate(num.columns): B[:, list(t.columns).index(c)] = bm[:, j]
    few = [PNAME.get(m, m) for m, n in zip(d.measurement, d.n_violating) if n < 5]
    show_table("domain_cons", f"Do the explanations follow the CCME objectives? Domain-consistency test ({xc}, tier A)", t, stub="Measurement", bold=B,
               note="A measurement that violates its objective should lower the predicted class: consistency = probability that a violating "
                    "sample receives a lower φ than a sample meeting the objective (0.5 = no relation, 1 = fully consistent). "
                    + (f"Not interpretable for {', '.join(few)} (fewer than 5 violating samples). " if few else "")
                    + "Bold = more consistent model.")
_present()

### 13d. LIME explanations and SHAP–LIME agreement
LIME (Ribeiro et al., 2016) fits a sparse local linear surrogate around each sample from perturbed measurements (quartile
discretisation; perturbation distribution = the same target-country tier A population that serves as Shapley background); the site
history is held at the sample's own values. Agreement
with the measurement-level Shapley values of 13c is measured on the six measurements (rank correlation of |attribution|, sign agreement
and top-1 agreement).

In [ ]:
import lime.lime_tabular
lime_bg = np.c_[SENCUR[rA], LABCUR[rA][:, F2x]]          # same reference population as the Shapley background (target-country tier A samples)
LX = lime.lime_tabular.LimeTabularExplainer(lime_bg, feature_names=PLAYERS[:6], mode="regression", discretize_continuous=True, random_state=0)
def lime_fn_for(r, mdl, fm):
    def f(Z):
        R = np.full(len(Z), r); xb, lab, lag, sen = XBASE[R].copy(), LABCUR[R].copy(), LABLAG[R].copy(), SENCUR[R].copy()
        sen[:, :] = Z[:, :4]; xb[:, SIDX] = Z[:, :4]
        for q, j in enumerate(F2x): lab[:, j] = Z[:, 4 + q]
        return predict_arrays(mdl, xb, lab, lag, sen, np.repeat(fixed_mask(1, F2x), len(Z), 0), fm) @ np.arange(K)
    return f
li_idx = np.random.default_rng(4).choice(len(xr), min(LIME_N, len(xr)), replace=False); agree = []; LW = {}
t_l = time.time()
for nm_, mdl, fm in [("Proposed", e_x["prop"], True), ("Black box", e_x["bb"], False)]:
    W_ = np.zeros((len(li_idx), 6))
    for t_, i_ in enumerate(li_idx):
        ex_ = LX.explain_instance(np.r_[SENCUR[xr[i_]], LABCUR[xr[i_]][F2x]], lime_fn_for(xr[i_], mdl, fm), num_features=6, num_samples=1000)
        for j_, w_ in ex_.as_map()[1]: W_[t_, j_] = w_          # regression mode: key 1 holds the weights (key 0 holds their negation)
    LW[nm_] = W_; ph = MSH[nm_].iloc[li_idx, :6].to_numpy()
    rho_ = [sps.spearmanr(np.abs(W_[t]), np.abs(ph[t])).statistic for t in range(len(li_idx))]
    top_ = (np.abs(W_).argmax(1) == np.abs(ph).argmax(1)).mean(); big = np.abs(ph) > 0.05
    agree.append(dict(model=nm_, explanations=len(li_idx), median_rank_corr=np.nanmedian(rho_), top1_agreement_pct=100 * top_,
                      sign_agreement_pct=100 * (np.sign(W_[big]) == np.sign(ph[big])).mean()))
AGR = pd.DataFrame(agree).round(3); savetab(AGR, "tab_xai_shap_lime_agreement")
ex_i = [li_idx[np.argmin(MSH["Proposed"].iloc[li_idx].sum(1).to_numpy())], li_idx[np.argmax(MSH["Proposed"].iloc[li_idx].sum(1).to_numpy())]]
fig, ax = plt.subplots(1, 4, figsize=(24, 4.2))
for k_, i_ in enumerate(ex_i):
    t_ = list(li_idx).index(i_); yh = int(np.round(predict_arrays(e_x["prop"], *[a[[xr[i_]]] for a in (XBASE, LABCUR, LABLAG, SENCUR)], fixed_mask(1, F2x)) @ np.arange(K))[0])
    vals = ", ".join(f"{p_}={v:.3g}" for p_, v in zip(PLAYERS[:6], np.r_[SENCUR[xr[i_]], LABCUR[xr[i_]][F2x]]))
    for a, (ttl, v_) in zip(ax[2 * k_: 2 * k_ + 2], [("measurement SHAP", MSH["Proposed"].iloc[i_].to_numpy()), ("LIME", np.r_[LW["Proposed"][t_], np.nan])]):
        a.barh(PLAYERS, np.nan_to_num(v_), color=["#c44e52" if v > 0 else "#4c72b0" for v in np.nan_to_num(v_)]); a.axvline(0, c="k", lw=.6)
        a.set_title(f"{ttl}: sample {k_+1} (true {CLASSES[y_all[xr[i_]]]}, predicted ~{CLASSES[yh]})", fontsize=9)
    ax[2 * k_].set_xlabel(vals, fontsize=7)
plt.tight_layout(); savefig("fig10_shap_vs_lime"); plt.show()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    a = AGR.round(3)
    t = pd.DataFrame({"Explanations compared": fmt_list(a.explanations, f_int), "Median rank correlation": fmt_list(a.median_rank_corr, lambda v: f_num(v, 3)),
                      "Top-1 agreement (%)": fmt_list(a.top1_agreement_pct, lambda v: f_num(v, 1)),
                      "Sign agreement (%)": fmt_list(a.sign_agreement_pct, lambda v: f_num(v, 1))},
                     index=["Proposed model" if m == "Proposed" else m for m in a.model])
    show_table("lime", f"Agreement between LIME and measurement-level SHAP explanations ({xc}, tier A)", t, stub="Model",
               note="LIME (regression mode) and exact Shapley values on the same samples and the same six measurements. Rank correlation = "
                    "Spearman correlation of the absolute attributions per sample; top-1 = same most important measurement; sign agreement "
                    "counted where |φ| > 0.05.")
_present()

### 13e. SHAP-space embedding
t-SNE of the class-wise SHAP vectors of the proposed model (13a samples) versus t-SNE of the model inputs. Silhouette scores
(by true class) quantify how cleanly the explanations organise the samples by water-quality class.

In [ ]:
yS = y_all[rows[sub]]; Zs = sv.reshape(len(sub), -1); Zx = np.nan_to_num(StandardScaler().fit_transform(Xs.to_numpy()))
emb = {nm_: TSNE(2, perplexity=30, random_state=0, init="pca").fit_transform(Z_) for nm_, Z_ in [("SHAP space", Zs), ("input space", Zx)]}
SIL = {nm_: silhouette_score(E_, yS) if len(set(yS)) > 1 else np.nan for nm_, E_ in emb.items()}
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
for a, (nm_, E_) in zip(ax, emb.items()):
    for k in range(K):
        if (yS == k).any(): a.scatter(E_[yS == k, 0], E_[yS == k, 1], s=7, label=CLASSES[k], c=plt.cm.RdYlGn(k / (K - 1)))
    a.set_title(f"t-SNE of the {nm_} ({xc}; silhouette {SIL[nm_]:.3f})"); a.legend(fontsize=7, markerscale=2)
plt.tight_layout(); savefig("fig11_shap_tsne"); plt.show()
print("silhouette by true class:", {k: round(v, 3) for k, v in SIL.items()})

## 14. Ablation (tier A, 2 laboratory tests)

In [ ]:
ABL = ["Black box – 2 tests", "Formula rule (last value) – 2 tests", "Proposed – 0 tests", "Proposed – 2 tests random", "Proposed – 2 tests fixed",
       "Proposed – 2 tests adaptive", "Ablation: without monotonicity constraint", "Ablation: without history", "Ablation: without current sensors"]
ABT = summarise(boot(ABL), ABL).join(summarise_auc(boot_auc(ABL), ABL)).round(4); savetab(ABT.reset_index(), "tab_ablation")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    pf = "Proposed – 2 tests fixed"
    G = [("Full model", [pf]),
         ("Full model without one component", ["Ablation: without monotonicity constraint", "Ablation: without history", "Ablation: without current sensors"]),
         ("Other laboratory-test settings", ["Proposed – 0 tests", "Proposed – 2 tests random", "Proposed – 2 tests adaptive"]),
         ("Reference methods", ["Black box – 2 tests", "Formula rule (last value) – 2 tests"])]
    keys = [(g, m) for g, ms in G for m in ms if m in ABT.index]; d = ABT.loc[[m for _, m in keys]]
    idx = pd.MultiIndex.from_tuples([(g, mname(m)) for g, m in keys]); ref = float(ABT.loc[pf, "MacroF1"])
    t = pd.DataFrame({("Accuracy (%)", "mean"): fmt_list(d.Accuracy, f_pct), ("Accuracy (%)", "95% CI"): fmt_list(d.Accuracy_CI, lambda s: f_ci(s, 2, 100)),
                      ("Macro-F1", "mean"): fmt_list(d.MacroF1), ("Macro-F1", "95% CI"): fmt_list(d.MacroF1_CI, f_ci),
                      ("MCC", ""): fmt_list(d.MCC), ("QWK", ""): fmt_list(d.QWK),
                      ("ROC-AUC", "mean"): fmt_list(d.ROC_AUC), ("ROC-AUC", "95% CI"): fmt_list(d.ROC_AUC_CI, f_ci),
                      ("Δ macro-F1 vs full model", ""): [DASH if (m == pf or g == "Reference methods") else f_num(round(float(v) - ref, 4), 4, True) for (g, m), v in zip(keys, d.MacroF1)]},
                     index=idx)
    num = pd.DataFrame(np.nan, index=idx, columns=t.columns)
    for c, src in [(("Accuracy (%)", "mean"), "Accuracy"), (("Macro-F1", "mean"), "MacroF1"), (("MCC", ""), "MCC"), (("QWK", ""), "QWK"), (("ROC-AUC", "mean"), "ROC_AUC")]:
        num[c] = d[src].to_numpy(float)
    show_table("ablation", "Ablation study (tier A, 2 laboratory tests)", t, stub="Variant", bold=best_mask(num),
               note="Δ macro-F1 = variant − full model (difference of the point estimates shown; negative = the variant is worse). Paired tests, and the "
                    "differences to the reference methods, are in " + tref("stats_f1") + ". "
                    "95% CI = site-clustered bootstrap. Bold = best value in each column.")
_present()

## 14b. Deep-learning baselines and few-shot adaptation to a new country
**Setting.** For each primary test country, every model below is trained on the **same** `DEEP_TRAIN_CAP` source-country rows
(a subset of the repeat-0 source sample) with the validation-chosen pair of laboratory tests, and evaluated on the **same** tier A
samples of the target country's later history as in §10.
- **Zero-shot (no target labels):** MLP, tabular transformer (feature tokens + [CLS]), the transformer with CCME-consistency features,
  XGBoost black box and the proposed model (XGBoost + CCME features + monotonicity constraint) trained on the identical rows,
  and the formula rule.
- **Few-shot (K = 1, 5, 10 labelled target samples per class)** drawn from the target country's *earlier* history (before every
  evaluation sample), `KSHOT_EPISODES` support sets per K, identical for every method:
  (i) XGBoost black box and proposed model retrained on source rows + support; (ii) **Transformer-ProtoNet** — transformer encoder
  pre-trained on the source rows, class prototypes from the target support (source prototype if a class has no support sample);
  (iii) **Transformer-ProtoNet + WGAN-GP** — our re-implementation of the framework of Rahman et al. (2026): `GAN_SYN` synthetic
  support samples per class from a class-specific WGAN-GP trained on that episode's support only (`GAN_EPOCHS` epochs; original 1000);
  (iv) Transformer-ProtoNet with CCME-consistency features. Synthetic samples are never used for evaluation. Support samples, like
  all training labels, come from rows of the training tier (§8).

**Limitation.** The deep models have one fixed architecture, `DEEP_EPOCHS` epochs and no early stopping or hyper-parameter search
(CPU budget), and the WGAN-GP uses fewer epochs than the original. Their results are untuned reference points; report them as such.

In [ ]:
import torch, torch.nn as nn
torch.set_num_threads(max(1, os.cpu_count() or 1))
class FTTransformer(nn.Module):
    # tabular transformer: one learnable token per feature (value * w_j + b_j), a [CLS] token, pre-norm encoder layers
    def __init__(self, n_feat, n_cls=K, d=TF_DIM, layers=TF_LAYERS, heads=TF_HEADS, drop=0.1):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_feat, d) * 0.1); self.b = nn.Parameter(torch.zeros(n_feat, d))
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        lay = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(lay, layers, enable_nested_tensor=False); self.norm = nn.LayerNorm(d); self.head = nn.Linear(d, n_cls)
    def embed(self, x):
        t = x[:, :, None] * self.w[None] + self.b[None]
        return self.norm(self.enc(torch.cat([self.cls.expand(len(x), -1, -1), t], 1))[:, 0])
    def forward(self, x): return self.head(self.embed(x))
class MLPNet(nn.Module):
    def __init__(self, n_feat, n_cls=K, hidden=(256, 128, 64), drop=0.3):
        super().__init__(); L, d = [], n_feat
        for h in hidden: L += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop)]; d = h
        self.body = nn.Sequential(*L); self.head = nn.Linear(d, n_cls)
    def embed(self, x): return self.body(x)
    def forward(self, x): return self.head(self.body(x))
class DeepPrep:
    # quantile -> normal scores fitted on training rows only; missing (hidden) values -> 0 (the median); clipped to [-5, 5]
    def __init__(self, X): self.qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200_000, random_state=0).fit(X)
    def __call__(self, X): return np.clip(np.nan_to_num(self.qt.transform(X), nan=0.0), -5, 5).astype(np.float32)
def fit_deep(kind, X, y_, w, seed, epochs=DEEP_EPOCHS, bs=256, lr=1e-3):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    m = FTTransformer(X.shape[1]) if kind == "transformer" else MLPNet(X.shape[1])
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    Xt, yt, wt = torch.tensor(X), torch.tensor(y_, dtype=torch.long), torch.tensor(w / w.mean(), dtype=torch.float32)
    m.train()
    for ep in range(epochs):
        for s in np.array_split(rng.permutation(len(X)), max(1, len(X) // bs)):
            s = torch.tensor(s); opt.zero_grad()
            loss = (nn.functional.cross_entropy(m(Xt[s]), yt[s], reduction="none") * wt[s]).mean(); loss.backward(); opt.step()
    return m.eval()
@torch.no_grad()
def deep_out(m, X, what="proba", bs=4096):
    out = []
    for s in range(0, len(X), bs):
        xb = torch.tensor(X[s:s + bs]); out.append((torch.softmax(m(xb), 1) if what == "proba" else m.embed(xb)).numpy())
    return np.concatenate(out).astype(float)
def wgan_gp(Xc, n_syn, seed, epochs=GAN_EPOCHS, z_dim=16, n_critic=5, lam=10.0):
    # class-specific WGAN-GP as described by Rahman et al. (2026): G 16-64-128-64-d, critic d-64-128-64-1, Adam 5e-4 (0.5, 0.9)
    torch.manual_seed(seed); d = Xc.shape[1]
    G = nn.Sequential(nn.Linear(z_dim, 64), nn.ReLU(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, d))
    D = nn.Sequential(nn.Linear(d, 64), nn.LeakyReLU(.2), nn.Linear(64, 128), nn.LeakyReLU(.2), nn.Linear(128, 64), nn.LeakyReLU(.2), nn.Linear(64, 1))
    oG = torch.optim.Adam(G.parameters(), 5e-4, betas=(.5, .9)); oD = torch.optim.Adam(D.parameters(), 5e-4, betas=(.5, .9))
    real = torch.tensor(Xc, dtype=torch.float32)
    if len(real) == 1: real = real.repeat(2, 1)                    # 1-shot: the single sample is duplicated, as in the original
    n = len(real)
    for _ in range(epochs):
        for _ in range(n_critic):
            fake = G(torch.randn(n, z_dim)).detach(); e = torch.rand(n, 1)
            xh = (e * real + (1 - e) * fake).requires_grad_(True)
            gp = ((torch.autograd.grad(D(xh).sum(), xh, create_graph=True)[0].norm(2, dim=1) - 1) ** 2).mean()
            oD.zero_grad(); (D(fake).mean() - D(real).mean() + lam * gp).backward(); oD.step()
        oG.zero_grad(); (-D(G(torch.randn(n, z_dim))).mean()).backward(); oG.step()
    with torch.no_grad(): return G(torch.randn(n_syn, z_dim)).numpy().astype(np.float32)
def protonet_P(m, Xs, ys, Xq, fallback=None):
    # prototypical network: class prototype = mean support embedding; P(class) = softmax(-squared Euclidean distance)
    Eq = deep_out(m, Xq, "embed"); Es = deep_out(m, Xs, "embed") if len(Xs) else np.zeros((0, Eq.shape[1])); protos = np.full((K, Eq.shape[1]), np.nan)
    for k in range(K):
        if (ys == k).any(): protos[k] = Es[ys == k].mean(0)
        elif fallback is not None: protos[k] = fallback[k]
    d2 = ((Eq[:, None, :] - protos[None]) ** 2).sum(2); d2 = np.where(np.isnan(d2), np.inf, d2)
    z = -d2 + d2.min(1, keepdims=True); P = np.exp(z); return P / P.sum(1, keepdims=True)
print("deep models defined | torch", torch.__version__, "| threads", torch.get_num_threads())

In [ ]:
ZS_ROWS, KS_ROWS = [], []
if RUN_DEEP:
    for c in PRIMARY:
        ck = ck_load(f"deep_{c}")
        if ck is not None: ZS_ROWS += ck["zs"]; KS_ROWS += ck["ks"]; print(f"[{c}] deep/few-shot restored", flush=True); continue
        t_d = time.time(); n_z, n_k = len(ZS_ROWS), len(KS_ROWS); T, FIXED = TUNED[c]; F2 = FIXED[2]
        e0 = MAINSTORE[c]["reps"][0]; evA = e0["ev"][TIER_A[e0["ev"]]]; yA = y_all[evA]; MA = fixed_mask(len(evA), F2)
        src_all = sources(c); src0 = np.random.default_rng(100).choice(src_all, min(TRAIN_CAP, len(src_all)), replace=False)   # repeat-0 source sample
        dsrc = np.sort(np.random.default_rng(200).choice(src0, min(DEEP_TRAIN_CAP, len(src0)), replace=False)); ysrc = y_all[dsrc]
        wsrc = compute_sample_weight("balanced", ysrc); Mtr = fixed_mask(len(dsrc), F2)
        P0 = {"Formula rule (last value)": rule_P(evA, MA, "last")}
        enc, prepK, ZqK, protos = {}, {}, {}, {}
        for fm in (False, True):
            Xs_ = featurize(dsrc, Mtr, fm); prepK[fm] = DeepPrep(Xs_); Zs_ = prepK[fm](Xs_); ZqK[fm] = prepK[fm](featurize(evA, MA, fm))
            enc[fm] = fit_deep("transformer", Zs_, ysrc, wsrc, 1)
            Es_ = deep_out(enc[fm], Zs_, "embed")
            protos[fm] = np.array([Es_[ysrc == k].mean(0) if (ysrc == k).any() else np.full(Es_.shape[1], np.nan) for k in range(K)])
            P0["Transformer + CCME features" if fm else "Transformer"] = deep_out(enc[fm], ZqK[fm])
            P0["Transformer-ProtoNet + CCME features" if fm else "Transformer-ProtoNet"] = protonet_P(enc[fm], np.zeros((0, Zs_.shape[1]), np.float32), np.zeros(0, int), ZqK[fm], protos[fm])
            if not fm: P0["MLP"] = deep_out(fit_deep("mlp", Zs_, ysrc, wsrc, 1), ZqK[fm])
        P0["Black box (XGBoost)"] = predict(train(dsrc, np.array([], int), T[False]["prm"], 1.0, 1, False), evA, MA, False)
        P0["Proposed (XGBoost + CCME + constraint)"] = predict(train(dsrc, np.array([], int), T[True]["prm"], 1.0, 1, True), evA, MA)
        for nm, P_ in P0.items(): ZS_ROWS.append(dict(test_country=c, method=nm, n_train=len(dsrc), n_eval=len(evA), **metrics9(yA, P_)))
        idx_c = np.where((ctry == c) & STATION)[0]; d_ = days[idx_c]; early = idx_c[d_ <= np.quantile(d_, .5)]; early = early[LABEL_OK[early]]
        assert days[early].max() < days[evA].min(), "support samples must precede every evaluation sample"
        for Kk in KSHOTS:
            for ep in range(KSHOT_EPISODES):
                rng = np.random.default_rng(10_000 + 100 * Kk + ep); seed = 5000 + 100 * Kk + ep
                sup = np.concatenate([rng.choice(early[y_all[early] == k], min(Kk, int((y_all[early] == k).sum())), replace=False)
                                      for k in range(K) if (y_all[early] == k).any()]); ys = y_all[sup]
                P = {"Formula rule (last value)": P0["Formula rule (last value)"]}
                P["Black box (XGBoost)"] = predict(train(dsrc, sup, T[False]["prm"], T[False]["w"], seed, False), evA, MA, False)
                P["Proposed (XGBoost + CCME + constraint)"] = predict(train(dsrc, sup, T[True]["prm"], T[True]["w"], seed, True), evA, MA)
                for fm in (False, True):
                    Zsup = prepK[fm](featurize(sup, fixed_mask(len(sup), F2), fm))
                    P["Transformer-ProtoNet + CCME features" if fm else "Transformer-ProtoNet"] = protonet_P(enc[fm], Zsup, ys, ZqK[fm], protos[fm])
                    if not fm:
                        syn = [wgan_gp(Zsup[ys == k], GAN_SYN, seed + k) for k in np.unique(ys)]
                        ysyn = np.concatenate([[k] * GAN_SYN for k in np.unique(ys)])
                        P["Transformer-ProtoNet + WGAN-GP (Rahman et al. 2026)"] = protonet_P(enc[fm], np.r_[Zsup, np.concatenate(syn)], np.r_[ys, ysyn], ZqK[fm], protos[fm])
                for nm, P_ in P.items():
                    KS_ROWS.append(dict(test_country=c, K=Kk, episode=ep, support=len(sup), method=nm, n_eval=len(evA), **metrics9(yA, P_)))
        ck_save(f"deep_{c}", dict(zs=ZS_ROWS[n_z:], ks=KS_ROWS[n_k:])); print(f"[{c}] deep baselines + few-shot done | {time.time()-t_d:.0f}s", flush=True)
    ZS, KS = pd.DataFrame(ZS_ROWS), pd.DataFrame(KS_ROWS); savetab(ZS, "tab_deep_zero_shot_all"); savetab(KS, "tab_few_shot_all")
    ZM = ["Formula rule (last value)", "MLP", "Transformer", "Transformer-ProtoNet", "Transformer + CCME features", "Transformer-ProtoNet + CCME features",
          "Black box (XGBoost)", "Proposed (XGBoost + CCME + constraint)"]
    zt = ZS.groupby("method")[["Accuracy", "MacroF1", "MCC", "QWK", "ROC_AUC", "PR_AUC", "LogLoss"]].mean().reindex(ZM).round(4)
    zt = zt.join(ZS.pivot_table(index="method", columns="test_country", values="Accuracy").add_prefix("Acc_").round(4))
    savetab(zt.reset_index(), "tab_deep_zero_shot")
    KM = ["Formula rule (last value)", "Black box (XGBoost)", "Transformer-ProtoNet", "Transformer-ProtoNet + WGAN-GP (Rahman et al. 2026)",
          "Transformer-ProtoNet + CCME features", "Proposed (XGBoost + CCME + constraint)"]
    kt = KS.groupby(["method", "K"])[["Accuracy", "MacroF1", "ROC_AUC"]].agg(["mean", "std"])
    kt = pd.DataFrame({m_: kt[(m_, "mean")].map("{:.4f}".format) + " ± " + kt[(m_, "std")].map("{:.4f}".format) for m_ in ["Accuracy", "MacroF1", "ROC_AUC"]})
    kt = kt.reindex(pd.MultiIndex.from_product([KM, KSHOTS], names=["method", "K"]))
    savetab(kt.reset_index(), "tab_few_shot")
    pv_ = KS.pivot_table(index=["test_country", "K", "episode"], columns="method", values="Accuracy")
    kd = []
    for b_ in ["Transformer-ProtoNet + WGAN-GP (Rahman et al. 2026)", "Black box (XGBoost)", "Formula rule (last value)"]:
        for Kk in KSHOTS:
            d_ = (pv_["Proposed (XGBoost + CCME + constraint)"] - pv_[b_]).xs(Kk, level="K")
            kd.append(dict(comparison=f"Proposed vs {b_}", K=Kk, mean_diff_accuracy=d_.mean(), min=d_.min(), max=d_.max(), proposed_better=f"{int((d_ > 0).sum())}/{len(d_)}"))
    KD = pd.DataFrame(kd).round(4); savetab(KD, "tab_few_shot_differences")
    fig, ax = plt.subplots(1, 2, figsize=(16, 4.8)); sty2 = ["k:", "s--", "v-.", "D-", "p-.", "o-"]
    for a, met in zip(ax, ["Accuracy", "MacroF1"]):
        g_ = KS.groupby(["method", "K"])[met].agg(["mean", "std"])
        for m_, st_ in zip(KM, sty2):
            if m_ in g_.index.get_level_values(0):
                d_ = g_.xs(m_, level="method"); z0 = ZS[ZS.method == m_][met].mean() if m_ in set(ZS.method) else np.nan
                xs_, ys_ = np.r_[0, d_.index.to_numpy()], np.r_[z0, d_["mean"].to_numpy()]
                a.errorbar(xs_, ys_, yerr=np.r_[0, d_["std"].to_numpy()], fmt=st_, capsize=3, label=m_)
        a.set_xlabel("labelled target samples per class (K; 0 = zero-shot)"); a.set_ylabel(f"{met} (tier A)"); a.set_xticks([0] + KSHOTS); a.legend(fontsize=6.5)
        a.set_title(f"({'ab'[['Accuracy', 'MacroF1'].index(met)]}) Few-shot adaptation to an unseen country — {met}")
    plt.tight_layout(); savefig("fig12_few_shot"); plt.show()

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    DEEP_NAMES = {"Formula rule (last value)": ("No learning", "Rule: unmeasured tests at last value"),
                  "MLP": ("Without CCME features", "MLP"), "Transformer": ("Without CCME features", "Transformer"),
                  "Transformer-ProtoNet": ("Without CCME features", "Transformer-ProtoNet"),
                  "Transformer-ProtoNet + WGAN-GP (Rahman et al. 2026)": ("Without CCME features", "Transformer-ProtoNet + WGAN-GP (Rahman et al., 2026)"),
                  "Black box (XGBoost)": ("Without CCME features", "Black box (XGBoost)"),
                  "Transformer + CCME features": ("With CCME-consistency features", "Transformer + CCME features"),
                  "Transformer-ProtoNet + CCME features": ("With CCME-consistency features", "Transformer-ProtoNet + CCME features"),
                  "Proposed (XGBoost + CCME + constraint)": ("With CCME-consistency features", "Proposed (XGBoost + CCME features + constraint)")}
    keys = [m for m in DEEP_NAMES if m in zt.index]; d = zt.loc[keys]; idx = pd.MultiIndex.from_tuples([DEEP_NAMES[m] for m in keys])
    cs = countries_in([c[4:] for c in zt.columns if c.startswith("Acc_")])
    spec = [(("Accuracy (%)", ""), "Accuracy", f_pct, True), (("Macro-F1", ""), "MacroF1", f_num, True), (("MCC", ""), "MCC", f_num, True),
            (("QWK", ""), "QWK", f_num, True), (("ROC-AUC", ""), "ROC_AUC", f_num, True), (("PR-AUC", ""), "PR_AUC", f_num, True),
            (("Log-loss ↓", ""), "LogLoss", f_num, False)] + [(("Accuracy by test country (%)", c), f"Acc_{c}", f_pct, True) for c in cs]
    t = pd.DataFrame({lab: fmt_list(d[src], fn) for lab, src, fn, _ in spec}, index=idx)
    num = pd.DataFrame({lab: d[src].to_numpy(float) for lab, src, _, _ in spec}, index=idx)
    show_table("zeroshot_deep", "Zero-shot transfer to an unseen country: deep-learning baselines vs proposed (tier A, 2 laboratory tests)", t, stub="Model",
               bold=best_mask(num, higher={lab: hb for lab, _, _, hb in spec}),
               note=f"No labels from the test country; every model is trained on the same {DEEP_TRAIN_CAP:,} source rows. Mean over "
                    f"{', '.join(PRIMARY)}. Bold = best value in each column.")
    keys = [m for m in DEEP_NAMES if m in kt.index.get_level_values("method")]
    MET = [("Accuracy (%)", "Accuracy", 100, 2), ("Macro-F1", "MacroF1", 1, 4), ("ROC-AUC", "ROC_AUC", 1, 4)]
    idx = pd.MultiIndex.from_tuples([(f"K = {k} labelled sample{'s' if k > 1 else ''} per class", DEEP_NAMES[m][1]) for k in KSHOTS for m in keys])
    cols = pd.MultiIndex.from_tuples([(lab, "mean ± SD") for lab, _, _, _ in MET])
    rows = [[f_pm(kt.loc[(m, k), src], sc, dd) for _, src, sc, dd in MET] for k in KSHOTS for m in keys]
    nums = [[pm_parts(kt.loc[(m, k), src])[0] for _, src, _, _ in MET] for k in KSHOTS for m in keys]
    show_table("fewshot", "Few-shot adaptation: K labelled samples per class from the new country (tier A)", pd.DataFrame(rows, index=idx, columns=cols),
               stub="Model", bold=best_mask(pd.DataFrame(nums, index=idx, columns=cols), by_group=True),
               note=f"Mean ± SD over {len(PRIMARY)} countries × {KSHOT_EPISODES} support sets; identical support sets for every method. "
                    "The formula rule uses no labels (same value for every K). Rahman et al. (2026) = our re-implementation of their "
                    "Transformer-ProtoNet + WGAN-GP framework. Bold = best mean for each K.")
    KB = ["Transformer-ProtoNet + WGAN-GP (Rahman et al. 2026)", "Black box (XGBoost)", "Formula rule (last value)"]
    comp = lambda s: next(b for b in KB if f"Proposed vs {b}".startswith(str(s).rstrip(".").rstrip()))
    d = KD.copy(); d["_b"] = [comp(s) for s in d.comparison]; d["_o"] = [KB.index(b) for b in d._b]; d = d.sort_values(["_o", "K"], kind="stable")
    t = pd.DataFrame({"Mean Δ accuracy (pp)": [f_pct(v, 2, True) for v in d.mean_diff_accuracy], "Smallest Δ (pp)": [f_pct(v, 2, True) for v in d["min"]],
                      "Largest Δ (pp)": [f_pct(v, 2, True) for v in d["max"]],
                      "Proposed better (support sets)": [str(v).replace("/", " of ") for v in d.proposed_better]},
                     index=pd.MultiIndex.from_tuples([(f"Proposed vs {DEEP_NAMES[b][1]}", f"K = {k}") for b, k in zip(d._b, d.K)]))
    show_table("fewshot_diff", "Few-shot: paired accuracy differences on identical support sets", t, stub="Comparison",
               note=f"Δ = proposed − comparator in percentage points, paired over {len(PRIMARY)} countries × {KSHOT_EPISODES} support sets.")
if RUN_DEEP: _present()

## 14c. Replication of the prior-work protocol on the same Canadian data
Rahman et al. (2026, *Frontiers in Water*, Transformer-FSL-GAN) classify the **published** CCME_WQI label of the Canadian subset of this
dataset (3,949 records) from **all eight parameters**, with a stratified random 70/30 split (70 % to pre-train the encoder, 30 % for
5-way K-shot episodes with 15 queries per class) and report 97.33 % accuracy (one 10-shot episode; 96.77 % over 100 episodes).
This section re-runs that protocol on the identical records: (a) the published CCME formula without learning; (b) plain XGBoost
trained on the 70 % base set, with a random and a station-grouped split (5 seeds); (c) our re-implementation of the
Transformer-ProtoNet ± WGAN-GP framework in their episodic protocol (`PRIOR_EPISODES` episodes per K; queries limited so that
support and query never overlap — the Poor class has only ≈ 19 samples in the 30 % pool). It also documents data issues of that
setting (gap-filled values, records that violate CCME objectives but are labelled Excellent).

In [ ]:
t_p = time.time(); pc_ = CANADA_RAW; X8 = pc_[F].to_numpy(float); y8 = pc_.y_pub.to_numpy().astype(int); site8 = pc_.site.to_numpy()
s8 = ccme_parts({f: X8[:, j] for j, f in enumerate(F)}, GUIDE_PUB, 12)[0]; yform = to_class(s8)
med8 = np.nanmedian(X8, 0); imp8 = np.isclose(X8, med8[None, :], rtol=0, atol=1e-9)
facts = pd.DataFrame([dict(records=len(pc_), stations=pc_.site.nunique(), **{f"n_{CLASSES[k]}": int((y8 == k).sum()) for k in range(K)},
                           published_formula_reproduces_label_pct=100 * (yform == y8).mean(),
                           rows_with_country_median_value_pct=100 * imp8.any(1).mean(), nitrogen_equal_to_median_pct=100 * imp8[:, F.index("N")].mean(),
                           excellent_with_DO_below_objective=int(((pc_.DO < DO_MIN) & (y8 == 4)).sum()),
                           excellent_with_pH_outside_objective=int((~pc_.pH.between(*PH_RANGE) & (y8 == 4)).sum()))]).T.rename(columns={0: "value"})
savetab(facts.reset_index().rename(columns={"index": "item"}), "tab_prior_setting_facts")
PR = ck_load("prior_protocol")
if PR is None:
    PR = []
    for split in ["random stratified 70/30", "station-grouped 70/30"]:
        for sd8 in range(5):
            if split.startswith("random"): a8, b8 = train_test_split(np.arange(len(y8)), test_size=.3, stratify=y8, random_state=sd8)
            else: a8, b8 = next(GroupShuffleSplit(1, test_size=.3, random_state=sd8).split(X8, y8, site8))
            m8 = fitx(make_xgb(XGB_GRID[0], sd8), X8[a8], y8[a8])
            PR.append(dict(protocol=split, K="all (70 % labelled)", model="XGBoost, 8 inputs", seed=sd8, **metrics9(y8[b8], proba(m8, X8[b8]))))
            PR.append(dict(protocol=split, K="no labels", model="Published CCME formula (no learning)", seed=sd8, **metrics9(y8[b8], onehot(yform[b8]))))
    a8, b8 = train_test_split(np.arange(len(y8)), test_size=.3, stratify=y8, random_state=42)
    prep8 = DeepPrep(X8[a8]); enc8 = fit_deep("transformer", prep8(X8[a8]), y8[a8], compute_sample_weight("balanced", y8[a8]), 42, epochs=PRIOR_PRETRAIN_EPOCHS, bs=64)
    xgb8 = fitx(make_xgb(XGB_GRID[0], 42), X8[a8], y8[a8])
    for Kk in KSHOTS:
        for ep in range(PRIOR_EPISODES):
            rng = np.random.default_rng(77_000 + 100 * Kk + ep); sup, qry = [], []
            for k in range(K):
                pk = rng.permutation(b8[y8[b8] == k]); sup += list(pk[:Kk]); qry += list(pk[Kk:Kk + 15])
            sup, qry = np.array(sup), np.array(qry); ys = y8[sup]; Zs8, Zq8 = prep8(X8[sup]), prep8(X8[qry]); seed = 9000 + 100 * Kk + ep
            syn = [wgan_gp(Zs8[ys == k], GAN_SYN, seed + k) for k in np.unique(ys)]; ysyn = np.concatenate([[k] * GAN_SYN for k in np.unique(ys)])
            res8 = {"Transformer-ProtoNet": protonet_P(enc8, Zs8, ys, Zq8),
                    "Transformer-ProtoNet + WGAN-GP (re-implementation)": protonet_P(enc8, np.r_[Zs8, np.concatenate(syn)], np.r_[ys, ysyn], Zq8),
                    "XGBoost on support only": proba(fitx(make_xgb(XGB_GRID[0], seed), X8[sup], ys), X8[qry]),
                    "XGBoost, 8 inputs (70 % base set)": proba(xgb8, X8[qry]), "Published CCME formula (no learning)": onehot(yform[qry])}
            for nm, P_ in res8.items(): PR.append(dict(protocol="episodic (seed-42 split)", K=Kk, model=nm, seed=ep, n_query=len(qry), **metrics9(y8[qry], P_)))
    ck_save("prior_protocol", PR)
PR = pd.DataFrame(PR); savetab(PR, "tab_prior_protocol_all")
prt = PR.groupby(["protocol", "K", "model"], sort=False)[["Accuracy", "MacroF1", "ROC_AUC"]].agg(["mean", "std"])
prt = pd.DataFrame({m_: prt[(m_, "mean")].map("{:.4f}".format) + " ± " + prt[(m_, "std")].fillna(0).map("{:.4f}".format) for m_ in ["Accuracy", "MacroF1", "ROC_AUC"]})
savetab(prt.reset_index(), "tab_prior_protocol")
print("-> with all eight inputs the published label is a closed-form function of the inputs (the formula reproduces it exactly), so this setting "
      "measures how well a model re-learns the formula, not water-quality prediction; our protocol (§7, §9) removes this circularity.")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    fv = facts.round(2)["value"]
    spec = ([("Size", "records", "Records", f_int), ("Size", "stations", "Stations (site identifiers)", f_int)]
            + [("Records per published class", f"n_{k}", k, f_int) for k in CLASSES]
            + [("Published label", "published_formula_reproduces_label_pct", "Published CCME formula reproduces the label (%)", lambda v: f_num(v, 2)),
               ("Data quality", "rows_with_country_median_value_pct", "Records containing a country-median (imputed) value (%)", lambda v: f_num(v, 2)),
               ("Data quality", "nitrogen_equal_to_median_pct", "Nitrogen equal to the country median (%)", lambda v: f_num(v, 2)),
               ("Data quality", "excellent_with_DO_below_objective", f"'Excellent' records with DO below {DO_MIN} mg/L", f_int),
               ("Data quality", "excellent_with_pH_outside_objective", f"'Excellent' records with pH outside {PH_RANGE[0]}–{PH_RANGE[1]}", f_int)])
    spec = [s for s in spec if s[1] in fv.index]
    t = pd.DataFrame({"Value": [fn(fv[k]) for _, k, _, fn in spec]}, index=pd.MultiIndex.from_tuples([(g, lab) for g, _, lab, _ in spec]))
    show_table("prior_facts", "The prior-work setting: Canadian subset as distributed, published label", t, stub="Item",
               note="Setting of Rahman et al. (2026): published label, all eight parameters as inputs, random split.")
    P_ = prt.reset_index(); MET = [("Accuracy (%)", "Accuracy", 100, 2), ("Macro-F1", "MacroF1", 1, 4), ("ROC-AUC", "ROC_AUC", 1, 4)]
    PL = {"random stratified 70/30": "Random stratified 70/30 split", "station-grouped 70/30": "Station-grouped 70/30 split (no station in both parts)"}
    ML = {"XGBoost, 8 inputs": "XGBoost, 8 inputs (70 % of records labelled)", "Published CCME formula (no learning)": "Published CCME formula (no learning)"}
    d = P_[P_.protocol.isin(list(PL))]
    t = pd.DataFrame({lab: [f_pm(v, sc, dd) for v in d[src]] for lab, src, sc, dd in MET},
                     index=pd.MultiIndex.from_tuples([(PL[p], ML.get(m, m)) for p, m in zip(d.protocol, d.model)]))
    show_table("prior_full", "Prior-work protocol with all labels: random vs station-grouped split (Canada, published label)", t, stub="Model",
               note="Mean ± SD over five random splits. With all eight inputs the published label is a closed-form function of the inputs "
                    "(" + tref("prior_facts") + "), so this setting measures how well a model re-learns the formula, not water-quality prediction.")
    e = P_[P_.protocol.str.startswith("episodic")]
    MO = [("Few-shot methods", "Transformer-ProtoNet"), ("Few-shot methods", "Transformer-ProtoNet + WGAN-GP (re-implementation)"),
          ("Few-shot methods", "XGBoost on support only"), ("References", "XGBoost, 8 inputs (70 % base set)"),
          ("References", "Published CCME formula (no learning)")]
    MO = [(g, m) for g, m in MO if m in set(e.model)]
    ks = [k for k in KSHOTS if k in set(e.K)]
    idx = pd.MultiIndex.from_tuples([(f"K = {k} labelled sample{'s' if k > 1 else ''} per class", m) for k in ks for _, m in MO])
    cols = pd.MultiIndex.from_tuples([(lab, "mean ± SD") for lab, _, _, _ in MET])
    get = lambda m, k, src: e[(e.model == m) & (e.K == k)][src].iloc[0]
    rows = [[f_pm(get(m, k, src), sc, dd) for _, src, sc, dd in MET] for k in ks for _, m in MO]
    nums = [[pm_parts(get(m, k, src))[0] for _, src, _, _ in MET] for k in ks for _, m in MO]
    show_table("prior_episodic", "Prior-work episodic few-shot protocol (Canada, published label, all eight inputs)", pd.DataFrame(rows, index=idx, columns=cols),
               stub="Model", bold=best_mask(pd.DataFrame(nums, index=idx, columns=cols), by_group=True, exclude=["Published CCME formula (no learning)"]),
               note=f"Seed-42 70/30 split; {PRIOR_EPISODES} episodes per K, each with K support and 15 query samples per class; the encoder is "
                    "pre-trained on the 70 % base set. Values reported by Rahman et al. (2026): Transformer-ProtoNet 85.3 / 90.7 / 96.0 % at "
                    "K = 1 / 5 / 10; support-only XGBoost 20.0 / 68.0 / 80.0 %; 97.33 % (one 10-shot episode), 96.77 % (100 episodes), "
                    "macro ROC-AUC 99.53 %. The published formula reproduces the label exactly and is not ranked. Bold = best learned model for each K.")
_present()

## 15. Robustness checks
**15a** data as distributed (no imputation handling); **15b** alternative DO, pH and nitrate objectives; **15c** alternative
imputation-detection rules; **15d** alternative phosphate (PO₄) and ammonia (NH₃) objectives — the two laboratory tests the method
measures and the parameters that dominate the label (PO₄ fails its objective in most failing samples); **15e** other rows for the
training labels (tier A; all station rows with imputed values in the labels, as in v13). Nitrate is also tested at 10 mg/L as N
(the drinking-water value of the primary label expressed in the unit Table 9 indicates for England and Ireland). For every variant the label
is recomputed, the share of changed classes is counted on all station site-days and on the experiment pool, and the proposed model,
the black box and the formula rule are re-evaluated on the same samples. Sources of the alternative objectives are in §5b.

In [ ]:
rob = []
TT_LABEL = {"A": "Training labels from tier A rows (all eight values genuine)", "B": "Training labels from tier B rows (laboratory values genuine)",
            "all": "Training labels from all station rows (imputed values in the labels, v13)"}
def run_check(analysis, variant, countries):
    out = []
    for c in countries:
        name = f"rob_{analysis}_{variant}_{c}".replace(" ", "_").replace("/", "_")
        ck = ck_load(name)
        if ck is None:
            T_, FIX_ = TUNED.get(c, (None, None))
            if T_ is None: T_ = {True: dict(prm=XGB_GRID[0], w=5.0), False: dict(prm=XGB_GRID[0], w=5.0)}
            if FIX_ is None: FIX_ = MAINSTORE[PRIMARY[0]]["FIXED"]
            src_all = sources(c); src = np.random.default_rng(100).choice(src_all, min(TRAIN_CAP, len(src_all)), replace=False)
            lab, ev = target_split(c, MAIN_BUDGET); sel = TIER_A[ev] if PROTOCOL == "genuine" else np.ones(len(ev), bool); rows_ = []
            if sel.sum() >= 10:
                M2 = fixed_mask(len(ev), FIX_[2]); P_ = {"Formula rule (last value) – 2 tests": rule_P(ev, M2, "last")}
                for nm, fm in [("Proposed – 2 tests fixed", True), ("Black box – 2 tests", False)]:
                    m_ = train(src, lab, T_[fm]["prm"], T_[fm]["w"], 8000, fm); P_[nm] = predict(m_, ev, M2, fm)
                for nm, Pm in P_.items():
                    rows_.append(dict(analysis=analysis, variant=variant, test_country=c, method=nm, n_eval=int(sel.sum()),
                                      **metrics9(y_all[ev][sel], Pm[sel])))
            ck_save(name, rows_); ck = rows_
        out += ck
    return out
SD_ST = sd.loc[sd.station.to_numpy(), F].to_numpy(float)          # every station site-day of the cleaned dataset
SD_GEN = ~bits_to_mask(SD_BITS["union"])[sd.station.to_numpy()]    # genuinely measured values of those site-days
def nh3_ccme_N(pH, T):
    # CCME (2010): 0.019 mg/L un-ionized NH3; total ammonia from Emerson et al. (1975) pKa; x 0.8224 to express as N; table range pH 6-10, 0-30 °C
    pKa = 0.0901821 + 2729.92 / (np.clip(T, 0, 30) + 273.15); fr = 1 / (10 ** (pKa - np.clip(pH, 6.0, 10.0)) + 1)
    return 0.019 / fr * 0.8224
def nh3_scale(X):
    # rescales NH3 so that the fixed objective GUIDE["NH3"] equals each sample's CCME guideline (identical failures and excursions)
    return GUIDE["NH3"][1] / nh3_ccme_N(X[:, F.index("pH")], X[:, F.index("T")])
def label_of(X, guide, scale_nh3=False):
    V = {f: X[:, j].copy() for j, f in enumerate(F)}
    if scale_nh3: V["NH3"] = V["NH3"] * nh3_scale(X)
    return to_class(ccme_parts(V, guide)[0])
LAB_BASE = {"all": label_of(SD_ST, GUIDE), "pool": label_of(SDF[I0], GUIDE)}
def label_change(guide, scale_nh3=False, par=None):
    a, b = label_of(SD_ST, guide, scale_nh3), label_of(SDF[I0], guide, scale_nh3)
    r_ = dict(changed_all_pct=100 * (a != LAB_BASE["all"]).mean(), changed_pool_pct=100 * (b != LAB_BASE["pool"]).mean(),
              changed_tierA_pct=100 * (b != LAB_BASE["pool"])[TIER_A].mean(), site_days=len(a))
    if par is not None:
        j = F.index(par); x = SD_ST[:, j] * (nh3_scale(SD_ST) if scale_nh3 else 1); lo, hi = guide[par]; fl = np.zeros(len(x), bool)
        if hi is not None: fl |= x > hi
        if lo is not None: fl |= x < lo
        r_["fail_share_pct"] = 100 * fl[SD_GEN[:, j]].mean()          # genuinely measured values only (v13: all values)
    return r_
LABCHG = []
if RUN_ROBUSTNESS:
    t_r = time.time(); set_state("raw"); rob += run_check("as-distributed", "raw data, all samples", ALL_COUNTRIES)
    set_state("genuine", "union"); print(f"15a done ({time.time()-t_r:.0f}s)", flush=True)
    for vname, (par, val) in SENS_VARIANTS.items():
        old = GUIDE[par]; GUIDE[par] = val; LABCHG.append(dict(analysis="guideline", variant=vname, parameter=par, **label_change(dict(GUIDE), par=par)))
        set_state("genuine", "union"); rob += run_check("guideline", vname, PRIMARY); GUIDE[par] = old; set_state("genuine", "union")
        print(f"15b {vname}: {LABCHG[-1]['changed_pool_pct']:.1f}% of labels change", flush=True)
    for rv_ in IMPUTATION_VARIANTS:
        set_state("genuine", rv_); rob += run_check("imputation rule", rv_, PRIMARY); print(f"15c {rv_} done", flush=True)
    set_state("genuine", "union")
    TT_MAIN = TRAIN_TIER
    for tt_ in TRAIN_TIER_VARIANTS:
        TRAIN_TIER = tt_; set_state("genuine", "union"); rob += run_check("training labels", TT_LABEL[tt_], PRIMARY)
        print(f"15e training tier {tt_} done", flush=True)
    TRAIN_TIER = TT_MAIN; set_state("genuine", "union")
    for vname, (par, val) in THRESH_VARIANTS.items():
        if val == "ccme":
            LABCHG.append(dict(analysis="PO4/NH3 objective", variant=vname, parameter=par, **label_change(dict(GUIDE), scale_nh3=True, par=par)))
            jn = F.index("NH3"); bk_ = SDF[:, jn].copy(); SDF[:, jn] = SDF[:, jn] * nh3_scale(SDF)
            set_state("genuine", "union"); rob += run_check("PO4/NH3 objective", vname, PRIMARY); SDF[:, jn] = bk_
        else:
            old = GUIDE[par]; GUIDE[par] = (None, val)
            LABCHG.append(dict(analysis="PO4/NH3 objective", variant=vname, parameter=par, **label_change(dict(GUIDE), par=par)))
            set_state("genuine", "union"); rob += run_check("PO4/NH3 objective", vname, PRIMARY); GUIDE[par] = old
        set_state("genuine", "union"); print(f"15d {vname}: {LABCHG[-1]['changed_pool_pct']:.1f}% of labels change", flush=True)
    LABCHG.insert(0, dict(analysis="primary", variant="primary objectives", parameter="PO4", **label_change(dict(GUIDE), par="PO4")))
    rob = pd.DataFrame(rob); savetab(rob, "tab_robustness_all"); LABCHG = pd.DataFrame(LABCHG).round(4); savetab(LABCHG, "tab_label_change")
    RB3 = ["Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 2 tests fixed"]
    base = mA[mA.method.isin(RB3)].groupby(["test_country", "method"])[["Accuracy", "MacroF1", "MCC", "QWK"]].mean().reset_index().assign(
        analysis="primary", variant=f"union rule, DO ≥ {DO_MIN}, pH {PH_RANGE}")
    allr = pd.concat([base, rob], ignore_index=True)
    rt = allr.groupby(["analysis", "variant", "method"])[["Accuracy", "MacroF1", "MCC", "QWK"]].mean().round(4)
    savetab(rt.reset_index(), "tab_robustness_summary")
    gp = allr.pivot_table(index=["analysis", "variant", "test_country"], columns="method", values="MacroF1")
    g_bb = gp["Proposed – 2 tests fixed"] - gp["Black box – 2 tests"]; g_ru = gp["Proposed – 2 tests fixed"] - gp["Formula rule (last value) – 2 tests"]
    gain = pd.DataFrame({"gain_vs_black_box": g_bb.groupby(level=[0, 1]).mean(), "gain_vs_black_box_min": g_bb.groupby(level=[0, 1]).min(),
                         "gain_vs_rule": g_ru.groupby(level=[0, 1]).mean(), "gain_vs_rule_min": g_ru.groupby(level=[0, 1]).min(),
                         "countries": g_bb.groupby(level=[0, 1]).count()}).round(4)
    savetab(gain.reset_index(), "tab_robustness_gain")
    rc = rob[rob.analysis == "as-distributed"].pivot_table(index="test_country", columns="method", values=["Accuracy", "n_eval"]).round(4)
    fig, ax = plt.subplots(figsize=(9, 4.6)); lc_ = LABCHG.set_index("variant")
    for an_, mk in [("guideline", "o"), ("PO4/NH3 objective", "s")]:
        vv = [v for v in lc_.index if lc_.loc[v, "analysis"] == an_ and (an_, v) in gain.index]
        ax.scatter([lc_.loc[v, "changed_pool_pct"] for v in vv], [gain.loc[(an_, v), "gain_vs_rule"] for v in vv], marker=mk, s=45, label=f"{an_}: proposed − rule")
        ax.scatter([lc_.loc[v, "changed_pool_pct"] for v in vv], [gain.loc[(an_, v), "gain_vs_black_box"] for v in vv], marker=mk, s=45, facecolors="none", edgecolors="k",
                   label=f"{an_}: proposed − black box")
    ax.axhline(0, c="k", lw=.7); ax.set_xlabel("classes changed by the alternative objective (% of experiment pool)"); ax.set_ylabel("Δ macro-F1 (tier A)")
    ax.set_title("Sensitivity to the objectives: label change vs advantage of the proposed model"); ax.legend(fontsize=7); plt.tight_layout()
    savefig("fig14_objective_sensitivity"); plt.show()
    print(f"robustness elapsed {(time.time()-t_r)/60:.1f} min")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    RU, BB_, PF_ = "Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 2 tests fixed"
    IVL = {"documented": "Documented rule only (country median)", "heuristic_2": "Dominant-constant rule, threshold 2 %",
           "heuristic_5": "Dominant-constant rule, threshold 5 %", "heuristic_10": "Dominant-constant rule, threshold 10 %"}
    plan = [("primary", None, "Primary analysis"), ("guideline", list(SENS_VARIANTS), "Alternative DO, pH and nitrate objectives (15b)"),
            ("PO4/NH3 objective", list(THRESH_VARIANTS), "Alternative phosphate and ammonia objectives (15d)"),
            ("imputation rule", IMPUTATION_VARIANTS, "Alternative imputation-detection rules (15c)"),
            ("training labels", None, "Alternative training-label rows (15e)"),
            ("as-distributed", None, "Data as distributed (imputed values kept, all countries, 15a)")]
    lab_ = lambda an, v: (f"Union rule, DO ≥ {DO_MIN} mg/L, pH {PH_RANGE[0]}–{PH_RANGE[1]}, PO₄ ≤ {GUIDE['PO4'][1]}, NH₃ ≤ {GUIDE['NH3'][1]}, training tier {TRAIN_TIER}" if an == "primary"
                          else IVL.get(v, v) if an == "imputation rule" else "Raw data, all samples" if an == "as-distributed" else v)
    def variants(an, order):
        if an not in gain.index.get_level_values(0): return []
        vs = list(gain.loc[an].index)
        return ([v for v in order if v in vs] + [v for v in vs if v not in order]) if order else vs
    lc = LABCHG.set_index("variant"); rows, idx = [], []
    for an, order, glab in plan[:3]:
        for v in (["primary objectives"] if an == "primary" else variants(an, order)):
            if v not in lc.index: continue
            r = lc.loc[v]; rows.append([PNAME.get(r.parameter, r.parameter), f_num(r.fail_share_pct, 1), f_num(r.changed_all_pct, 1),
                                        f_num(r.changed_pool_pct, 1), f_num(r.changed_tierA_pct, 1)])
            idx.append((glab, "Primary objectives" if an == "primary" else v))
    cols = pd.MultiIndex.from_tuples([("Parameter", ""), ("Station site-days failing its objective (%, genuine values)", ""), ("Classes changed (%)", "all station site-days"),
                                      ("Classes changed (%)", "experiment pool"), ("Classes changed (%)", "tier-A pool")])
    show_table("robust_labels", "How much the label depends on each objective", pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(idx), columns=cols),
               stub="Objective", note=f"All station site-days of the cleaned dataset ({f_int(lc.site_days.iloc[0])}) and the experiment pool. The "
                                      "CCME ammonia guideline is computed for each sample from its pH and temperature. Sources of the alternative "
                                      f"values: {tref('objectives')}.")
    MET = [("Accuracy (%)", "Accuracy", f_pct), ("Macro-F1", "MacroF1", f_num), ("MCC", "MCC", f_num)]; W = [(RU, "rule"), (BB_, "black box"), (PF_, "proposed")]
    rows, nums, gr, idx = [], [], [], []
    for an, order, glab in plan:
        for v in variants(an, order):
            rows.append([fn(rt.loc[(an, v, m), src]) if (an, v, m) in rt.index else DASH for _, src, fn in MET for m, _ in W])
            nums.append([float(rt.loc[(an, v, m), src]) if (an, v, m) in rt.index else np.nan for _, src, _ in MET for m, _ in W])
            g = gain.loc[(an, v)]
            gr.append([f_num(g.gain_vs_black_box, 4, True), f_num(g.gain_vs_black_box_min, 4, True), f_num(g.gain_vs_rule, 4, True),
                       f_num(g.gain_vs_rule_min, 4, True), f_int(g.countries)]); idx.append((glab, lab_(an, v)))
    idx = pd.MultiIndex.from_tuples(idx); cols = pd.MultiIndex.from_tuples([(lab, w) for lab, _, _ in MET for _, w in W])
    num = np.array(nums, float); B = np.zeros(num.shape, bool)
    for j0 in range(0, num.shape[1], 3):
        blk = num[:, j0:j0 + 3]; mx = np.nanmax(blk, 1, keepdims=True); B[:, j0:j0 + 3] = np.isclose(blk, mx, atol=1e-12)
    show_table("robust_models", "Robustness checks: formula rule, black box and proposed model under each variant (tier A, 2 laboratory tests)",
               pd.DataFrame(rows, index=idx, columns=cols), stub="Check", bold=B,
               note=f"Mean over the test countries (as-distributed row: all samples of all countries without imputation handling, up to "
                    f"{TEST_CAP:,} evaluation samples per country). The label is recomputed under every variant. Bold = best of the three methods.")
    allpos = bool((gain.gain_vs_black_box_min > 0).all())
    show_table("robust_gains", "Robustness checks: advantage of the proposed model in macro-F1",
               pd.DataFrame(gr, index=idx, columns=pd.MultiIndex.from_tuples([("Proposed − black box", "mean"), ("Proposed − black box", "smallest country"),
                                                                              ("Proposed − formula rule", "mean"), ("Proposed − formula rule", "smallest country"),
                                                                              ("Countries", "")])),
               stub="Check", note="Smallest country = minimum over the test countries. "
                                  + ("The advantage over the black box is positive in every country under every check. " if allpos else "")
                                  + "Differences to the formula rule are reported without a test here; paired tests for the primary analysis are in "
                                  + tref("xbase_tests") + ".")
    r = rc.reindex(countries_in(rc.index))
    t = pd.DataFrame({("Evaluation samples", ""): fmt_list(r[("n_eval", PF_)], f_int), **{("Accuracy (%)", w): fmt_list(r[("Accuracy", m)], f_pct) for m, w in W}},
                     index=list(r.index))
    num = pd.DataFrame({("Accuracy (%)", w): r[("Accuracy", m)].to_numpy(float) for m, w in W}, index=t.index)
    B = np.zeros(t.shape, bool); B[:, 1:] = np.isclose(num.to_numpy(), num.max(1).to_numpy()[:, None], atol=1e-12)
    show_table("asdist", "Data as distributed, by country (all samples, imputed values kept)", t, stub="Test country", bold=B,
               note=f"Accuracy is inflated where imputed constants are common (compare the tier-A shares in {tref('panel')}); these rows are "
                    "supplementary. Bold = best of the three methods.")
if RUN_ROBUSTNESS: _present()

## 16. Quantum reservoir computing — controlled comparison (issue 8)
Eight qubits encode the four current sensors, the two acquired laboratory tests (validation-chosen set) and the two unacquired tests
at *t − 1*. **Equivalence checks:** every model uses the identical training rows and tier-A evaluation rows (asserted); classical
models are also given the **same quantile encoding** as the qubits; QRC and its dimension-matched random-feature control are read out
by both a **linear** (standardised logistic regression) and an **XGBoost** readout; the circuit (time step, encoding scale, number of
time steps) is tuned on the validation country; results are averaged over **3 reservoir seeds**.

In [ ]:
def _ztab(n):
    idx=np.arange(2**n)
    return 1.0-2.0*((idx[:,None]>>np.arange(n-1,-1,-1)[None,:])&1)

def init_state(N,n):
    psi=np.zeros((N,2**n),dtype=np.complex128); psi[:,0]=1.0; return psi

def encoding_layer(psi,X,n,scale):
    """Per-sample RZ(s·x)RY(s·x) applied to an existing state (enables re-uploading)."""
    N=psi.shape[0]; d=X.shape[1]
    for q in range(n):
        th=scale*X[:,q%d]; c,s_=np.cos(th/2),np.sin(th/2); e=np.exp(1j*th)
        U=np.empty((N,2,2),dtype=np.complex128)
        U[:,0,0]=c; U[:,0,1]=-s_; U[:,1,0]=s_*e; U[:,1,1]=c*e
        psi=np.einsum("iab,ipbk->ipak",U,psi.reshape(N,2**q,2,2**(n-1-q))).reshape(N,-1)
    return psi

class Reservoir:
    def __init__(self,n,seed=0,J=1.0,h=1.0,g=0.3):
        self.n=n; rng=np.random.default_rng(seed)
        self.J=np.triu(rng.uniform(-J,J,(n,n)),1)
        self.h=rng.uniform(.5*h,1.5*h,n); self.g=rng.uniform(-g,g,n)
        z=_ztab(n); self.Hd=np.einsum("bi,bj,ij->b",z,z,self.J)+z@self.g
    def step(self,psi,dt,trotter=4):
        sub=dt/trotter
        for _ in range(trotter):
            psi=psi*np.exp(-1j*(sub/2)*self.Hd)[None,:]
            N=psi.shape[0]
            for q in range(self.n):
                th=2*self.h[q]*sub; c,s=np.cos(th/2),-1j*np.sin(th/2)
                U=np.array([[c,s],[s,c]],dtype=np.complex128)
                psi=np.einsum("ab,ipbk->ipak",U,psi.reshape(N,2**q,2,2**(self.n-1-q))).reshape(N,-1)
            psi=psi*np.exp(-1j*(sub/2)*self.Hd)[None,:]
        return psi
    def measure(self,psi):
        N,n=psi.shape[0],self.n; probs=np.abs(psi)**2; z=_ztab(n)
        out=[probs@z]
        xs,ys=np.empty((N,n)),np.empty((N,n))
        for q in range(n):
            p=psi.reshape(N,2**q,2,2**(n-1-q))
            c=np.einsum("ipk,ipk->i",np.conj(p[:,:,0,:]),p[:,:,1,:])
            xs[:,q],ys[:,q]=2*c.real,2*c.imag
        out+=[xs,ys]
        iu,ju=np.triu_indices(n,1)
        out.append(probs@(z[:,iu]*z[:,ju]))
        xx,yy=np.empty((N,len(iu))),np.empty((N,len(iu)))
        for k,(i_,j_) in enumerate(zip(iu,ju)):
            a=psi.reshape(N,2**i_,2,2**(j_-i_-1),2,2**(n-1-j_))
            t1=np.einsum("ipqk,ipqk->i",np.conj(a[:,:,0,:,0,:]),a[:,:,1,:,1,:])
            t2=np.einsum("ipqk,ipqk->i",np.conj(a[:,:,0,:,1,:]),a[:,:,1,:,0,:])
            xx[:,k]=2*(t1+t2).real; yy[:,k]=2*(t2-t1).real
        out+=[xx,yy]
        return np.concatenate(out,1)

def extract(Xq,n,R=R_RES,V=V_STEPS,dt=DT,seed=RESERVOIR_SEED,scale=SCALE,
            reupload=True,shots=None,batch=512):
    N=Xq.shape[0]; rng=np.random.default_rng(seed+999); blocks=[]
    for r in range(R):
        res=Reservoir(n,seed=seed+100*r); chunks=[]
        for s0 in range(0,N,batch):
            xb=Xq[s0:s0+batch]
            psi=encoding_layer(init_state(len(xb),n),xb,n,scale); sl=[]
            for v in range(V):
                if reupload and v>0: psi=encoding_layer(psi,xb,n,scale)
                psi=res.step(psi,dt); sl.append(res.measure(psi))
            chunks.append(np.concatenate(sl,1))
        b=np.concatenate(chunks,0)
        if shots:
            p=np.clip((1+b)/2,0,1); b=2*rng.binomial(shots,p)/shots-1
        blocks.append(b)
    return np.concatenate(blocks,1)
print("quantum reservoir defined")

In [ ]:
qrc_rows = []
def lin(): return Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=3000, C=1.0))])
def fit_any(m, X, y_, w):
    if isinstance(m, Pipeline): return m.fit(X, y_, m__sample_weight=w)
    return fitx(m, X, y_, w)
if RUN_QRC:
    for c in PRIMARY:
        ck = ck_load(f"qrc_{c}")
        if ck is not None: qrc_rows += ck; print(f"[{c}] QRC restored", flush=True); continue
        t_q = time.time(); n_q = len(qrc_rows); FIX2 = list(MAINSTORE[c]["FIXED"][2]); oth = [j for j in range(4) if j not in FIX2]
        q_in = lambda r: np.c_[SENCUR[r], LABCUR[r][:, FIX2], LABLAG[r][:, oth]]
        xcols = [i for i, n_ in enumerate(NAMES[True]) if n_ in H_COLS + CTX_COLS + FORM_NAMES]
        extra = lambda r: featurize(r, fixed_mask(len(r), FIX2), True)[:, xcols]
        lab, ev = target_split(c, MAIN_BUDGET); ev = ev[TIER_A[ev]][:QRC_EVAL_CAP]; yt = y_all[ev]
        rq = np.random.default_rng(5); src_all = sources(c); qsrc = rq.choice(src_all, min(QRC_TRAIN_CAP, len(src_all)), replace=False)
        tr = np.r_[qsrc, lab]; ytr = y_all[tr]; w = compute_sample_weight("balanced", ytr) * np.where(np.isin(tr, lab), MAINSTORE[c]["T"][True]["w"], 1.0)
        qt = QuantileTransformer(n_quantiles=500, random_state=0).fit(q_in(qsrc))
        enc = lambda r: np.nan_to_num(qt.transform(q_in(r)), nan=0.5)             # hidden (imputed) inputs -> neutral angle
        val_c = VAL_OF[c]; tv = src_all[ctry[src_all] != val_c]
        a_ = rq.choice(tv, min(QRC_TUNE_N[0], len(tv)), replace=False); pvv = np.where((ctry == val_c) & STATION & TIER_B)[0]
        b_ = rq.choice(pvv, min(QRC_TUNE_N[1], len(pvv)), replace=False); best = (None, -1)
        for dt_, sc_, v_ in QRC_GRID:
            Fa = extract(enc(a_), N_QUBITS, R=R_RES, V=v_, dt=dt_, scale=sc_, seed=QRC_SEEDS[0]); Fb = extract(enc(b_), N_QUBITS, R=R_RES, V=v_, dt=dt_, scale=sc_, seed=QRC_SEEDS[0])
            s_ = f1_score(y_all[b_], proba(fitx(make_xgb(XGB_GRID[0]), Fa, y_all[a_], compute_sample_weight("balanced", y_all[a_])), Fb).argmax(1), average="macro")
            if s_ > best[1]: best = ((dt_, sc_, v_), s_)
        dtq, scq, vq = best[0]
        Etr, Eev = enc(tr), enc(ev)
        for sd_ in QRC_SEEDS:
            FQtr = extract(Etr, N_QUBITS, R=R_RES, V=vq, dt=dtq, scale=scq, seed=sd_); FQev = extract(Eev, N_QUBITS, R=R_RES, V=vq, dt=dtq, scale=scq, seed=sd_)
            rgc = np.random.default_rng(1000 + sd_); Wc = rgc.normal(0, 1, (N_QUBITS, FQtr.shape[1])); bc = rgc.uniform(0, 2 * np.pi, FQtr.shape[1])
            Ctr, Cev = np.tanh(Etr @ Wc + bc), np.tanh(Eev @ Wc + bc)
            models = {"XGBoost – 8 raw inputs": (make_xgb(XGB_GRID[0], sd_), q_in(tr), q_in(ev)),
                      "XGBoost – 8 encoded inputs": (make_xgb(XGB_GRID[0], sd_), Etr, Eev),
                      "Linear – 8 encoded inputs": (lin(), Etr, Eev),
                      "QRC – XGBoost readout": (make_xgb(XGB_GRID[0], sd_), FQtr, FQev),
                      "QRC – linear readout": (lin(), FQtr, FQev),
                      "Random features – XGBoost readout": (make_xgb(XGB_GRID[0], sd_), Ctr, Cev),
                      "Random features – linear readout": (lin(), Ctr, Cev),
                      "XGBoost – 8 inputs + temporal & CCME": (make_xgb(XGB_GRID[0], sd_), np.hstack([q_in(tr), extra(tr)]), np.hstack([q_in(ev), extra(ev)])),
                      "QRC + temporal & CCME – XGBoost readout": (make_xgb(XGB_GRID[0], sd_), np.hstack([FQtr, extra(tr)]), np.hstack([FQev, extra(ev)]))}
            assert len({A_.shape[0] for _, A_, _ in models.values()}) == 1 and len({B_.shape[0] for _, _, B_ in models.values()}) == 1, "identical rows required"
            for nm, (m_, A_, B_) in models.items():
                qrc_rows.append(dict(test_country=c, reservoir_seed=sd_, method=nm, dt=dtq, scale=scq, steps=vq, n_train=len(tr), n_eval=len(ev),
                                     **metrics9(yt, proba(fit_any(m_, A_, ytr, w), B_))))
        ck_save(f"qrc_{c}", qrc_rows[n_q:]); print(f"[{c}] QRC dt={dtq} scale={scq:.2f} steps={vq} | {FQtr.shape[1]} observables | {time.time()-t_q:.0f}s", flush=True)
    qrc = pd.DataFrame(qrc_rows); savetab(qrc, "tab_qrc_all")
    QM = ["XGBoost – 8 raw inputs", "XGBoost – 8 encoded inputs", "Linear – 8 encoded inputs", "QRC – linear readout", "Random features – linear readout",
          "QRC – XGBoost readout", "Random features – XGBoost readout", "XGBoost – 8 inputs + temporal & CCME", "QRC + temporal & CCME – XGBoost readout"]
    qc = qrc.groupby(["test_country", "method"])[["Accuracy", "MacroF1", "MCC", "QWK", "ROC_AUC"]].mean()
    qs = qrc.groupby(["test_country", "method"]).MacroF1.std().groupby("method").mean()
    TQ = qc.groupby("method").mean().reindex(QM); TQ["MacroF1_SD_over_seeds"] = qs.reindex(QM); TQ = TQ.round(4)
    savetab(TQ.reset_index(), "tab_qrc_summary")
    qd = []
    for a, b in [("QRC – linear readout", "Random features – linear readout"), ("QRC – XGBoost readout", "Random features – XGBoost readout"),
                 ("QRC – XGBoost readout", "XGBoost – 8 encoded inputs"), ("QRC – linear readout", "Linear – 8 encoded inputs"),
                 ("QRC + temporal & CCME – XGBoost readout", "XGBoost – 8 inputs + temporal & CCME")]:
        dd = qrc.pivot_table(index=["test_country", "reservoir_seed"], columns="method", values="MacroF1"); d = dd[a] - dd[b]
        qd.append(dict(comparison=f"{a} vs {b}", mean_diff_MacroF1=d.mean(), min=d.min(), max=d.max(), positive=f"{int((d > 0).sum())}/{len(d)}"))
    QD = pd.DataFrame(qd); savetab(QD, "tab_qrc_differences")

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    QG = [("Classical models on the same eight inputs", ["XGBoost – 8 raw inputs", "XGBoost – 8 encoded inputs", "Linear – 8 encoded inputs"]),
          ("Reservoir features with a linear readout", ["QRC – linear readout", "Random features – linear readout"]),
          ("Reservoir features with an XGBoost readout", ["QRC – XGBoost readout", "Random features – XGBoost readout"]),
          ("With temporal and CCME-consistency features (XGBoost)", ["XGBoost – 8 inputs + temporal & CCME", "QRC + temporal & CCME – XGBoost readout"])]
    QN = {"XGBoost – 8 raw inputs": "XGBoost, 8 raw inputs", "XGBoost – 8 encoded inputs": "XGBoost, 8 encoded inputs",
          "Linear – 8 encoded inputs": "Linear model, 8 encoded inputs", "QRC – linear readout": "Quantum reservoir (QRC)",
          "Random features – linear readout": "Classical random features", "QRC – XGBoost readout": "Quantum reservoir (QRC)",
          "Random features – XGBoost readout": "Classical random features", "XGBoost – 8 inputs + temporal & CCME": "XGBoost, 8 inputs + temporal & CCME",
          "QRC + temporal & CCME – XGBoost readout": "QRC + temporal & CCME"}
    keys = [(g, m) for g, ms in QG for m in ms if m in TQ.index]; d = TQ.loc[[m for _, m in keys]]
    idx = pd.MultiIndex.from_tuples([(g, QN.get(m, m)) for g, m in keys])
    spec = [("Accuracy (%)", "Accuracy", f_pct), ("Macro-F1", "MacroF1", f_num), ("MCC", "MCC", f_num), ("QWK", "QWK", f_num),
            ("ROC-AUC", "ROC_AUC", f_num), ("Macro-F1 SD over seeds", "MacroF1_SD_over_seeds", f_num)]
    t = pd.DataFrame({lab: fmt_list(d[src], fn) for lab, src, fn in spec}, index=idx)
    num = pd.DataFrame({lab: d[src].to_numpy(float) for lab, src, _ in spec}, index=idx)
    show_table("qrc", "Quantum reservoir computing (QRC) vs classical models with identical inputs (tier A)", t, stub="Model",
               bold=best_mask(num, cols=[lab for lab, _, _ in spec[:-1]], by_group=True),
               note=f"Identical training and evaluation rows for every model (up to {QRC_EVAL_CAP:,} tier-A evaluation samples per country); mean "
                    f"over {', '.join(PRIMARY)}, each averaged over {len(QRC_SEEDS)} seeds. The quantum reservoir is classically simulated. "
                    "Bold = best within each group.")
    QP = [("QRC – linear readout", "Random features – linear readout", "QRC vs classical random features (linear readout)"),
          ("QRC – XGBoost readout", "Random features – XGBoost readout", "QRC vs classical random features (XGBoost readout)"),
          ("QRC – XGBoost readout", "XGBoost – 8 encoded inputs", "QRC (XGBoost readout) vs XGBoost on the same 8 encoded inputs"),
          ("QRC – linear readout", "Linear – 8 encoded inputs", "QRC (linear readout) vs linear model on the same 8 encoded inputs"),
          ("QRC + temporal & CCME – XGBoost readout", "XGBoost – 8 inputs + temporal & CCME", "QRC + temporal & CCME vs XGBoost + temporal & CCME")]
    q = QD.round(4); assert len(q) == len(QP)
    for (a, b, _), s in zip(QP, q.comparison): assert f"{a} vs {b}".startswith(str(s).rstrip(".").rstrip()), s
    t = pd.DataFrame({"Mean Δ macro-F1": [f_num(v, 4, True) for v in q.mean_diff_MacroF1], "Smallest Δ": [f_num(v, 4, True) for v in q["min"]],
                      "Largest Δ": [f_num(v, 4, True) for v in q["max"]], "Δ > 0 (country × seed)": [str(v).replace("/", " of ") for v in q.positive]},
                     index=[lab for _, _, lab in QP])
    show_table("qrc_diff", "QRC: paired differences in macro-F1", t, stub="Comparison",
               note="Δ = first − second model, paired over test countries × seeds.")
if RUN_QRC: _present()

## 16b. Changes relative to v13
v14 corrects the v13 method (change log at the top), so its values are expected to differ. The cell below lists, value by value,
how the headline, calibration, statistical, deep-learning and QRC results changed relative to v13 (whose values were identical to
v12). Report these changes in the response to reviewers; they are not a reproducibility test.


In [ ]:
V13_REF = {   # values of the executed v13 notebook (identical to the v12 run)
    ('headline', 'Persistence', 'Accuracy'): 0.5061,
    ('headline', 'Persistence', 'MacroF1'): 0.4004,
    ('headline', 'Persistence', 'MCC'): 0.2362,
    ('headline', 'Persistence', 'QWK'): 0.3746,
    ('headline', 'Persistence', 'ROC_AUC'): 0.6302,
    ('headline', 'Majority class', 'Accuracy'): 0.2631,
    ('headline', 'Majority class', 'MacroF1'): 0.0736,
    ('headline', 'Majority class', 'MCC'): 0.0,
    ('headline', 'Majority class', 'QWK'): 0.0,
    ('headline', 'Majority class', 'ROC_AUC'): 0.5,
    ('headline', 'Forecast only (history)', 'Accuracy'): 0.4817,
    ('headline', 'Forecast only (history)', 'MacroF1'): 0.3037,
    ('headline', 'Forecast only (history)', 'MCC'): 0.1526,
    ('headline', 'Forecast only (history)', 'QWK'): 0.3202,
    ('headline', 'Forecast only (history)', 'ROC_AUC'): 0.7421,
    ('headline', 'Formula rule (assume pass) – 2 tests', 'Accuracy'): 0.9451,
    ('headline', 'Formula rule (assume pass) – 2 tests', 'MacroF1'): 0.87,
    ('headline', 'Formula rule (assume pass) – 2 tests', 'MCC'): 0.9161,
    ('headline', 'Formula rule (assume pass) – 2 tests', 'QWK'): 0.9621,
    ('headline', 'Formula rule (assume pass) – 2 tests', 'ROC_AUC'): 0.9186,
    ('headline', 'Formula rule (last value) – 2 tests', 'Accuracy'): 0.9472,
    ('headline', 'Formula rule (last value) – 2 tests', 'MacroF1'): 0.9202,
    ('headline', 'Formula rule (last value) – 2 tests', 'MCC'): 0.9206,
    ('headline', 'Formula rule (last value) – 2 tests', 'QWK'): 0.9636,
    ('headline', 'Formula rule (last value) – 2 tests', 'ROC_AUC'): 0.9416,
    ('headline', 'Black box – 2 tests', 'Accuracy'): 0.9076,
    ('headline', 'Black box – 2 tests', 'MacroF1'): 0.787,
    ('headline', 'Black box – 2 tests', 'MCC'): 0.8541,
    ('headline', 'Black box – 2 tests', 'QWK'): 0.9302,
    ('headline', 'Black box – 2 tests', 'ROC_AUC'): 0.9708,
    ('headline', 'Proposed – 1 tests adaptive', 'Accuracy'): 0.9011,
    ('headline', 'Proposed – 1 tests adaptive', 'MacroF1'): 0.8135,
    ('headline', 'Proposed – 1 tests adaptive', 'MCC'): 0.847,
    ('headline', 'Proposed – 1 tests adaptive', 'QWK'): 0.9247,
    ('headline', 'Proposed – 1 tests adaptive', 'ROC_AUC'): 0.9651,
    ('headline', 'Proposed – 2 tests fixed', 'Accuracy'): 0.9454,
    ('headline', 'Proposed – 2 tests fixed', 'MacroF1'): 0.9028,
    ('headline', 'Proposed – 2 tests fixed', 'MCC'): 0.9166,
    ('headline', 'Proposed – 2 tests fixed', 'QWK'): 0.9632,
    ('headline', 'Proposed – 2 tests fixed', 'ROC_AUC'): 0.986,
    ('headline', 'Proposed – 2 tests adaptive', 'Accuracy'): 0.9422,
    ('headline', 'Proposed – 2 tests adaptive', 'MacroF1'): 0.8736,
    ('headline', 'Proposed – 2 tests adaptive', 'MCC'): 0.9092,
    ('headline', 'Proposed – 2 tests adaptive', 'QWK'): 0.9595,
    ('headline', 'Proposed – 2 tests adaptive', 'ROC_AUC'): 0.9827,
    ('calibration', 'Formula rule (last value) – 2 tests', 'ECE'): 0.0565,
    ('calibration', 'Formula rule (last value) – 2 tests', 'Brier'): 0.1497,
    ('calibration', 'Formula rule (last value) – 2 tests', 'LogLoss'): 0.424,
    ('calibration', 'Black box – 2 tests', 'ECE'): 0.0371,
    ('calibration', 'Black box – 2 tests', 'Brier'): 0.1745,
    ('calibration', 'Black box – 2 tests', 'LogLoss'): 0.3549,
    ('calibration', 'Proposed – 2 tests fixed', 'ECE'): 0.0439,
    ('calibration', 'Proposed – 2 tests fixed', 'Brier'): 0.1212,
    ('calibration', 'Proposed – 2 tests fixed', 'LogLoss'): 0.2943,
    ('calibration', 'Proposed – 2 tests adaptive', 'ECE'): 0.0403,
    ('calibration', 'Proposed – 2 tests adaptive', 'Brier'): 0.115,
    ('calibration', 'Proposed – 2 tests adaptive', 'LogLoss'): 0.2723,
    ('inference', 0, 'mean_diff'): 0.0377,
    ('inference', 0, 'boot_CI_low'): 0.0268,
    ('inference', 0, 'boot_CI_high'): 0.0523,
    ('inference', 0, 'RE_diff'): 0.0414,
    ('inference', 1, 'mean_diff'): 0.1159,
    ('inference', 1, 'boot_CI_low'): 0.0573,
    ('inference', 1, 'boot_CI_high'): 0.1827,
    ('inference', 1, 'RE_diff'): 0.1073,
    ('inference', 2, 'mean_diff'): -0.0018,
    ('inference', 2, 'boot_CI_low'): -0.0153,
    ('inference', 2, 'boot_CI_high'): 0.0065,
    ('inference', 2, 'RE_diff'): 0.0048,
    ('inference', 3, 'mean_diff'): -0.0173,
    ('inference', 3, 'boot_CI_low'): -0.0329,
    ('inference', 3, 'boot_CI_high'): 0.003,
    ('inference', 3, 'RE_diff'): -0.005,
    ('inference', 4, 'mean_diff'): 0.0003,
    ('inference', 4, 'boot_CI_low'): -0.002,
    ('inference', 4, 'boot_CI_high'): 0.0025,
    ('inference', 4, 'RE_diff'): 0.0007,
    ('inference', 5, 'mean_diff'): 0.0329,
    ('inference', 5, 'boot_CI_low'): -0.0011,
    ('inference', 5, 'boot_CI_high'): 0.0511,
    ('inference', 5, 'RE_diff'): 0.0078,
    ('inference', 6, 'mean_diff'): -0.0032,
    ('inference', 6, 'boot_CI_low'): -0.023,
    ('inference', 6, 'boot_CI_high'): 0.0159,
    ('inference', 6, 'RE_diff'): -0.0001,
    ('inference', 7, 'mean_diff'): -0.0292,
    ('inference', 7, 'boot_CI_low'): -0.0715,
    ('inference', 7, 'boot_CI_high'): 0.0038,
    ('inference', 7, 'RE_diff'): -0.0178,
    ('inference', 8, 'mean_diff'): 0.4393,
    ('inference', 8, 'boot_CI_low'): 0.3807,
    ('inference', 8, 'boot_CI_high'): 0.4995,
    ('inference', 8, 'RE_diff'): 0.4431,
    ('inference', 9, 'mean_diff'): 0.5025,
    ('inference', 9, 'boot_CI_low'): 0.4297,
    ('inference', 9, 'boot_CI_high'): 0.5937,
    ('inference', 9, 'RE_diff'): 0.48,
    ('inference', 10, 'mean_diff'): 0.2747,
    ('inference', 10, 'boot_CI_low'): 0.2245,
    ('inference', 10, 'boot_CI_high'): 0.3273,
    ('inference', 10, 'RE_diff'): 0.2758,
    ('inference', 11, 'mean_diff'): 0.2424,
    ('inference', 11, 'boot_CI_low'): 0.1869,
    ('inference', 11, 'boot_CI_high'): 0.3516,
    ('inference', 11, 'RE_diff'): 0.2427,
    ('inference', 12, 'mean_diff'): 0.0015,
    ('inference', 12, 'boot_CI_low'): -0.0005,
    ('inference', 12, 'boot_CI_high'): 0.0036,
    ('inference', 12, 'RE_diff'): 0.0012,
    ('inference', 13, 'mean_diff'): -0.0121,
    ('inference', 13, 'boot_CI_low'): -0.0449,
    ('inference', 13, 'boot_CI_high'): 0.0067,
    ('inference', 13, 'RE_diff'): 0.0078,
    ('inference', 14, 'mean_diff'): 0.0,
    ('inference', 14, 'boot_CI_low'): 0.0,
    ('inference', 14, 'boot_CI_high'): 0.0,
    ('inference', 14, 'RE_diff'): 0.0,
    ('inference', 15, 'mean_diff'): 0.0,
    ('inference', 15, 'boot_CI_low'): 0.0,
    ('inference', 15, 'boot_CI_high'): 0.0,
    ('inference', 15, 'RE_diff'): 0.0,
    ('inference', 16, 'mean_diff'): 0.0125,
    ('inference', 16, 'boot_CI_low'): -0.0018,
    ('inference', 16, 'boot_CI_high'): 0.0266,
    ('inference', 16, 'RE_diff'): 0.008,
    ('inference', 17, 'mean_diff'): 0.0253,
    ('inference', 17, 'boot_CI_low'): -0.0299,
    ('inference', 17, 'boot_CI_high'): 0.0515,
    ('inference', 17, 'RE_diff'): 0.0206,
    ('zero-shot deep', 'Formula rule (last value)', 'Accuracy'): 0.9472,
    ('zero-shot deep', 'Formula rule (last value)', 'MacroF1'): 0.9202,
    ('zero-shot deep', 'Formula rule (last value)', 'ROC_AUC'): 0.9416,
    ('zero-shot deep', 'MLP', 'Accuracy'): 0.4165,
    ('zero-shot deep', 'MLP', 'MacroF1'): 0.4264,
    ('zero-shot deep', 'MLP', 'ROC_AUC'): 0.7805,
    ('zero-shot deep', 'Transformer', 'Accuracy'): 0.5545,
    ('zero-shot deep', 'Transformer', 'MacroF1'): 0.5081,
    ('zero-shot deep', 'Transformer', 'ROC_AUC'): 0.8593,
    ('zero-shot deep', 'Transformer-ProtoNet', 'Accuracy'): 0.5411,
    ('zero-shot deep', 'Transformer-ProtoNet', 'MacroF1'): 0.5095,
    ('zero-shot deep', 'Transformer-ProtoNet', 'ROC_AUC'): 0.7475,
    ('zero-shot deep', 'Transformer + CCME features', 'Accuracy'): 0.9432,
    ('zero-shot deep', 'Transformer + CCME features', 'MacroF1'): 0.9173,
    ('zero-shot deep', 'Transformer + CCME features', 'ROC_AUC'): 0.9747,
    ('zero-shot deep', 'Transformer-ProtoNet + CCME features', 'Accuracy'): 0.9435,
    ('zero-shot deep', 'Transformer-ProtoNet + CCME features', 'MacroF1'): 0.9182,
    ('zero-shot deep', 'Transformer-ProtoNet + CCME features', 'ROC_AUC'): 0.9199,
    ('zero-shot deep', 'Black box (XGBoost)', 'Accuracy'): 0.7543,
    ('zero-shot deep', 'Black box (XGBoost)', 'MacroF1'): 0.6227,
    ('zero-shot deep', 'Black box (XGBoost)', 'ROC_AUC'): 0.9038,
    ('zero-shot deep', 'Proposed (XGBoost + CCME + constraint)', 'Accuracy'): 0.9447,
    ('zero-shot deep', 'Proposed (XGBoost + CCME + constraint)', 'MacroF1'): 0.9188,
    ('zero-shot deep', 'Proposed (XGBoost + CCME + constraint)', 'ROC_AUC'): 0.9817,
    ('QRC', 'XGBoost – 8 raw inputs', 'Accuracy'): 0.9195,
    ('QRC', 'XGBoost – 8 raw inputs', 'MacroF1'): 0.8653,
    ('QRC', 'XGBoost – 8 raw inputs', 'ROC_AUC'): 0.9745,
    ('QRC', 'XGBoost – 8 encoded inputs', 'Accuracy'): 0.9125,
    ('QRC', 'XGBoost – 8 encoded inputs', 'MacroF1'): 0.8551,
    ('QRC', 'XGBoost – 8 encoded inputs', 'ROC_AUC'): 0.9728,
    ('QRC', 'Linear – 8 encoded inputs', 'Accuracy'): 0.5721,
    ('QRC', 'Linear – 8 encoded inputs', 'MacroF1'): 0.41,
    ('QRC', 'Linear – 8 encoded inputs', 'ROC_AUC'): 0.8229,
    ('QRC', 'QRC – linear readout', 'Accuracy'): 0.6628,
    ('QRC', 'QRC – linear readout', 'MacroF1'): 0.5671,
    ('QRC', 'QRC – linear readout', 'ROC_AUC'): 0.8851,
    ('QRC', 'Random features – linear readout', 'Accuracy'): 0.635,
    ('QRC', 'Random features – linear readout', 'MacroF1'): 0.538,
    ('QRC', 'Random features – linear readout', 'ROC_AUC'): 0.8439,
    ('QRC', 'QRC – XGBoost readout', 'Accuracy'): 0.7845,
    ('QRC', 'QRC – XGBoost readout', 'MacroF1'): 0.731,
    ('QRC', 'QRC – XGBoost readout', 'ROC_AUC'): 0.9374,
    ('QRC', 'Random features – XGBoost readout', 'Accuracy'): 0.6935,
    ('QRC', 'Random features – XGBoost readout', 'MacroF1'): 0.6429,
    ('QRC', 'Random features – XGBoost readout', 'ROC_AUC'): 0.9045,
    ('QRC', 'XGBoost – 8 inputs + temporal & CCME', 'Accuracy'): 0.9484,
    ('QRC', 'XGBoost – 8 inputs + temporal & CCME', 'MacroF1'): 0.9462,
    ('QRC', 'XGBoost – 8 inputs + temporal & CCME', 'ROC_AUC'): 0.9807,
    ('QRC', 'QRC + temporal & CCME – XGBoost readout', 'Accuracy'): 0.9471,
    ('QRC', 'QRC + temporal & CCME – XGBoost readout', 'MacroF1'): 0.9209,
    ('QRC', 'QRC + temporal & CCME – XGBoost readout', 'ROC_AUC'): 0.9817,
}          # tables of §10, §10b, §12, §14b, §16 of v13
rep_rows = []
def _cmp(tab, obj, rkey, col, old):
    try: new = float(obj.loc[rkey, col])
    except (KeyError, TypeError): new = np.nan
    rep_rows.append(dict(table=tab, row=str(rkey), column=col, v13=old, v14=new, change=new - old if not np.isnan(new) else np.nan))
for (tab, rkey, col), v in V13_REF.items():
    obj = {"headline": HEAD, "calibration": CAL, "inference": INF, "zero-shot deep": zt if RUN_DEEP else None, "QRC": TQ if RUN_QRC else None}[tab]
    if obj is not None: _cmp(tab, obj, rkey, col, v)
REPRO = pd.DataFrame(rep_rows); REPRO["unchanged_at_4dp"] = REPRO.change.abs() < 5e-5; savetab(REPRO, "tab_changes_vs_v13")
print(REPRO.groupby("table").agg(values=("v13", "size"), unchanged=("unchanged_at_4dp", "sum"), max_abs_change=("change", lambda s: s.abs().max())))


In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    g = REPRO.groupby("table", sort=False).agg(n=("v13", "size"), same=("unchanged_at_4dp", "sum"), mx=("change", lambda s: s.abs().max()))
    t = pd.DataFrame({"Values compared": fmt_list(g.n, f_int), "Unchanged at 4 decimals": fmt_list(g.same, f_int),
                      "Largest |change|": [DASH if _isnan(v) else f"{v:.4f}" for v in g.mx]}, index=list(g.index))
    show_table("repro", "Changes relative to v13 (effect of the v14 corrections)", t, stub="Results table",
               note="v13 values (identical to the v12 run). v14 uses no test-country labels in the headline, no imputed values in the training "
                    "labels, the last genuine value instead of the value at t−1, all Monte-Carlo draws in the adaptive selection and fixed QWK "
                    "weights, so changes are expected. Value-by-value changes: tab_changes_vs_v13.csv.")
_present()


## 17. Final conclusion — every statement below is computed from this run

In [ ]:
# Tables (presentation of the results above; numbered automatically)
def _present():
    pf, bb, rl = "Proposed – 2 tests fixed", "Black box – 2 tests", "Formula rule (last value) – 2 tests"
    I = inf_pairs(INF.round(4), COMP); inf = lambda a, b, met: I[(I.pair_a == a) & (I.pair_b == b) & (I.metric == met)].iloc[0]
    H = HEAD; rows = []; v = VSRULE.set_index(["method", "metric"])
    sig = lambda lo, hi: "no significant difference" if lo <= 0 <= hi else ("better" if lo > 0 else "worse")
    rows.append(("Data", f"{AUDIT['raw']:,} records, {AUDIT['after_cleaning']:,} after range checks and de-duplication; test countries {', '.join(PRIMARY)}",
                 tref("overview", "range", "panel")))
    rows.append(("Published label", f"reproduced exactly by a formula with DO and pH as maxima and denominator 12; {AUDIT['hypoxic_excellent']:,} hypoxic records "
                                    f"labelled Excellent; the correction changes {AUDIT['class_changed_by_correction_pct']:.1f} % of classes", tref("label_audit")))
    rows.append(("Single-sample label", f"agrees with the standard site-year CCME class in {100 * STD_CMP['class_agreement']:.1f} % of site-years "
                                        f"(Spearman {STD_CMP['spearman']:.3f})", tref("label_audit")))
    if RUN_ROBUSTNESS:
        lc = LABCHG.set_index("variant"); po = lc[(lc.analysis == "PO4/NH3 objective") & (lc.parameter == "PO4")]; nh = lc[(lc.analysis == "PO4/NH3 objective") & (lc.parameter == "NH3")]
        rows.append(("Label sensitivity", f"PO₄ fails its objective in {lc.loc['primary objectives', 'fail_share_pct']:.1f} % of station site-days; other PO₄ objectives change "
                                          f"{po.changed_all_pct.min():.1f}–{po.changed_all_pct.max():.1f} % of classes, NH₃ objectives "
                                          f"{nh.changed_all_pct.min():.1f}–{nh.changed_all_pct.max():.1f} %", tref("robust_labels")))
    rows.append(("Leakage audit", f"{(LEAKAGE.status == 'pass').sum()} tested checks passed, none failed", tref("leakage")))
    rows.append(("Training labels", f"no imputed value of tier {TRAIN_TIER} in any training, tuning or support label; alternatives in §15e", tref("panel", "robust_models")))
    rows.append((f"Main result ({budget_txt()}): proposed, 2 tests", f"accuracy {f_pct(H.loc[pf, 'Accuracy'])} % {f_ci(H.loc[pf, 'Accuracy_CI'], 2, 100)}; macro-F1 "
                                                   f"{f_num(H.loc[pf, 'MacroF1'])}; MCC {f_num(H.loc[pf, 'MCC'])}; ROC-AUC {f_num(H.loc[pf, 'ROC_AUC'])}", tref("headline")))
    ra, rf_, rm = v.loc[(pf, "Accuracy")], v.loc[(pf, "MacroF1")], v.loc[(pf, "MCC")]
    rows.append(("Proposed vs formula rule", f"accuracy {f_pct(ra.mean_diff, 2, True)} pp [{f_pct(ra.boot_CI_low, 2, True)}, {f_pct(ra.boot_CI_high, 2, True)}], "
                                             f"macro-F1 {f_num(rf_.mean_diff, 4, True)}, MCC {f_num(rm.mean_diff, 4, True)}: "
                                             f"{sig(ra.boot_CI_low, ra.boot_CI_high)} in accuracy", tref("xbase_tests")))
    r1, r2 = inf(pf, bb, "MacroF1"), inf(pf, bb, "Accuracy")
    rows.append(("Proposed vs black box", f"macro-F1 {f_num(r1.mean_diff, 4, True)} [{f_num(r1.boot_CI_low, 4, True)}, {f_num(r1.boot_CI_high, 4, True)}]; "
                                          f"avoids {r2.error_rate_reduction_pct:.1f} % of the black box's errors", tref("stats_f1", "stats_acc")))
    x = XBT
    rows.append(("CCME features help every learner", "; ".join(f"{a} {f_num(x.loc[b, 'MacroF1'])} → {f_num(x.loc[c_, 'MacroF1'])}" for a, b, c_ in
                  [("logistic regression", XN["lr"], XN["lr+"]), ("random forest", XN["rf"], XN["rf+"]), ("XGBoost", bb, pf)]) + " (macro-F1)", tref("xbase")))
    rows.append(("Calibrated probabilities", f"Brier {f_num(x.loc[pf, 'Brier'])} (proposed) vs {f_num(x.loc[XN['rcal'], 'Brier'])} (calibrated rule) and "
                                             f"{f_num(x.loc[bb, 'Brier'])} (black box)", tref("xbase")))
    f0 = FEWER.set_index("k")
    rows.append(("Fewer laboratory tests", "; ".join(f"{k} test{'s' if k != 1 else ''}: proposed − rule {f_pct(f0.loc[k, 'dAcc'], 2, True)} pp "
                                                    f"[{f_pct(f0.loc[k, 'dAcc_lo'], 2, True)}, {f_pct(f0.loc[k, 'dAcc_hi'], 2, True)}]" for k in f0.index), tref("fewer_tests")))
    gm = GAPS.set_index("group")
    if "last value of an unmeasured test not available" in gm.index:
        r = gm.loc["last value of an unmeasured test not available"]
        rows.append(("Last laboratory value missing", f"accuracy {f_pct(r.prop_Accuracy)} % (proposed) vs {f_pct(r.rule_Accuracy)} % (rule): "
                                                     f"{f_pct(r.dAcc, 2, True)} pp [{f_pct(r.dAcc_lo, 2, True)}, {f_pct(r.dAcc_hi, 2, True)}]", tref("gap_groups")))
    hd = HDET_S
    rows.append(("Flagging hidden failures (ROC-AUC)", "; ".join(f"{k.split(':')[0].lower()} {f_num(hd.loc[k, 'ROC_AUC'], 3)}" for k in HDN if k in hd.index), tref("hidden_detect")))
    tr = TRIAGE.set_index("method"); RT = "Formula rule: risk from last values and history first"; pq = tr.loc["Proposed: least confident first"]
    cst = COST[COST.price_list == list(LAB_PRICES)[0]].set_index("policy")
    p98 = cst.loc[[i for i in cst.index if i.startswith("Proposed") and "98" in i][0]]
    rows.append(("Selective full-panel testing", f"98 % accuracy when {100 * pq.share_for_98pct:.0f} % of samples get all four tests (rule-based selection: "
                                                 f"{100 * tr.loc[RT, 'share_for_98pct']:.0f} %); laboratory cost {p98.cost_per_sample:.2f} US$ per sample, "
                                                 f"{p98.saving_vs_full_pct:.1f} % below the full panel", tref("triage", "cost")))
    off = RHO.where(~np.eye(len(RHO), dtype=bool)).stack()
    rows.append(("What the model relies on", f"CCME-consistency features carry {float(GS.loc['CCME consistency', 'proposed_%']):.1f} % of |SHAP| ({xc}); "
                                             f"rankings agree across countries (Spearman {off.min():.3f}–{off.max():.3f})", tref("shap_groups", "xai_countries")))
    if RUN_DEEP:
        z = lambda m: f_pct(zt.loc[m, "Accuracy"]); kk = lambda m: " / ".join(f"{100 * pm_parts(kt.loc[(m, k), 'Accuracy'])[0]:.2f}" for k in KSHOTS)
        rows.append(("Zero-shot transfer (accuracy)", f"Transformer {z('Transformer')} %, black box {z('Black box (XGBoost)')} %; with CCME features: "
                     f"Transformer {z('Transformer + CCME features')} %, proposed {z('Proposed (XGBoost + CCME + constraint)')} %", tref("zeroshot_deep")))
        rows.append((f"Few-shot accuracy, K = {' / '.join(map(str, KSHOTS))}", f"Rahman et al. (2026) re-implementation {kk('Transformer-ProtoNet + WGAN-GP (Rahman et al. 2026)')} %; "
                     f"proposed {kk('Proposed (XGBoost + CCME + constraint)')} %", tref("fewshot")))
    rows.append(("Prior-work setting", f"the published formula reproduces the published label for {float(facts.loc['published_formula_reproduces_label_pct', 'value']):.2f} % of records",
                 tref("prior_facts", "prior_full", "prior_episodic")))
    if RUN_ROBUSTNESS:
        rows.append(("Robustness", f"proposed − black box macro-F1 {gain.gain_vs_black_box.min():+.4f} to {gain.gain_vs_black_box.max():+.4f}; proposed − rule "
                                   f"{gain.gain_vs_rule.min():+.4f} to {gain.gain_vs_rule.max():+.4f} over {len(gain)} checks", tref("robust_models", "robust_gains")))
    if RUN_QRC:
        q = QD.round(4)
        rows.append(("Quantum reservoir", f"vs XGBoost on the same encoded inputs: Δ macro-F1 {q.mean_diff_MacroF1.iloc[2]:+.4f}; vs classical random features: "
                                          f"{q.mean_diff_MacroF1.iloc[1]:+.4f} (XGBoost readout)", tref("qrc", "qrc_diff")))
    rows.append(("Changes vs v13", f"{int(REPRO.unchanged_at_4dp.sum())} of {len(REPRO)} v13 values unchanged at 4 decimals (changes are expected, §16b)", tref("repro")))
    t = pd.DataFrame({"Result": [r for _, r, _ in rows], "Table": [s_ for _, _, s_ in rows]}, index=[f_ for f_, _, _ in rows])
    show_table("summary", "Summary of the main results (every value is taken from the tables above)", t, stub="Finding", left=["Result", "Table"])
_present()

In [ ]:
# Claim check (Table 60). Every candidate statement of the manuscript is tested against the values computed in this run.
# "supported" = the 95 % site-bootstrap interval excludes zero in the claimed direction AND the p-value (Holm-adjusted within its
# family where a family of comparisons was made) is below 0.05. "descriptive" rows have no interval. Write the paper from this table.
CL = []
def _claim(family, claim, est, lo=None, hi=None, p=None, table="", interval="95% CI"):
    if lo is None: verdict = "supported (descriptive)" if est > 0 else "not supported (descriptive)"
    else: verdict = "supported" if (lo > 0 and (p is None or p < 0.05)) else "not supported"
    CL.append(dict(family=family, claim=claim, estimate=f"{est:+.4f}", interval=DASH if lo is None else f"{interval} [{lo:+.4f}, {hi:+.4f}]",
                   p=DASH if p is None else f"{p:.3f}", verdict=verdict, table=table))
pf, bb = "Proposed – 2 tests fixed", "Black box – 2 tests"
v = VSRULE.set_index(["method", "metric"])
for met, nm in [("Accuracy", "accuracy"), ("MacroF1", "macro-F1")]:
    r = v.loc[(pf, met)]
    _claim("Main comparison", f"Proposed beats the formula rule in {nm} (2 tests)", r.mean_diff, r.boot_CI_low, r.boot_CI_high, r.p_boot_holm, tref("xbase_tests"))
I = inf_pairs(INF, COMP)
def _inf(a, b, met): return I[(I.pair_a == a) & (I.pair_b == b) & (I.metric == met)].iloc[0]
for met, nm in [("Accuracy", "accuracy"), ("MacroF1", "macro-F1")]:
    r = _inf(pf, bb, met); _claim("Main comparison", f"Proposed beats the black box in {nm}", r.mean_diff, r.boot_CI_low, r.boot_CI_high, r.p_boot_holm, tref("stats_f1", "stats_acc"))
for b_, nm in [("Ablation: without monotonicity constraint", "The monotonicity constraint improves macro-F1"),
               ("Ablation: without history", "Site history improves macro-F1"), ("Ablation: without current sensors", "Current field sensors improve macro-F1")]:
    r = _inf(pf, b_, "MacroF1"); _claim("Components", nm, r.mean_diff, r.boot_CI_low, r.boot_CI_high, r.p_boot_holm, tref("stats_f1"))
r = _inf("Proposed – 2 tests adaptive", pf, "MacroF1")
_claim("Components", "Adaptive test selection beats the validation-chosen fixed pair (macro-F1)", r.mean_diff, r.boot_CI_low, r.boot_CI_high, r.p_boot_holm, tref("stats_f1"))
cf = [(nm, paired(BTX, a_, b_, 1)) for nm, a_, b_ in [("logistic regression", XN["lr+"], XN["lr"]), ("random forest", XN["rf+"], XN["rf"]), ("XGBoost", pf, bb)]]
for (nm, r), ph in zip(cf, holm([r["p_boot"] for _, r in cf])):
    _claim("CCME-consistency features", f"CCME-consistency features improve {nm} (macro-F1)", r["mean_diff"], r["boot_CI_low"], r["boot_CI_high"], ph, tref("xbase"))
for _, r in FEWER.iterrows():
    _claim("Fewer laboratory tests", f"Proposed beats the rule in accuracy with {int(r.k)} laboratory test{'s' if r.k != 1 else ''}", r.dAcc, r.dAcc_lo, r.dAcc_hi, r.dAcc_p_holm, tref("fewer_tests"))
for _, r in GAPS.iterrows():
    _claim("Age of the last value", f"Proposed beats the rule in accuracy: {r.group}", r.dAcc, r.dAcc_lo, r.dAcc_hi, r.dAcc_p_holm, tref("gap_groups"))
r = TRIAGE.set_index("method").loc["Proposed: least confident first"]
_claim("Practical value", "Selective full-panel testing needs the model (accuracy at 20 % sent vs rule-based selection)", r.d20_vs_rule_triage, r.d20_CI_low, r.d20_CI_high, r.d20_p, tref("triage"))
hd = HDET_S
_claim("Practical value", "The proposed model flags hidden failures better than the site's past exceedance rates (ROC-AUC difference)",
       float(hd.loc["Proposed: probability of a class below the ceiling", "ROC_AUC"] - hd.loc["Site history: probability from past exceedance rates", "ROC_AUC"]),
       table=tref("hidden_detect"))
if RUN_QRC:
    for _, r in QD.iterrows():
        _claim("Quantum reservoir", f"{r.comparison}: first model better (macro-F1)", r.mean_diff_MacroF1, r["min"], r["max"], table=tref("qrc_diff"), interval="range")
CLAIMS = pd.DataFrame(CL); savetab(CLAIMS, "tab_claim_check")
def _present():
    t = pd.DataFrame({"Estimate": list(CLAIMS.estimate), "Interval": list(CLAIMS.interval), "p": list(CLAIMS.p), "Verdict": list(CLAIMS.verdict),
                      "Table": list(CLAIMS.table)}, index=pd.MultiIndex.from_tuples(list(zip(CLAIMS.family, CLAIMS.claim))))
    B = np.zeros(t.shape, bool); B[:, 3] = CLAIMS.verdict.str.startswith("supported").to_numpy()
    show_table("claims", "Claim check: which statements the results support", t, stub="Claim", left=["Interval", "Verdict", "Table"], bold=B,
               note="Estimate = difference (first − second, 0–1 scale; accuracy as a fraction). Supported = interval excludes zero in the claimed "
                    "direction and p < 0.05; p is Holm-adjusted within each family of comparisons (none for the selective-testing row). QRC rows "
                    "use the range over countries × seeds instead of a CI. Descriptive rows compare point estimates only. Bold = supported.")
_present()


In [ ]:
L = []
L.append(f"Data: {AUDIT['raw']:,} records -> {AUDIT['after_cleaning']:,} after range checks and (country, site, date, value) de-duplication. "
         f"Primary test countries (station series with >= {MIN_TIER_A_EVAL} fully genuine evaluation samples): {', '.join(PRIMARY)}.")
L.append(f"Label: record-level CCME (unit-tested against the standard formula); published label reproduced for {100*AUDIT['reproduced_values']:.2f}% of records; "
         f"correction (CCME DO ≥ {DO_MIN} mg/L, pH {PH_RANGE[0]}–{PH_RANGE[1]}, denominator 8) changes {AUDIT['class_changed_by_correction_pct']:.1f}% of classes. "
         f"Standard site-year CCME vs record-level index: Spearman {STD_CMP['spearman']:.3f}.")
L.append("HEADLINE (tier A, equal-weight mean over countries, 95% site-bootstrap CI):")
for m in ["Persistence", "Formula rule (last value) – 2 tests", "Black box – 2 tests", "Proposed – 1 tests adaptive", "Proposed – 2 tests fixed", "Proposed – 2 tests adaptive"]:
    r = HEAD.loc[m]; L.append(f"  {m:<38s} Acc {r.Accuracy:.4f} {r.Accuracy_CI} | MacroF1 {r.MacroF1:.4f} {r.MacroF1_CI} | MCC {r.MCC:.4f} | QWK {r.QWK:.4f} | ROC-AUC {r.ROC_AUC:.4f} {r.ROC_AUC_CI}")
pp = pcr[pcr.method == "Proposed – 2 tests fixed"]
L.append("Per country (proposed, 2 fixed tests): " + "; ".join(f"{r.country} {100*r.Accuracy:.2f}% {r.Acc_CI} (n={r.n_tierA}, {r.sites} sites, {r.errors:.1f} errors)" for r in pp.itertuples()))
hf = sub_tab[sub_tab.subset.str.startswith("hidden failure")].set_index("method")
if len(hf):
    L.append(f"Hidden-failure subset (an unmeasured test fails; {int(hf.n.iloc[0])} samples): accuracy rule(last value) {hf.loc['Formula rule (last value) – 2 tests','Accuracy']:.4f}, "
             f"black box {hf.loc['Black box – 2 tests','Accuracy']:.4f}, proposed {hf.loc['Proposed – 2 tests fixed','Accuracy']:.4f}.")
for _, r in INF[INF.metric == "MacroF1"].iterrows():
    L.append(f"{r.comparison}: Δ macro-F1 {r.mean_diff:+.4f}, site-bootstrap CI [{r.boot_CI_low:+.4f}, {r.boot_CI_high:+.4f}] (p = {r.p_boot:.3g}); "
             f"random effects {r.RE_diff:+.4f} [{r.RE_CI_low:+.4f}, {r.RE_CI_high:+.4f}] (p = {r.p_RE:.3g}, I² = {r.I2_pct:.0f}%); "
             f"positive in {r.countries_positive} countries; per-country CI excludes 0 in {r.countries_CI_excludes_0}.")
sk = sel_tab[sel_tab.strategy.str.startswith("Δ")]
for _, r in sk.iterrows(): L.append(f"Selection k={r.k} {r.strategy}: Δ macro-F1 {r.MacroF1:+.4f} {r.MacroF1_CI}.")
L.append("Calibration (pooled tier A): " + "; ".join(f"{m} ECE {r.ECE:.4f}, Brier {r.Brier:.4f}" for m, r in CAL.iterrows()))
L.append(f"XAI: attribution share proposed — " + ", ".join(f"{g} {v:.1f}%" for g, v in GS["proposed_%"].items()) +
         f"; cross-country Spearman of importance rankings {RHO.where(~np.eye(len(RHO), dtype=bool)).stack().min():.3f}–{RHO.where(~np.eye(len(RHO), dtype=bool)).stack().max():.3f}.")
for _, r in CONS.iterrows():
    if r.n_violating > 0 and r.n_meeting > 0:
        L.append(f"XAI domain consistency [{r.measurement} {r.objective}; {int(r.n_violating)} violating vs {int(r.n_meeting)} meeting]: "
                 f"proposed {r['Proposed: consistency']:.3f}, black box {r['Black box: consistency']:.3f}.")
for _, r in AGR.iterrows():
    L.append(f"SHAP–LIME agreement ({r.model}): median rank correlation {r.median_rank_corr:.3f}, top-1 agreement {r.top1_agreement_pct:.1f}%, sign agreement {r.sign_agreement_pct:.1f}%.")
if RUN_DEEP:
    for m, r in zt.iterrows(): L.append(f"Zero-shot [{m}]: accuracy {r.Accuracy:.4f}, macro-F1 {r.MacroF1:.4f}, ROC-AUC {r.ROC_AUC:.4f}.")
    for (m, Kk), r in kt.iterrows(): L.append(f"Few-shot K={Kk} [{m}]: accuracy {r.Accuracy}, macro-F1 {r.MacroF1}, ROC-AUC {r.ROC_AUC}.")
    for (pr_, Kk, m), r in prt.iterrows():
        L.append(f"Prior-work protocol [{pr_} | K={Kk} | {m}]: accuracy {r.Accuracy}, macro-F1 {r.MacroF1}, ROC-AUC {r.ROC_AUC}.")
if RUN_ROBUSTNESS:
    for (an, va), r in gain.iterrows():
        L.append(f"Robustness [{an} | {va}]: proposed − black box macro-F1 {r.gain_vs_black_box:+.4f} (minimum {r.gain_vs_black_box_min:+.4f}); "
                 f"proposed − formula rule {r.gain_vs_rule:+.4f} (minimum {r.gain_vs_rule_min:+.4f}) over {int(r.countries)} countries.")
    for _, r in LABCHG.iterrows():
        L.append(f"Label sensitivity [{r.variant}]: {r.changed_all_pct:.1f}% of station site-day classes change ({r.changed_pool_pct:.1f}% of the experiment pool).")
if RUN_QRC:
    for _, r in QD.iterrows(): L.append(f"QRC {r.comparison}: Δ macro-F1 {r.mean_diff_MacroF1:+.4f} (range {r['min']:+.4f} to {r['max']:+.4f}; positive {r.positive}).")
L.append(f"Leakage audit: {(LEAKAGE.status == 'pass').sum()} tested checks passed, none failed.")
for _, r in VSRULE.iterrows():
    L.append(f"vs formula rule [{r.method} | {r.metric}]: Δ {r.mean_diff:+.4f} [{r.boot_CI_low:+.4f}, {r.boot_CI_high:+.4f}], Holm-adjusted p = {r.p_boot_holm:.3g}.")
for _, r in FEWER.iterrows():
    L.append(f"{int(r.k)} laboratory tests: accuracy rule {r.rule_Accuracy:.4f}, black box {r.bb_Accuracy:.4f}, proposed {r.prop_Accuracy:.4f}; "
             f"proposed − rule {r.dAcc:+.4f} [{r.dAcc_lo:+.4f}, {r.dAcc_hi:+.4f}].")
for _, r in GAPS.iterrows():
    L.append(f"Last laboratory value [{r.group}]: accuracy rule {r.rule_Accuracy:.4f}, proposed {r.prop_Accuracy:.4f}; Δ {r.dAcc:+.4f} [{r.dAcc_lo:+.4f}, {r.dAcc_hi:+.4f}].")
for m, r in HDET_S.iterrows(): L.append(f"Hidden-failure flagging [{m}]: ROC-AUC {r.ROC_AUC:.3f}, failures found when flagging 20% {r.recall_at_20pct_flagged:.3f}.")
for _, r in TRIAGE.iterrows(): L.append(f"Selective full-panel testing [{r.method}]: accuracy with 20% sent {r.acc_at_20pct:.4f}; share sent for 98% accuracy {r.share_for_98pct:.2f}.")
L.append(f"Changes vs v13: {int(REPRO.unchanged_at_4dp.sum())}/{len(REPRO)} values unchanged at 4 decimals (corrections, §16b).")
for _, r in CLAIMS.iterrows(): L.append(f"Claim check [{r.claim}]: {r.verdict} (estimate {r.estimate}, {r.interval}, p {r.p}).")
import sklearn, xgboost
env = f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | shap {shap.__version__}"
print("\nEnvironment:", env, f"| pipeline {PIPELINE_ID} | runtime of this execution {(time.time()-T0)/60:.1f} min")
_ = open(f"{TAB_DIR}/results_summary.md", "w").write("# Results summary (auto-generated)\n\n" + "\n".join("- " + l for l in L) + f"\n\nEnvironment: {env}\nPipeline: {PIPELINE_ID}\n")

## 18. Notes for the manuscript (v14)

**Every number in the paper must come from the tables of a v14 run.** This section contains no numbers on purpose: the v13 notes
quoted values that the corrections change. Claims are decided by Table 60, not by these notes.

**Task statement (use verbatim).** "We estimate the record-level CCME-type class of a sample from its four field-sensor values, a
subset of its four laboratory values and the site's recent history, for monitoring stations in countries not seen during training."
State that the field-sensor values are inputs of the label by design, and report the formula-rule baseline, which shows how much of
the performance needs learning. If `MAIN_BUDGET` > 0 is used for any headline, do not write "countries not seen during training".

**How to describe the label.** "A single-sample CCME-type index: the CCME WQI 1.0 formula applied to one record, with the CCME
dissolved-oxygen minimum (5.5 mg/L) and pH range (6.5–9.0) and, for the other six parameters, the objectives of the published
dataset label." Do not call it "the CCME WQI" (CCME requires at least four parameters and four samples per site and year; report the
site-year agreement of Table 7). List every objective with its basis (Table 8). State that the dataset does not report units, that
nitrate appears to be reported as N in England and Ireland (Table 9) while the nitrate objective of 45 mg/L is a value *as NO₃*
(10 mg/L as N; tested in §15b), and that "N" behaves like total oxidised nitrogen in England and Ireland (median close to nitrate).

**Training labels and imputation.** State that labels used for training, tuning and few-shot support contain no imputed laboratory
value (`TRAIN_TIER`), that evaluation uses tier A only, and report §15e. Values equal to a frequent constant are treated as imputed;
some of these constants are reporting limits (e.g. England NH₃ 0.03 mg/L), i.e. censored measurements rather than imputations —
say so, because excluding them removes clean samples from tier A.

### How to frame the contribution
Decide the framing after the run, from Table 60:
- If the proposed model does **not** beat the formula rule (as in v13), the paper is an *audited benchmark*: (i) audit and
  correction of a widely used public label; (ii) a leakage-free reduced-panel task on genuinely measured samples from unseen
  countries; (iii) evidence that CCME-consistency features are what lets learners transfer across countries; (iv) an honest
  comparison showing that a no-learning formula rule with the last genuine laboratory values is as accurate as every learner.
  Title suggestion: *Does machine learning beat the formula? An audited cross-country benchmark for water-quality classification
  with a reduced laboratory panel.*
- Claim an advantage of learning only for rows marked "supported" in Table 60.
- Do not present the monotonicity constraint, adaptive selection or QRC as contributions unless Table 60 supports them; QRC is
  better placed in the supplement.

### Limitations to state
Single-sample index; five of eight objectives are those of the published label, not CCME guidelines; reporting units are not
documented (nitrate as N vs NO₃; "N" possibly total oxidised N); nitrate is treated as a field-sensor value although it is a
laboratory analysis in these archives (in-situ optical/ISE nitrate sensors exist but have different accuracy); three primary test
countries, Canada small but weighted equally; tier-A evaluation covers a minority of samples and may differ from the population;
heuristic imputation detection (validated in §4b, varied in §15c) that also flags reporting limits; freshwater objectives applied
to some non-freshwater samples (freshwater subset in §10); untuned deep baselines; fees from two US public laboratories; selective
testing requires that the sample is kept for the remaining tests within their holding time (BOD: 48 h, US EPA 40 CFR Part 136,
Table II); the quantum reservoir is classically simulated.

### Positioning against prior work (Table 37)
Reduced-panel prediction of water-quality indices is established (Xu et al., 2024; García-Ceja & González-Mendoza, 2026; Bordjihene
et al., 2026), mostly with random splits within one region and labels computed from the model's own inputs. A 2026 review (Ab Karim
et al., *Environments* 13:267) notes that the deterministic structure of WQIs often makes high predictive performance reflect the
predefined formulation rather than independent pattern learning; paraphrase it (the v13 notes quoted a sentence that does not appear
in its abstract — check the full text before quoting).

### References used in this notebook
Checked against bibliographic databases (October 2026): Karim et al. 2025; Rahman, Chung & Ng 2026; Xu et al. 2024; Bordjihene et
al. 2026; Ab Karim et al. 2026 (*Environments*). **Not found — verify before citing:** García-Ceja & González-Mendoza (2026), LNCS
16221:80–90. The regulatory sources (CCME, S.I. 272/2009, US EPA 1986, Health Canada, fee schedules) should be checked at the cited URLs.
- CCME (2001; user's manual 2017) Water Quality Index 1.0. https://ccme.ca/en/res/wqimanualen.pdf
- CCME (2017) Synthesis of research and application of the CCME WQI. https://ccme.ca/en/res/synthesis-of-research-and-application-of-the-ccme-water-quality-index-2017.pdf
- CCME Canadian Water Quality Guidelines: dissolved oxygen (1999) https://ccme.ca/en/chemical/154 ; ammonia (2010) https://ccme.ca/en/res/ammonia-en-canadian-water-quality-guidelines-for-the-protection-of-aquatic-life.pdf ; nitrate (2012) https://ccme.ca/en/chemical/140 ; phosphorus guidance framework (2004) https://ccme.ca/en/chemical/167
- Ireland, S.I. No. 272/2009 as amended by S.I. No. 77/2019, Schedule 5 (river standards).
- US EPA (1986) Quality Criteria for Water, EPA 440/5-86-001. https://www.epa.gov/sites/default/files/2018-10/documents/quality-criteria-water-1986.pdf
- Health Canada, Guidelines for Canadian Drinking Water Quality — nitrate and nitrite (MAC 45 mg/L as nitrate = 10 mg/L as nitrate-N).
- Standard Methods 5210 B (5-day BOD test). https://nemi.gov/methods/method_summary/5715
- Oklahoma DEQ State Environmental Laboratory Services, Lab Services Fee Schedule FY27; Montana DPHHS Environmental Laboratory Testing Fee Schedule (2024).
- Karim et al. (2025) Scientific Data 12:391, https://doi.org/10.1038/s41597-025-04715-4 ; data https://doi.org/10.6084/m9.figshare.27800394
- Rahman, Chung & Ng (2026) Front. Water 8:1902772, https://doi.org/10.3389/frwa.2026.1902772
- Xu et al. (2024) Heliyon 10:e33695, https://doi.org/10.1016/j.heliyon.2024.e33695 ; García-Ceja & González-Mendoza (2026) LNCS 16221:80–90, https://doi.org/10.1007/978-3-032-09037-9_7 (not verified) ; Bordjihene et al. (2026) Front. Earth Sci. 14, https://doi.org/10.3389/feart.2026.1761770
- Ab Karim et al. (2026) Water quality monitoring and assessment using machine learning: a review of formulation, modeling approaches, and explainable artificial intelligence. Environments 13:267, https://doi.org/10.3390/environments13050267
